In [1]:
!pip install -q transformers==4.44.2 accelerate==0.34.2 peft==0.12.0 \
    datasets==2.21.0 sentence-transformers==3.1.1 \
    rouge-score==0.1.2 bert-score==0.3.13 nltk==3.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 59.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. 

In [2]:
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient

# HuggingFace token
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Paths
INPUT_DIR  = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")
OUTPUT_DIR = Path("/kaggle/working")

LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}

# Load classifier results (SecureBERT predictions + window texts)
classifier_results = pd.read_parquet(
    INPUT_DIR / "section9a_classifier_results.parquet"
)
print(f"\n✓ Loaded classifier results: {len(classifier_results):,} rows")
print(f"  Columns: {list(classifier_results.columns)}")
print(f"  Accuracy: {classifier_results['correct'].mean():.3f}")

Device: cuda
GPU count: 2
GPU name: Tesla T4

✓ Loaded classifier results: 1,500 rows
  Columns: ['text', 'label', 'label_name', 'pred_label', 'pred_name', 'confidence', 'correct', 'prob_NORMAL', 'prob_DOS', 'prob_FUZZY', 'prob_GEAR', 'prob_RPM', 'frames_json', 'window_start_ts']
  Accuracy: 1.000


In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/section9a_classifier_results.parquet
/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/task2_seed_datasets.parquet
/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/section10_perturbed_dataset.parquet
/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/task4_road_windows.parquet
/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/section7_windows_with_text.parquet
/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/task4_road_p6_dataset.parquet
/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark/section10_p6_false_context.parquet


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_ID}...")
t0 = time.time()

qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
qwen_model.eval()

print(f"✓ Qwen2.5-1.5B loaded in {time.time()-t0:.1f}s")
print(f"  Parameters: {sum(p.numel() for p in qwen_model.parameters())/1e9:.2f}B")
print(f"  Device map: {qwen_model.hf_device_map if hasattr(qwen_model, 'hf_device_map') else device}")

Loading Qwen/Qwen2.5-1.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Qwen2.5-1.5B loaded in 24.6s
  Parameters: 1.54B
  Device map: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1}


In [ ]:
SYSTEM_PROMPT = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection. 
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def build_prompt(telemetry_text, securebert_prediction, securebert_confidence):
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {securebert_prediction} (confidence: {securebert_confidence:.3f})

{telemetry_text}

Provide your classification and detailed explanation in the required JSON format."""
    return user_content

def generate_explanation(telemetry_text, securebert_pred, securebert_conf,
                          max_new_tokens=512, temperature=0.1):
    """
    Generate a structured explanation from Qwen for a single CAN window.
    Returns raw response string.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_prompt(
            telemetry_text, securebert_pred, securebert_conf
        )},
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = qwen_tokenizer(
        [text],
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False,       # greedy decoding for consistency
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

print("✓ Prompt builder and generator ready")
print("\nSample prompt preview:")
sample_row = classifier_results.iloc[0]
print(build_prompt(
    sample_row["text"][:300] + "...",
    sample_row["pred_name"],
    sample_row["confidence"]
))

In [ ]:
# Test on one sample before running the full batch
print("Testing single inference...")
sample = classifier_results[classifier_results["label_name"] == "DOS"].iloc[0]

t0 = time.time()
response = generate_explanation(
    sample["text"],
    sample["pred_name"],
    sample["confidence"],
)
elapsed = time.time() - t0

print(f"✓ Inference completed in {elapsed:.1f}s")
print(f"\nRaw response:")
print("-"*60)
print(response)
print("-"*60)

# Try parsing as JSON
try:
    # Strip markdown code fences if present
    clean = response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    parsed = json.loads(clean.strip())
    print(f"\n✓ Valid JSON response")
    print(f"  Classification: {parsed.get('classification')}")
    print(f"  Confidence:     {parsed.get('confidence')}")
    print(f"  Explanation:    {parsed.get('explanation', '')[:150]}...")
except json.JSONDecodeError as e:
    print(f"\n⚠ JSON parse failed: {e}")
    print("  Will handle in batch processing")

In [ ]:
import json
from tqdm import tqdm

# Stratified sample: 100 per class
SAMPLES_PER_CLASS = 100
np.random.seed(42)

sample_dfs = []
for lbl, name in LABEL_MAP.items():
    class_df = classifier_results[classifier_results["label"] == lbl]
    sampled = class_df.sample(n=min(SAMPLES_PER_CLASS, len(class_df)), random_state=42)
    sample_dfs.append(sampled)
    print(f"  {name:7s}: {len(sampled)} samples")

eval_df = pd.concat(sample_dfs).reset_index(drop=True)
print(f"\nTotal evaluation samples: {len(eval_df)}")

def safe_parse_json(response_str):
    """Parse Qwen JSON response, handling markdown code fences."""
    clean = response_str.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

# Run inference
results = []
errors  = 0

print(f"\nStarting inference on {len(eval_df)} windows...")
print(f"Estimated time: ~{len(eval_df) * 20 / 3600:.1f} hours")
print()

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Qwen inference"):
    t0 = time.time()
    try:
        raw_response = generate_explanation(
            row["text"],
            row["pred_name"],
            row["confidence"],
        )
        parsed, parse_error = safe_parse_json(raw_response)
        elapsed = time.time() - t0

        results.append({
            "window_idx":          idx,
            "true_label":          row["label"],
            "true_label_name":     row["label_name"],
            "securebert_pred":     row["pred_label"],
            "securebert_pred_name": row["pred_name"],
            "securebert_confidence": row["confidence"],
            "qwen_raw":            raw_response,
            "qwen_classification": parsed.get("classification") if parsed else None,
            "qwen_confidence":     parsed.get("confidence") if parsed else None,
            "qwen_explanation":    parsed.get("explanation") if parsed else None,
            "qwen_key_indicators": json.dumps(parsed.get("key_indicators", [])) if parsed else None,
            "qwen_temporal":       parsed.get("temporal_reasoning") if parsed else None,
            "json_valid":          parsed is not None,
            "parse_error":         parse_error,
            "inference_time_s":    elapsed,
        })

    except Exception as e:
        errors += 1
        results.append({
            "window_idx":          idx,
            "true_label":          row["label"],
            "true_label_name":     row["label_name"],
            "securebert_pred":     row["pred_label"],
            "securebert_pred_name": row["pred_name"],
            "securebert_confidence": row["confidence"],
            "qwen_raw":            f"ERROR: {str(e)}",
            "qwen_classification": None,
            "qwen_confidence":     None,
            "qwen_explanation":    None,
            "qwen_key_indicators": None,
            "qwen_temporal":       None,
            "json_valid":          False,
            "parse_error":         str(e),
            "inference_time_s":    time.time() - t0,
        })

    # Save checkpoint every 50 samples
    if len(results) % 50 == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_parquet(f"/kaggle/working/baseline_checkpoint_{len(results)}.parquet")
        print(f"  Checkpoint saved at {len(results)} samples")

print(f"\n✓ Inference complete")
print(f"  Total: {len(results)}, Errors: {errors}")
print(f"  JSON valid: {sum(r['json_valid'] for r in results)}/{len(results)}")

In [ ]:
results_df = pd.DataFrame(results)

# Summary statistics
print("BASELINE INFERENCE SUMMARY")
print("="*50)
print(f"Total windows:     {len(results_df)}")
print(f"JSON valid:        {results_df['json_valid'].sum()} ({results_df['json_valid'].mean()*100:.1f}%)")
print(f"Mean inference time: {results_df['inference_time_s'].mean():.1f}s")
print(f"\nQwen classification vs true label:")
print(pd.crosstab(
    results_df["true_label_name"],
    results_df["qwen_classification"].fillna("PARSE_ERROR"),
    margins=True
))

# Save
output_path = "/kaggle/working/section9b_baseline_explanations.parquet"
results_df.to_parquet(output_path, index=False)
print(f"\n✓ Saved to {output_path}")
print(f"  Rows: {len(results_df)}")
print(f"  Size: {Path(output_path).stat().st_size / 1e6:.1f} MB")

In [ ]:
import os

output_path = "/kaggle/working/section9b_baseline_explanations.parquet"
size_bytes = os.path.getsize(output_path)
print(f"File size: {size_bytes:,} bytes ({size_bytes/1e6:.2f} MB)")

# Verify by reloading
verify_df = pd.read_parquet(output_path)
print(f"Rows: {len(verify_df)}")
print(f"Columns: {list(verify_df.columns)}")
print(f"\nSample explanation (row 0):")
print(verify_df['qwen_explanation'].iloc[0])
print(f"\nSample key indicators (row 0):")
print(verify_df['qwen_key_indicators'].iloc[0])
print(f"\nSample temporal reasoning (row 0):")
print(verify_df['qwen_temporal'].iloc[0])

In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working')

In [ ]:
import zipfile
import os

# Zip all output files into one archive
zip_path = "/kaggle/working/section9b_all_outputs.zip"

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for filename in os.listdir("/kaggle/working"):
        if filename.endswith(".parquet") and filename != "section9b_all_outputs.zip":
            filepath = f"/kaggle/working/{filename}"
            zipf.write(filepath, filename)
            print(f"  Added: {filename} ({os.path.getsize(filepath)/1e3:.1f} KB)")

print(f"\n✓ Zip created: {os.path.getsize(zip_path)/1e6:.2f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/section9b_all_outputs.zip')

In [ ]:
# Reload input path in case session restarted
INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

perturbed_df = pd.read_parquet(
    INPUT_DIR / "section10_perturbed_dataset.parquet"
)
print(f"✓ Loaded perturbed dataset: {len(perturbed_df):,} records")
print(f"\nPerturbation type distribution:")
print(perturbed_df["perturbation_type"].value_counts().to_string())
print(f"\nColumns: {list(perturbed_df.columns)}")

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
SYSTEM_PROMPT = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection. 
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def build_perturbed_prompt(telemetry_text, securebert_prediction,
                            securebert_confidence, steering_instruction=""):
    """
    Build prompt for perturbed inference.
    For P5 (semantic steering), appends the steering instruction.
    For P4 (confidence reduction), uses reduced confidence value.
    For P1/P2/P3, telemetry text is already perturbed — prompt structure same.
    """
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {securebert_prediction} (confidence: {securebert_confidence:.3f})
"""
    if steering_instruction:
        user_content += f"\n{steering_instruction}\n"

    user_content += f"""
{telemetry_text}

Provide your classification and detailed explanation in the required JSON format."""

    return user_content


def generate_perturbed_explanation(telemetry_text, securebert_pred,
                                    securebert_conf, steering="",
                                    max_new_tokens=512, temperature=0.1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_perturbed_prompt(
            telemetry_text, securebert_pred, securebert_conf, steering
        )},
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = qwen_tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


def safe_parse_json(response_str):
    clean = response_str.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

print("✓ Perturbed prompt builder ready")

In [ ]:
print(f"Model loaded: {qwen_model is not None}")
print(f"Device: {device}")

In [ ]:
# Run P1 and P2 only in this session (~2 hours)
# P3, P4, P5 in next session
RUN_TYPES = ["P1_PAYLOAD", "P2_INSERTION"]

run_df = perturbed_df[
    perturbed_df["perturbation_type"].isin(RUN_TYPES)
].reset_index(drop=True)

print(f"Running inference on {len(run_df)} perturbed windows")
print(f"Perturbation types: {RUN_TYPES}")
print(f"Estimated time: ~{len(run_df) * 6 / 3600:.1f} hours")
print()

perturbed_results = []
errors = 0

for idx, row in tqdm(run_df.iterrows(), total=len(run_df),
                      desc="Perturbed inference"):
    t0 = time.time()
    try:
        raw_response = generate_perturbed_explanation(
            row["perturbed_text"],
            row["securebert_pred_name"],
            row["securebert_conf_for_prompt"],
            steering=row["steering_instruction"],
        )
        parsed, parse_error = safe_parse_json(raw_response)
        elapsed = time.time() - t0

        perturbed_results.append({
            "window_idx":              row["window_idx"],
            "true_label":              row["true_label"],
            "true_label_name":         row["true_label_name"],
            "perturbation_type":       row["perturbation_type"],
            "baseline_explanation":    row["baseline_explanation"],
            "baseline_classification": row["baseline_classification"],
            "baseline_confidence":     row["baseline_confidence"],
            "baseline_alignment":      row["baseline_alignment"],
            "perturbed_raw":           raw_response,
            "perturbed_classification": parsed.get("classification") if parsed else None,
            "perturbed_confidence":    parsed.get("confidence") if parsed else None,
            "perturbed_explanation":   parsed.get("explanation") if parsed else None,
            "perturbed_indicators":    json.dumps(parsed.get("key_indicators", [])) if parsed else None,
            "perturbed_temporal":      parsed.get("temporal_reasoning") if parsed else None,
            "json_valid":              parsed is not None,
            "parse_error":             parse_error,
            "inference_time_s":        elapsed,
            "perturbation_meta":       row["perturbation_meta"],
            "steering_instruction":    row["steering_instruction"],
        })

    except Exception as e:
        errors += 1
        perturbed_results.append({
            "window_idx":              row["window_idx"],
            "true_label":              row["true_label"],
            "true_label_name":         row["true_label_name"],
            "perturbation_type":       row["perturbation_type"],
            "baseline_explanation":    row["baseline_explanation"],
            "baseline_classification": row["baseline_classification"],
            "baseline_confidence":     row["baseline_confidence"],
            "baseline_alignment":      row["baseline_alignment"],
            "perturbed_raw":           f"ERROR: {str(e)}",
            "perturbed_classification": None,
            "perturbed_confidence":    None,
            "perturbed_explanation":   None,
            "perturbed_indicators":    None,
            "perturbed_temporal":      None,
            "json_valid":              False,
            "parse_error":             str(e),
            "inference_time_s":        time.time() - t0,
            "perturbation_meta":       row["perturbation_meta"],
            "steering_instruction":    row["steering_instruction"],
        })

    # Checkpoint every 50
    if len(perturbed_results) % 50 == 0:
        cp = pd.DataFrame(perturbed_results)
        cp.to_parquet(
            f"/kaggle/working/perturbed_checkpoint_{len(perturbed_results)}.parquet"
        )
        print(f"  Checkpoint saved: {len(perturbed_results)} samples")

print(f"\n✓ Inference complete")
print(f"  Total: {len(perturbed_results)}, Errors: {errors}")
print(f"  JSON valid: {sum(r['json_valid'] for r in perturbed_results)}"
      f"/{len(perturbed_results)}")

In [ ]:
p1p2_df = pd.DataFrame(perturbed_results)
output_path = "/kaggle/working/section10_perturbed_p1p2.parquet"
p1p2_df.to_parquet(output_path, index=False)

print(f"✓ Saved P1+P2 results")
print(f"  Rows: {len(p1p2_df)}")
print(f"  Size: {Path(output_path).stat().st_size / 1e6:.2f} MB")
print(f"\nClassification stability check:")
print(pd.crosstab(
    p1p2_df["true_label_name"],
    p1p2_df["perturbed_classification"].fillna("PARSE_ERROR"),
    margins=False
).to_string())
print(f"\nJSON valid rate: {p1p2_df['json_valid'].mean()*100:.1f}%")

In [ ]:
# Run P3, P4, P5
RUN_TYPES = ["P3_REORDER", "P4_PROMPT", "P5_STEERING"]

In [ ]:
# Run P1 and P2 only in this session (~2 hours)

# Run P3, P4, P5
RUN_TYPES = ["P3_REORDER", "P4_PROMPT", "P5_STEERING"]

run_df = perturbed_df[
    perturbed_df["perturbation_type"].isin(RUN_TYPES)
].reset_index(drop=True)

print(f"Running inference on {len(run_df)} perturbed windows")
print(f"Perturbation types: {RUN_TYPES}")
print(f"Estimated time: ~{len(run_df) * 6 / 3600:.1f} hours")
print()

perturbed_results = []
errors = 0

for idx, row in tqdm(run_df.iterrows(), total=len(run_df),
                      desc="Perturbed inference"):
    t0 = time.time()
    try:
        raw_response = generate_perturbed_explanation(
            row["perturbed_text"],
            row["securebert_pred_name"],
            row["securebert_conf_for_prompt"],
            steering=row["steering_instruction"],
        )
        parsed, parse_error = safe_parse_json(raw_response)
        elapsed = time.time() - t0

        perturbed_results.append({
            "window_idx":              row["window_idx"],
            "true_label":              row["true_label"],
            "true_label_name":         row["true_label_name"],
            "perturbation_type":       row["perturbation_type"],
            "baseline_explanation":    row["baseline_explanation"],
            "baseline_classification": row["baseline_classification"],
            "baseline_confidence":     row["baseline_confidence"],
            "baseline_alignment":      row["baseline_alignment"],
            "perturbed_raw":           raw_response,
            "perturbed_classification": parsed.get("classification") if parsed else None,
            "perturbed_confidence":    parsed.get("confidence") if parsed else None,
            "perturbed_explanation":   parsed.get("explanation") if parsed else None,
            "perturbed_indicators":    json.dumps(parsed.get("key_indicators", [])) if parsed else None,
            "perturbed_temporal":      parsed.get("temporal_reasoning") if parsed else None,
            "json_valid":              parsed is not None,
            "parse_error":             parse_error,
            "inference_time_s":        elapsed,
            "perturbation_meta":       row["perturbation_meta"],
            "steering_instruction":    row["steering_instruction"],
        })

    except Exception as e:
        errors += 1
        perturbed_results.append({
            "window_idx":              row["window_idx"],
            "true_label":              row["true_label"],
            "true_label_name":         row["true_label_name"],
            "perturbation_type":       row["perturbation_type"],
            "baseline_explanation":    row["baseline_explanation"],
            "baseline_classification": row["baseline_classification"],
            "baseline_confidence":     row["baseline_confidence"],
            "baseline_alignment":      row["baseline_alignment"],
            "perturbed_raw":           f"ERROR: {str(e)}",
            "perturbed_classification": None,
            "perturbed_confidence":    None,
            "perturbed_explanation":   None,
            "perturbed_indicators":    None,
            "perturbed_temporal":      None,
            "json_valid":              False,
            "parse_error":             str(e),
            "inference_time_s":        time.time() - t0,
            "perturbation_meta":       row["perturbation_meta"],
            "steering_instruction":    row["steering_instruction"],
        })

    # Checkpoint every 50
    if len(perturbed_results) % 50 == 0:
        cp = pd.DataFrame(perturbed_results)
        cp.to_parquet(
            f"/kaggle/working/perturbed_checkpoint_{len(perturbed_results)}.parquet"
        )
        print(f"  Checkpoint saved: {len(perturbed_results)} samples")

print(f"\n✓ Inference complete")
print(f"  Total: {len(perturbed_results)}, Errors: {errors}")
print(f"  JSON valid: {sum(r['json_valid'] for r in perturbed_results)}"
      f"/{len(perturbed_results)}")

In [ ]:
# Quick explanation drift preview for P1 and P2
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading sentence transformer for similarity...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Loaded")

# Sample 5 windows per class per perturbation type
# and compare baseline vs perturbed explanation
print("\nEXPLANATION DRIFT PREVIEW")
print("="*70)

for ptype in ["P1_PAYLOAD", "P2_INSERTION"]:
    print(f"\n--- {ptype} ---")
    ptype_df = p1p2_df[p1p2_df["perturbation_type"] == ptype]
    
    similarities = []
    for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
        class_df = ptype_df[ptype_df["true_label_name"] == name].head(20)
        
        baseline_exps = class_df["baseline_explanation"].tolist()
        perturbed_exps = class_df["perturbed_explanation"].tolist()
        
        # Filter out None
        pairs = [(b, p) for b, p in zip(baseline_exps, perturbed_exps) 
                 if b and p]
        
        if not pairs:
            continue
            
        base_embs = embedder.encode([p[0] for p in pairs])
        pert_embs = embedder.encode([p[1] for p in pairs])
        
        # Cosine similarity per pair
        sims = []
        for b_emb, p_emb in zip(base_embs, pert_embs):
            cos_sim = np.dot(b_emb, p_emb) / (
                np.linalg.norm(b_emb) * np.linalg.norm(p_emb)
            )
            sims.append(cos_sim)
        
        mean_sim = np.mean(sims)
        similarities.append(mean_sim)
        print(f"  {name:7s}  cosine_sim={mean_sim:.3f}  "
              f"({'HIGH drift' if mean_sim < 0.85 else 'LOW drift' if mean_sim > 0.95 else 'MODERATE drift'})")
    
    print(f"  Overall mean similarity: {np.mean(similarities):.3f}")

print("\nInterpretation guide:")
print("  > 0.95 = minimal drift (nearly identical explanations)")
print("  0.85-0.95 = moderate drift (same theme, different wording)")
print("  < 0.85 = significant drift (explanation meaningfully changed)")

In [ ]:
# Find the most dramatic RPM drift example from P1
rpm_p1 = p1p2_df[
    (p1p2_df["true_label_name"] == "RPM") &
    (p1p2_df["perturbation_type"] == "P1_PAYLOAD")
].copy()

# Compute similarity for each row
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

sims = []
for _, row in rpm_p1.iterrows():
    if row["baseline_explanation"] and row["perturbed_explanation"]:
        b = embedder.encode([row["baseline_explanation"]])[0]
        p = embedder.encode([row["perturbed_explanation"]])[0]
        cos_sim = np.dot(b, p) / (np.linalg.norm(b) * np.linalg.norm(p))
        sims.append(cos_sim)
    else:
        sims.append(1.0)

rpm_p1["similarity"] = sims

# Find the most drifted example
most_drifted = rpm_p1.loc[rpm_p1["similarity"].idxmin()]
meta = json.loads(most_drifted["perturbation_meta"])

print("MOST DRAMATIC RPM DRIFT EXAMPLE")
print("="*70)
print(f"Similarity score: {most_drifted['similarity']:.3f}")
print(f"Perturbation: frame {meta['frame_idx']}, "
      f"byte {meta['byte_idx']}: "
      f"{meta['original']} → {meta['mutated']} (delta={meta['delta']})")
print()
print("BEFORE (baseline explanation):")
print("-"*70)
print(most_drifted["baseline_explanation"])
print()
print("AFTER (perturbed explanation):")
print("-"*70)
print(most_drifted["perturbed_explanation"])
print()
print(f"Classification BEFORE: {most_drifted['baseline_classification']}")
print(f"Classification AFTER:  {most_drifted['perturbed_classification']}")
print(f"Qwen confidence BEFORE: {most_drifted['baseline_confidence']}")
print(f"Qwen confidence AFTER:  {most_drifted['perturbed_confidence']}")

In [ ]:
# Top 5 most drifted examples across all classes for P1
p1_df = p1p2_df[p1p2_df["perturbation_type"] == "P1_PAYLOAD"].copy()

# Compute similarity for all rows
sims = []
for _, row in p1_df.iterrows():
    if row["baseline_explanation"] and row["perturbed_explanation"]:
        b = embedder.encode([row["baseline_explanation"]])[0]
        p = embedder.encode([row["perturbed_explanation"]])[0]
        cos_sim = np.dot(b, p) / (np.linalg.norm(b) * np.linalg.norm(p))
        sims.append(cos_sim)
    else:
        sims.append(1.0)

p1_df["similarity"] = sims

# Sort by most drifted
top5 = p1_df.nsmallest(5, "similarity")

for rank, (_, row) in enumerate(top5.iterrows(), 1):
    meta = json.loads(row["perturbation_meta"])
    print(f"RANK {rank} — {row['true_label_name']}  "
          f"similarity={row['similarity']:.3f}")
    print(f"Perturbation: frame {meta['frame_idx']}, "
          f"byte {meta['byte_idx']}: "
          f"{meta['original']} → {meta['mutated']} "
          f"(delta={meta['delta']})")
    print(f"Classification: {row['baseline_classification']} → "
          f"{row['perturbed_classification']}  "
          f"Confidence: {row['baseline_confidence']} → "
          f"{row['perturbed_confidence']}")
    print()
    print(f"BEFORE: {row['baseline_explanation']}")
    print()
    print(f"AFTER:  {row['perturbed_explanation']}")
    print("="*70)
    print()

In [ ]:
# Check if other classes also have identical perturbations
print("PERTURBATION DIVERSITY CHECK")
print("="*50)
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    class_p1 = p1_df[p1_df["true_label_name"] == name]
    unique_metas = class_p1["perturbation_meta"].nunique()
    unique_sims = class_p1["similarity"].nunique()
    print(f"  {name:7s}  unique perturbations: {unique_metas:3d}  "
          f"unique similarities: {unique_sims:3d}")

In [ ]:
# Show most drifted DOS and FUZZY examples
for target_class in ["DOS", "FUZZY", "NORMAL"]:
    sample = p1_df[p1_df["true_label_name"] == target_class].nsmallest(1, "similarity").iloc[0]
    meta = json.loads(sample["perturbation_meta"])
    print(f"\n{target_class} — similarity={sample['similarity']:.3f}")
    print(f"Perturbation: frame {meta['frame_idx']}, byte {meta['byte_idx']}: "
          f"{meta['original']} → {meta['mutated']} (delta={meta['delta']})")
    print(f"Classification: {sample['baseline_classification']} → "
          f"{sample['perturbed_classification']}")
    print(f"\nBEFORE: {sample['baseline_explanation']}")
    print(f"\nAFTER:  {sample['perturbed_explanation']}")
    print("="*70)

In [ ]:
# CONSISTENCY BASELINE EXPERIMENT
# Purpose: Prove Qwen is effectively deterministic under greedy decoding
# Method: Run same 20 unperturbed windows through Qwen twice
#         If similarity > 0.98, drift is caused by perturbations not model variance
# This directly answers: "How do you know drift isn't just natural LLM variance?"

print("CONSISTENCY BASELINE EXPERIMENT")
print("="*60)
print("Testing Qwen determinism under greedy decoding...")
print()

# Sample 4 windows per class = 20 total
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Get 4 windows per class from classifier results
consistency_samples = []
for lbl in range(5):
    class_rows = classifier_results[
        classifier_results["label"] == lbl
    ].head(4)
    consistency_samples.append(class_rows)

consistency_df = pd.concat(consistency_samples).reset_index(drop=True)
print(f"Selected {len(consistency_df)} windows (4 per class)")
print()

# Run each window through Qwen TWICE
run1_explanations = []
run2_explanations = []
similarities = []

for idx, row in tqdm(consistency_df.iterrows(),
                      total=len(consistency_df),
                      desc="Consistency check"):
    # Run 1
    response1 = generate_perturbed_explanation(
        row["text"],
        row["pred_name"],
        row["confidence"],
        steering="",
    )
    parsed1, _ = safe_parse_json(response1)
    exp1 = parsed1.get("explanation", "") if parsed1 else ""

    # Run 2 — identical input, no changes
    response2 = generate_perturbed_explanation(
        row["text"],
        row["pred_name"],
        row["confidence"],
        steering="",
    )
    parsed2, _ = safe_parse_json(response2)
    exp2 = parsed2.get("explanation", "") if parsed2 else ""

    run1_explanations.append(exp1)
    run2_explanations.append(exp2)

    # Compute similarity
    if exp1 and exp2:
        emb1 = embedder.encode([exp1])[0]
        emb2 = embedder.encode([exp2])[0]
        sim = np.dot(emb1, emb2) / (
            np.linalg.norm(emb1) * np.linalg.norm(emb2)
        )
    else:
        sim = 0.0
    similarities.append(sim)

# Results
consistency_df = consistency_df.copy()
consistency_df["run1_explanation"] = run1_explanations
consistency_df["run2_explanation"] = run2_explanations
consistency_df["consistency_similarity"] = similarities

print("\nCONSISTENCY RESULTS")
print("="*60)
print(f"Overall mean similarity (run1 vs run2): "
      f"{np.mean(similarities):.4f}")
print(f"Min similarity: {np.min(similarities):.4f}")
print(f"Max similarity: {np.max(similarities):.4f}")
print(f"Std: {np.std(similarities):.4f}")
print()

LABEL_MAP_STR = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}
print("Per-class consistency:")
for lbl, name in LABEL_MAP_STR.items():
    class_sims = [
        s for s, l in zip(similarities, consistency_df["label"].tolist())
        if l == lbl
    ]
    if class_sims:
        print(f"  {name:7s}  mean={np.mean(class_sims):.4f}  "
              f"min={np.min(class_sims):.4f}")

print()
print("INTERPRETATION:")
mean_sim = np.mean(similarities)
if mean_sim > 0.98:
    print(f"  ✓ Mean similarity {mean_sim:.4f} > 0.98")
    print(f"  ✓ Qwen is effectively deterministic under greedy decoding")
    print(f"  ✓ Observed explanation drift is caused by perturbations,")
    print(f"    NOT by inherent model variance")
    print(f"  ✓ This validates your experimental methodology")
elif mean_sim > 0.95:
    print(f"  ~ Mean similarity {mean_sim:.4f} — mostly deterministic")
    print(f"  ~ Small natural variance exists but is much lower than")
    print(f"    observed perturbation drift (mean 0.739)")
    print(f"  ~ Your findings remain valid but note variance in paper")
else:
    print(f"  ⚠ Mean similarity {mean_sim:.4f} — significant natural variance")
    print(f"  ⚠ Need to account for this in drift measurements")

print()
print("COMPARISON TO PERTURBATION DRIFT:")
print(f"  Natural variance (this experiment): {mean_sim:.4f}")
print(f"  P1 payload mutation drift:          0.739")
print(f"  P2 benign insertion drift:          0.776")
print(f"  Drift signal-to-noise ratio:        "
      f"{(1-0.739)/(1-mean_sim):.1f}x above natural variance"
      if mean_sim < 1.0 else "  Perfect determinism")

# Save results
consistency_path = "/kaggle/working/consistency_baseline.parquet"
consistency_df.to_parquet(consistency_path, index=False)
print(f"\n✓ Saved consistency results to {consistency_path}")

# Show one example of run1 vs run2 for paper
print("\nSAMPLE CONSISTENCY EXAMPLE (same input, two runs):")
print("="*60)
sample_idx = 0
print(f"Class: {LABEL_MAP_STR[consistency_df.iloc[sample_idx]['label']]}")
print(f"Similarity: {consistency_df.iloc[sample_idx]['consistency_similarity']:.4f}")
print(f"\nRun 1: {consistency_df.iloc[sample_idx]['run1_explanation']}")
print(f"\nRun 2: {consistency_df.iloc[sample_idx]['run2_explanation']}")

In [ ]:
p3p4p5_df = pd.DataFrame(perturbed_results)
output_path = "/kaggle/working/section10_perturbed_p3p4p5.parquet"
p3p4p5_df.to_parquet(output_path, index=False)
print(f"✓ Saved P3+P4+P5 results")
print(f"  Rows: {len(p3p4p5_df)}")
print(f"  Size: {Path(output_path).stat().st_size / 1e6:.2f} MB")
print(f"\nClassification stability check:")
print(pd.crosstab(
    p3p4p5_df["true_label_name"],
    p3p4p5_df["perturbed_classification"].fillna("PARSE_ERROR"),
    margins=False
).to_string())
print(f"\nJSON valid rate: {p3p4p5_df['json_valid'].mean()*100:.1f}%")

In [ ]:
# Drift preview for P3, P4, P5
print("EXPLANATION DRIFT PREVIEW — P3, P4, P5")
print("="*70)

for ptype in ["P3_REORDER", "P4_PROMPT", "P5_STEERING"]:
    print(f"\n--- {ptype} ---")
    ptype_df = p3p4p5_df[p3p4p5_df["perturbation_type"] == ptype]
    
    similarities = []
    for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
        class_df = ptype_df[ptype_df["true_label_name"] == name].head(20)
        
        pairs = [
            (b, p) for b, p in zip(
                class_df["baseline_explanation"].tolist(),
                class_df["perturbed_explanation"].tolist()
            ) if b and p
        ]
        
        if not pairs:
            continue
            
        base_embs = embedder.encode([p[0] for p in pairs])
        pert_embs = embedder.encode([p[1] for p in pairs])
        
        sims = []
        for b_emb, p_emb in zip(base_embs, pert_embs):
            cos_sim = np.dot(b_emb, p_emb) / (
                np.linalg.norm(b_emb) * np.linalg.norm(p_emb)
            )
            sims.append(cos_sim)
        
        mean_sim = np.mean(sims)
        similarities.append(mean_sim)
        print(f"  {name:7s}  cosine_sim={mean_sim:.3f}  "
              f"({'HIGH drift' if mean_sim < 0.85 else 'LOW drift' if mean_sim > 0.95 else 'MODERATE drift'})")
    
    print(f"  Overall mean: {np.mean(similarities):.3f}")

print("\n--- FULL COMPARISON ACROSS ALL PERTURBATION TYPES ---")
print(f"  P1 PAYLOAD:  0.739  (from earlier)")
print(f"  P2 INSERTION: 0.776  (from earlier)")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/section10_all_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for filename in os.listdir("/kaggle/working"):
        if filename.endswith(".parquet"):
            filepath = f"/kaggle/working/{filename}"
            zipf.write(filepath, filename)
            print(f"  Added: {filename}")

from IPython.display import FileLink
print(f"\n✓ Zip size: {os.path.getsize(zip_path)/1e6:.2f} MB")
FileLink('/kaggle/working/section10_all_results.zip')

In [ ]:
import os
print("Files in /kaggle/working/:")
for f in sorted(os.listdir("/kaggle/working/")):
    if f.endswith(".parquet"):
        size = os.path.getsize(f"/kaggle/working/{f}") / 1024
        print(f"  {f} ({size:.1f} KB)")

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
print(f"Model loaded: {qwen_model is not None}")
print(f"Device: {device}")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
import pandas as pd
import numpy as np
import json
import time
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR  = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")
OUTPUT_DIR = Path("/kaggle/working")

# Load seed datasets
seeds_df = pd.read_parquet(INPUT_DIR / "task2_seed_datasets.parquet")
print(f"✓ Loaded seed datasets: {len(seeds_df)} records")
print(f"  Seeds: {sorted(seeds_df['seed'].unique())}")
print(f"  Per seed: {seeds_df['seed'].value_counts().to_dict()}")

# Load embedder for similarity
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Embedder ready")

SYSTEM_PROMPT = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_explanation(text, pred, conf):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {pred} (confidence: {conf:.3f})

{text}

Provide your classification and detailed explanation in the required JSON format."""}
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

def cosine_sim(text1, text2):
    if not text1 or not text2:
        return None
    e1 = embedder.encode([text1])[0]
    e2 = embedder.encode([text2])[0]
    return float(np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2)))

# Run inference for all 300 records
# For each record: generate explanation for original AND p1 perturbed
# Then compute cosine similarity between the two
print(f"\nStarting inference on {len(seeds_df)} records...")
print(f"Estimated time: ~{len(seeds_df) * 2 * 6 / 3600:.1f} hours")
print()

results = []
errors = 0

for idx, row in tqdm(seeds_df.iterrows(),
                      total=len(seeds_df),
                      desc="Seed verification"):
    t0 = time.time()
    try:
        label_name = row["true_label_name"]

        # Generate baseline explanation
        resp_orig = generate_explanation(
            row["original_text"], label_name, 1.0
        )
        parsed_orig, _ = safe_parse(resp_orig)
        exp_orig = parsed_orig.get("explanation", "") if parsed_orig else ""

        # Generate perturbed explanation
        resp_p1 = generate_explanation(
            row["p1_text"], label_name, 1.0
        )
        parsed_p1, _ = safe_parse(resp_p1)
        exp_p1 = parsed_p1.get("explanation", "") if parsed_p1 else ""

        # Compute similarity
        sim = cosine_sim(exp_orig, exp_p1)

        results.append({
            "seed":              row["seed"],
            "window_idx":        row["window_idx"],
            "true_label_name":   label_name,
            "original_exp":      exp_orig,
            "p1_exp":            exp_p1,
            "cosine_similarity": sim,
            "orig_class":        parsed_orig.get("classification") if parsed_orig else None,
            "p1_class":          parsed_p1.get("classification") if parsed_p1 else None,
            "json_valid":        parsed_orig is not None and parsed_p1 is not None,
            "inference_time_s":  time.time() - t0,
        })

    except Exception as e:
        errors += 1
        results.append({
            "seed":              row["seed"],
            "window_idx":        row["window_idx"],
            "true_label_name":   row["true_label_name"],
            "original_exp":      "",
            "p1_exp":            "",
            "cosine_similarity": None,
            "orig_class":        None,
            "p1_class":          None,
            "json_valid":        False,
            "inference_time_s":  time.time() - t0,
        })

    # Checkpoint every 50
    if len(results) % 50 == 0:
        cp = pd.DataFrame(results)
        cp.to_parquet(f"/kaggle/working/task2_checkpoint_{len(results)}.parquet")
        print(f"  Checkpoint: {len(results)} records")

print(f"\n✓ Inference complete")
print(f"  Total: {len(results)}, Errors: {errors}")
print(f"  JSON valid: {sum(r['json_valid'] for r in results)}/{len(results)}")

# Save results
results_df = pd.DataFrame(results)
results_df.to_parquet("/kaggle/working/task2_seed_results.parquet", index=False)

# Summary
print("\nSEED STABILITY RESULTS")
print("="*60)
print("(If mean similarity is within ±0.05 across seeds,")
print(" findings are seed-stable)")
print()

seed_summary = []
for seed in [42, 123, 2024]:
    seed_rows = results_df[results_df["seed"] == seed]
    mean_sim = seed_rows["cosine_similarity"].dropna().mean()
    std_sim  = seed_rows["cosine_similarity"].dropna().std()
    seed_summary.append({"seed": seed, "mean": mean_sim, "std": std_sim})
    print(f"  Seed {seed:4d}:  mean={mean_sim:.3f}  std={std_sim:.3f}")

means = [r["mean"] for r in seed_summary]
spread = max(means) - min(means)
print(f"\n  Spread across seeds: {spread:.3f}")
if spread <= 0.05:
    print(f"  ✓ SEED STABLE — findings are reproducible")
else:
    print(f"  ⚠ Spread > 0.05 — some seed sensitivity detected")

print("\nPer-class breakdown:")
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    class_rows = results_df[results_df["true_label_name"] == name]
    sims = []
    for seed in [42, 123, 2024]:
        s = class_rows[class_rows["seed"] == seed]["cosine_similarity"].dropna().mean()
        sims.append(s)
    spread_class = max(sims) - min(sims)
    print(f"  {name:7s}  seed42={sims[0]:.3f}  "
          f"seed123={sims[1]:.3f}  "
          f"seed2024={sims[2]:.3f}  "
          f"spread={spread_class:.3f}")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/task2_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if "task2" in f and f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")

print(f"✓ Zip size: {os.path.getsize(zip_path)/1e6:.2f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/task2_results.zip')

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import copy
import random
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

# Load ROAD windows
road_df = pd.read_parquet(INPUT_DIR / "task4_road_windows.parquet")
print(f"✓ Loaded {len(road_df)} ROAD windows")
print(road_df["label_name"].value_counts().to_string())

embedder = SentenceTransformer("all-MiniLM-L6-v2")

SYSTEM_PROMPT = """You are a cybersecurity analyst specializing in 
Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_road_explanation(text, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"""Analyze this CAN bus telemetry 
sequence and classify it.

{text}

Provide your classification and detailed explanation in JSON format."""}
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

def perturb_p1(frames, seed=42):
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)
    target_idx = len(frames) // 2
    frame = perturbed[target_idx]
    byte_idx = rng.randint(0, 7)
    original_val = frame["data"][byte_idx]
    delta = rng.choice([-1, 1])
    frame["data"][byte_idx] = (original_val + delta) % 256
    return perturbed

def frames_to_text(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

# Run inference: baseline + P1 perturbed for each window
print(f"\nRunning ROAD inference...")
print(f"Estimated time: ~{len(road_df) * 2 * 6 / 3600:.1f} hours")

results = []
errors = 0

for idx, row in tqdm(road_df.iterrows(),
                      total=len(road_df),
                      desc="ROAD inference"):
    t0 = time.time()
    try:
        frames = json.loads(row["frames_json"])
        label_name = row["label_name"]

        # Baseline explanation
        resp_base = generate_road_explanation(row["text"])
        parsed_base, _ = safe_parse(resp_base)
        exp_base = parsed_base.get("explanation", "") if parsed_base else ""

        # P1 perturbed
        p1_frames = perturb_p1(frames, seed=42)
        p1_text = frames_to_text(p1_frames)
        resp_p1 = generate_road_explanation(p1_text)
        parsed_p1, _ = safe_parse(resp_p1)
        exp_p1 = parsed_p1.get("explanation", "") if parsed_p1 else ""

        sim = cosine_sim(exp_base, exp_p1)

        results.append({
            "label_name":        label_name,
            "baseline_exp":      exp_base,
            "p1_exp":            exp_p1,
            "baseline_class":    parsed_base.get("classification") if parsed_base else None,
            "p1_class":          parsed_p1.get("classification") if parsed_p1 else None,
            "cosine_similarity": sim,
            "json_valid":        parsed_base is not None and parsed_p1 is not None,
            "inference_time_s":  time.time() - t0,
        })

    except Exception as e:
        errors += 1
        results.append({
            "label_name":        row["label_name"],
            "baseline_exp":      "",
            "p1_exp":            "",
            "baseline_class":    None,
            "p1_class":          None,
            "cosine_similarity": None,
            "json_valid":        False,
            "inference_time_s":  time.time() - t0,
        })

    if len(results) % 50 == 0:
        cp = pd.DataFrame(results)
        cp.to_parquet(
            f"/kaggle/working/task4_checkpoint_{len(results)}.parquet"
        )
        print(f"  Checkpoint: {len(results)}")

print(f"\n✓ Complete: {len(results)}, Errors: {errors}")

# Summary
results_df = pd.DataFrame(results)
results_df.to_parquet("/kaggle/working/task4_road_results.parquet",
                       index=False)

print("\nROAD DATASET DRIFT RESULTS")
print("="*60)
print("Comparing against HCRL P1 results:")
print()

hcrl_p1_mean = 0.738  # from your earlier results

for name in ["AMBIENT", "ACCELERATOR", "FUZZING",
             "SPEEDOMETER", "CORRELATED"]:
    subset = results_df[results_df["label_name"] == name]
    sims = subset["cosine_similarity"].dropna()
    if len(sims) > 0:
        print(f"  {name:15s}  mean={sims.mean():.3f}  "
              f"std={sims.std():.3f}  n={len(sims)}")

road_overall = results_df["cosine_similarity"].dropna().mean()
print(f"\n  ROAD overall mean:  {road_overall:.3f}")
print(f"  HCRL P1 mean:       {hcrl_p1_mean:.3f}")
print(f"  Difference:         {road_overall - hcrl_p1_mean:+.3f}")

if abs(road_overall - hcrl_p1_mean) < 0.05:
    print(f"\n  ✓ CONSISTENT — drift pattern holds across datasets")
elif road_overall < hcrl_p1_mean:
    print(f"\n  ✓ ROAD shows MORE drift than HCRL")
    print(f"    Realistic attacks cause greater explanation instability")
else:
    print(f"\n  ~ ROAD shows LESS drift than HCRL")
    print(f"    More complex traffic may stabilize explanations")

In [ ]:
print(f"Model loaded: {qwen_model is not None}")
print(f"Device: {device}")

In [ ]:
!pip install -q --upgrade huggingface_hub
import importlib
import huggingface_hub
importlib.reload(huggingface_hub)
print(f"huggingface_hub version: {huggingface_hub.__version__}")

In [ ]:
!pip install -q huggingface_hub==0.24.0

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import copy
import random
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

# Load ROAD windows
road_df = pd.read_parquet(INPUT_DIR / "task4_road_windows.parquet")
print(f"✓ Loaded {len(road_df)} ROAD windows")
print(road_df["label_name"].value_counts().to_string())

embedder = SentenceTransformer("all-MiniLM-L6-v2")

SYSTEM_PROMPT = """You are a cybersecurity analyst specializing in 
Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_road_explanation(text, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"""Analyze this CAN bus telemetry 
sequence and classify it.

{text}

Provide your classification and detailed explanation in JSON format."""}
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

def perturb_p1(frames, seed=42):
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)
    target_idx = len(frames) // 2
    frame = perturbed[target_idx]
    byte_idx = rng.randint(0, 7)
    original_val = frame["data"][byte_idx]
    delta = rng.choice([-1, 1])
    frame["data"][byte_idx] = (original_val + delta) % 256
    return perturbed

def frames_to_text(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

# Run inference: baseline + P1 perturbed for each window
print(f"\nRunning ROAD inference...")
print(f"Estimated time: ~{len(road_df) * 2 * 6 / 3600:.1f} hours")

results = []
errors = 0

for idx, row in tqdm(road_df.iterrows(),
                      total=len(road_df),
                      desc="ROAD inference"):
    t0 = time.time()
    try:
        frames = json.loads(row["frames_json"])
        label_name = row["label_name"]

        # Baseline explanation
        resp_base = generate_road_explanation(row["text"])
        parsed_base, _ = safe_parse(resp_base)
        exp_base = parsed_base.get("explanation", "") if parsed_base else ""

        # P1 perturbed
        p1_frames = perturb_p1(frames, seed=42)
        p1_text = frames_to_text(p1_frames)
        resp_p1 = generate_road_explanation(p1_text)
        parsed_p1, _ = safe_parse(resp_p1)
        exp_p1 = parsed_p1.get("explanation", "") if parsed_p1 else ""

        sim = cosine_sim(exp_base, exp_p1)

        results.append({
            "label_name":        label_name,
            "baseline_exp":      exp_base,
            "p1_exp":            exp_p1,
            "baseline_class":    parsed_base.get("classification") if parsed_base else None,
            "p1_class":          parsed_p1.get("classification") if parsed_p1 else None,
            "cosine_similarity": sim,
            "json_valid":        parsed_base is not None and parsed_p1 is not None,
            "inference_time_s":  time.time() - t0,
        })

    except Exception as e:
        errors += 1
        results.append({
            "label_name":        row["label_name"],
            "baseline_exp":      "",
            "p1_exp":            "",
            "baseline_class":    None,
            "p1_class":          None,
            "cosine_similarity": None,
            "json_valid":        False,
            "inference_time_s":  time.time() - t0,
        })

    if len(results) % 50 == 0:
        cp = pd.DataFrame(results)
        cp.to_parquet(
            f"/kaggle/working/task4_checkpoint_{len(results)}.parquet"
        )
        print(f"  Checkpoint: {len(results)}")

print(f"\n✓ Complete: {len(results)}, Errors: {errors}")

# Summary
results_df = pd.DataFrame(results)
results_df.to_parquet("/kaggle/working/task4_road_results.parquet",
                       index=False)

print("\nROAD DATASET DRIFT RESULTS")
print("="*60)
print("Comparing against HCRL P1 results:")
print()

hcrl_p1_mean = 0.738  # from your earlier results

for name in ["AMBIENT", "ACCELERATOR", "FUZZING",
             "SPEEDOMETER", "CORRELATED"]:
    subset = results_df[results_df["label_name"] == name]
    sims = subset["cosine_similarity"].dropna()
    if len(sims) > 0:
        print(f"  {name:15s}  mean={sims.mean():.3f}  "
              f"std={sims.std():.3f}  n={len(sims)}")

road_overall = results_df["cosine_similarity"].dropna().mean()
print(f"\n  ROAD overall mean:  {road_overall:.3f}")
print(f"  HCRL P1 mean:       {hcrl_p1_mean:.3f}")
print(f"  Difference:         {road_overall - hcrl_p1_mean:+.3f}")

if abs(road_overall - hcrl_p1_mean) < 0.05:
    print(f"\n  ✓ CONSISTENT — drift pattern holds across datasets")
elif road_overall < hcrl_p1_mean:
    print(f"\n  ✓ ROAD shows MORE drift than HCRL")
    print(f"    Realistic attacks cause greater explanation instability")
else:
    print(f"\n  ~ ROAD shows LESS drift than HCRL")
    print(f"    More complex traffic may stabilize explanations")

In [ ]:
# Run P5 semantic steering on ROAD windows
ROAD_STEERING = "Note: Pay particular attention to inter-frame timing irregularities and arrival interval patterns."

print("Running P5 steering on ROAD windows...")
p5_results = []
errors = 0

for idx, row in tqdm(road_df.iterrows(),
                      total=len(road_df),
                      desc="ROAD P5"):
    t0 = time.time()
    try:
        label_name = row["label_name"]

        # Baseline explanation (reuse from results_df if available)
        existing = results_df[results_df.index == idx]
        if len(existing) > 0 and existing.iloc[0]["baseline_exp"]:
            exp_base = existing.iloc[0]["baseline_exp"]
        else:
            resp_base = generate_road_explanation(row["text"])
            parsed_base, _ = safe_parse(resp_base)
            exp_base = parsed_base.get("explanation", "") if parsed_base else ""

        # P5 steered explanation
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"""Analyze this CAN bus telemetry
sequence and classify it.

{ROAD_STEERING}

{row['text']}

Provide your classification and detailed explanation in JSON format."""}
        ]
        formatted = qwen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = qwen_tokenizer([formatted], return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False,
                pad_token_id=qwen_tokenizer.eos_token_id,
            )
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        resp_p5 = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        parsed_p5, _ = safe_parse(resp_p5)
        exp_p5 = parsed_p5.get("explanation", "") if parsed_p5 else ""

        sim = cosine_sim(exp_base, exp_p5)

        p5_results.append({
            "label_name":        label_name,
            "baseline_exp":      exp_base,
            "p5_exp":            exp_p5,
            "cosine_similarity": sim,
            "json_valid":        parsed_p5 is not None,
            "inference_time_s":  time.time() - t0,
        })

    except Exception as e:
        errors += 1
        p5_results.append({
            "label_name":        row["label_name"],
            "baseline_exp":      "",
            "p5_exp":            "",
            "cosine_similarity": None,
            "json_valid":        False,
            "inference_time_s":  time.time() - t0,
        })

    if len(p5_results) % 50 == 0:
        cp = pd.DataFrame(p5_results)
        cp.to_parquet(
            f"/kaggle/working/task4_p5_checkpoint_{len(p5_results)}.parquet"
        )
        print(f"  Checkpoint: {len(p5_results)}")

print(f"\n✓ P5 complete: {len(p5_results)}, Errors: {errors}")

p5_df = pd.DataFrame(p5_results)
p5_df.to_parquet("/kaggle/working/task4_road_p5_results.parquet", index=False)

print("\nROAD P5 STEERING RESULTS")
print("="*60)
hcrl_p5_mean = 0.753

for name in ["AMBIENT", "ACCELERATOR", "FUZZING", "SPEEDOMETER", "CORRELATED"]:
    subset = p5_df[p5_df["label_name"] == name]
    sims = subset["cosine_similarity"].dropna()
    if len(sims) > 0:
        print(f"  {name:15s}  mean={sims.mean():.3f}  std={sims.std():.3f}")

road_p5_mean = p5_df["cosine_similarity"].dropna().mean()
print(f"\n  ROAD P5 mean:   {road_p5_mean:.3f}")
print(f"  HCRL P5 mean:   {hcrl_p5_mean:.3f}")
print(f"  Difference:     {road_p5_mean - hcrl_p5_mean:+.3f}")

if abs(road_p5_mean - hcrl_p5_mean) < 0.08:
    print(f"\n  ✓ CONSISTENT — P5 drift pattern holds across datasets")
elif road_p5_mean < hcrl_p5_mean:
    print(f"\n  ✓ ROAD shows MORE P5 drift than HCRL")
else:
    print(f"\n  ~ ROAD shows LESS P5 drift than HCRL")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/task4_all_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if "task4" in f and f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")

print(f"✓ Size: {os.path.getsize(zip_path)/1e6:.2f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/task4_all_results.zip')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading {MODEL_ID}...")
t0 = time.time()

tiny_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tiny_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
tiny_model.eval()

print(f"✓ TinyLlama loaded in {time.time()-t0:.1f}s")
print(f"  Parameters: {sum(p.numel() for p in tiny_model.parameters())/1e9:.2f}B")
print(f"  Device map: {tiny_model.hf_device_map if hasattr(tiny_model, 'hf_device_map') else 'auto'}")

In [ ]:
print(list(p1_windows.columns))
print(f"\nSample baseline_explanation: {p1_windows['baseline_explanation'].iloc[0][:100]}")
print(f"Sample perturbed_text: {p1_windows['perturbed_text'].iloc[0][:100]}")

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

perturbed_dataset = pd.read_parquet(
    INPUT_DIR / "section10_perturbed_dataset.parquet"
)

print(f"Rows: {len(perturbed_dataset)}")
print(f"Columns: {list(perturbed_dataset.columns)}")
print(f"\nPerturbation types: {perturbed_dataset['perturbation_type'].unique()}")
print(f"\nSample P1 row:")
p1_sample_row = perturbed_dataset[
    perturbed_dataset["perturbation_type"] == "P1_PAYLOAD"
].iloc[0]
print(f"  baseline_explanation: {str(p1_sample_row['baseline_explanation'])[:100]}")
print(f"  perturbed_text: {str(p1_sample_row['perturbed_text'])[:100]}")

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

# Load data
perturbed_dataset = pd.read_parquet(
    INPUT_DIR / "section10_perturbed_dataset.parquet"
)
classifier_results = pd.read_parquet(
    INPUT_DIR / "section9a_classifier_results.parquet"
)

# Build lookup: window_idx → original text
idx_to_text = classifier_results["text"].to_dict()

# Get P1 and P5 windows
p1_windows = perturbed_dataset[
    perturbed_dataset["perturbation_type"] == "P1_PAYLOAD"
].reset_index(drop=True)

p5_windows = perturbed_dataset[
    perturbed_dataset["perturbation_type"] == "P5_STEERING"
].reset_index(drop=True)

# Add original text
p1_windows["original_text"] = p1_windows["window_idx"].map(idx_to_text)
p5_windows["original_text"] = p5_windows["window_idx"].map(idx_to_text)

print(f"✓ P1 windows: {len(p1_windows)}")
print(f"✓ P5 windows: {len(p5_windows)}")
print(f"✓ Original text available P1: {p1_windows['original_text'].notna().sum()}")
print(f"✓ Original text available P5: {p5_windows['original_text'].notna().sum()}")

# Sample 20 per class = 100 per perturbation type
SAMPLES_PER_CLASS = 20
np.random.seed(42)

def sample_balanced(df, n_per_class):
    samples = []
    for label in sorted(df["true_label_name"].unique()):
        class_df = df[df["true_label_name"] == label]
        n = min(n_per_class, len(class_df))
        samples.append(class_df.sample(n=n, random_state=42))
    return pd.concat(samples).reset_index(drop=True)

p1_sample = sample_balanced(p1_windows, SAMPLES_PER_CLASS)
p5_sample = sample_balanced(p5_windows, SAMPLES_PER_CLASS)

print(f"\nSampled:")
print(f"  P1: {len(p1_sample)} — {p1_sample['true_label_name'].value_counts().to_dict()}")
print(f"  P5: {len(p5_sample)} — {p5_sample['true_label_name'].value_counts().to_dict()}")

# Load embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Embedder ready")

# TinyLlama system prompt
TINY_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
Analyze the given CAN bus frames and classify the traffic.
Respond ONLY in valid JSON format:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float 0.0-1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>"],
    "temporal_reasoning": "<pattern observations>"
}"""

def generate_tiny(text, steering=""):
    """Generate TinyLlama explanation for a telemetry window."""
    user_content = "Analyze this CAN bus telemetry and classify it."
    if steering:
        user_content += f"\n\n{steering}"
    user_content += f"\n\n{text}\n\nRespond in JSON format only."

    messages = [
        {"role": "system", "content": TINY_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = tiny_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tiny_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = tiny_model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tiny_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tiny_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except:
        return None, "parse_error"

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

def run_inference(sample_df, ptype, label):
    """Run TinyLlama on baseline and perturbed windows."""
    results = []
    errors = 0

    for idx, row in tqdm(sample_df.iterrows(),
                          total=len(sample_df),
                          desc=f"TinyLlama {label}"):
        t0 = time.time()
        try:
            original_text = row["original_text"]
            perturbed_text = row["perturbed_text"]
            steering = row.get("steering_instruction", "")

            # Baseline — TinyLlama on original unperturbed text
            resp_base = generate_tiny(original_text, steering="")
            parsed_base, _ = safe_parse(resp_base)
            exp_base = parsed_base.get("explanation", "") if parsed_base else ""

            # Perturbed — TinyLlama on perturbed text (P1) or with steering (P5)
            if ptype == "P5_STEERING":
                resp_pert = generate_tiny(original_text, steering=steering)
            else:
                resp_pert = generate_tiny(perturbed_text, steering="")
            parsed_pert, _ = safe_parse(resp_pert)
            exp_pert = parsed_pert.get("explanation", "") if parsed_pert else ""

            sim = cosine_sim(exp_base, exp_pert)

            results.append({
                "window_idx":        row["window_idx"],
                "true_label_name":   row["true_label_name"],
                "perturbation_type": ptype,
                "baseline_exp":      exp_base,
                "perturbed_exp":     exp_pert,
                "baseline_class":    parsed_base.get("classification") if parsed_base else None,
                "perturbed_class":   parsed_pert.get("classification") if parsed_pert else None,
                "cosine_similarity": sim,
                "json_valid":        parsed_base is not None and parsed_pert is not None,
                "inference_time_s":  time.time() - t0,
            })

        except Exception as e:
            errors += 1
            results.append({
                "window_idx":        row["window_idx"],
                "true_label_name":   row["true_label_name"],
                "perturbation_type": ptype,
                "baseline_exp":      "",
                "perturbed_exp":     "",
                "baseline_class":    None,
                "perturbed_class":   None,
                "cosine_similarity": None,
                "json_valid":        False,
                "inference_time_s":  time.time() - t0,
            })

        if len(results) % 25 == 0:
            cp = pd.DataFrame(results)
            cp.to_parquet(
                f"/kaggle/working/task5_{label}_checkpoint_{len(results)}.parquet"
            )
            print(f"  Checkpoint {label}: {len(results)}")

    print(f"\n✓ {label} complete: {len(results)}, Errors: {errors}")
    print(f"  JSON valid: {sum(r['json_valid'] for r in results)}/{len(results)}")
    return pd.DataFrame(results)

# Run P1
print("\nRunning TinyLlama P1 (100 windows × 2 explanations)...")
print(f"Estimated: ~{100 * 2 * 5 / 3600:.1f} hours")
p1_results_df = run_inference(p1_sample, "P1_PAYLOAD", "P1")
p1_results_df.to_parquet("/kaggle/working/task5_tinyllama_p1.parquet", index=False)

# Run P5
print("\nRunning TinyLlama P5 (100 windows × 2 explanations)...")
p5_results_df = run_inference(p5_sample, "P5_STEERING", "P5")
p5_results_df.to_parquet("/kaggle/working/task5_tinyllama_p5.parquet", index=False)

# Results summary
print("\n" + "="*60)
print("TINYLLAMA vs QWEN DRIFT COMPARISON")
print("="*60)

qwen_p1_mean = 0.738
qwen_p5_mean = 0.753

tiny_p1_mean = p1_results_df["cosine_similarity"].dropna().mean()
tiny_p5_mean = p5_results_df["cosine_similarity"].dropna().mean()

print(f"\n{'Perturbation':<15} {'Qwen2.5-1.5B':>15} {'TinyLlama-1.1B':>15} {'Difference':>12}")
print("-"*60)
print(f"{'P1 Payload':<15} {qwen_p1_mean:>15.3f} {tiny_p1_mean:>15.3f} "
      f"{tiny_p1_mean - qwen_p1_mean:>+12.3f}")
print(f"{'P5 Steering':<15} {qwen_p5_mean:>15.3f} {tiny_p5_mean:>15.3f} "
      f"{tiny_p5_mean - qwen_p5_mean:>+12.3f}")

print(f"\nPer-class P1:")
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    tiny = p1_results_df[
        p1_results_df["true_label_name"] == name
    ]["cosine_similarity"].dropna().mean()
    print(f"  {name:7s}  TinyLlama={tiny:.3f}")

print(f"\nPer-class P5:")
for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    tiny = p5_results_df[
        p5_results_df["true_label_name"] == name
    ]["cosine_similarity"].dropna().mean()
    print(f"  {name:7s}  TinyLlama={tiny:.3f}")

if abs(tiny_p1_mean - qwen_p1_mean) < 0.10 and \
   abs(tiny_p5_mean - qwen_p5_mean) < 0.10:
    print(f"\n✓ CONSISTENT — drift pattern holds across models")
    print(f"  Finding is not Qwen-specific")
elif tiny_p1_mean > qwen_p1_mean + 0.10:
    print(f"\n~ TinyLlama MORE stable than Qwen")
    print(f"  Model architecture affects explanation stability")
else:
    print(f"\n~ TinyLlama LESS stable than Qwen")
    print(f"  Smaller/different models may be more vulnerable")

In [ ]:
# Load P1 checkpoint and look at raw responses
import pandas as pd
from pathlib import Path

# Load latest checkpoint
cp = pd.read_parquet("/kaggle/working/task5_p1_checkpoint_100.parquet")

print("Sample raw baseline responses:")
print("="*60)
for i in range(3):
    row = cp.iloc[i]
    print(f"\nWindow {i+1} — {row['true_label_name']}")
    print(f"Baseline exp: '{str(row['baseline_exp'])[:200]}'")
    print(f"Perturbed exp: '{str(row['perturbed_exp'])[:200]}'")
    print("-"*40)

In [ ]:
import os
files = [f for f in os.listdir("/kaggle/working") if "task5" in f]
print("Task 5 files:")
for f in sorted(files):
    print(f"  {f} ({os.path.getsize(f'/kaggle/working/{f}')/1e3:.1f} KB)")

In [ ]:
cp = pd.read_parquet("/kaggle/working/task5_P1_checkpoint_100.parquet")

print(f"Rows: {len(cp)}")
print(f"Columns: {list(cp.columns)}")
print()
print("Sample raw responses:")
print("="*60)
for i in range(3):
    row = cp.iloc[i]
    print(f"\nWindow {i+1} — {row['true_label_name']}")
    print(f"Baseline exp: '{str(row['baseline_exp'])[:300]}'")
    print(f"Perturbed exp: '{str(row['perturbed_exp'])[:300]}'")
    print("-"*40)

In [ ]:
# We need to re-run a single window to see raw TinyLlama output
sample = p1_sample.iloc[0]

response = generate_tiny(sample["original_text"], steering="")
print("RAW TINYLLAMA RESPONSE:")
print("="*60)
print(response)
print("="*60)
print(f"\nLength: {len(response)} chars")

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
print(f"Loading {MODEL_ID}...")
t0 = time.time()

mistral_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
mistral_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
mistral_model.eval()

print(f"✓ Loaded in {time.time()-t0:.1f}s")
print(f"  Parameters: {sum(p.numel() for p in mistral_model.parameters())/1e9:.2f}B")
print(f"  Device map: {mistral_model.hf_device_map}")

# Quick JSON compliance test
messages = [
    {"role": "user", "content": """You are a CAN bus security analyst.
Analyze this single CAN frame and respond in valid JSON:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float>,
    "explanation": "<reasoning>"
}

Frame: ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00"""}
]

text = mistral_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = mistral_tokenizer([text], return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=mistral_tokenizer.eos_token_id,
    )
new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = mistral_tokenizer.decode(new_tokens, skip_special_tokens=True)
print(f"\nJSON test response:")
print(response)

Loading mistralai/Mistral-7B-Instruct-v0.2...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✓ Loaded in 84.1s
  Parameters: 7.24B
  Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'lm_head': 1}


2026-06-08 03:25:25.351036: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780889125.903707      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780889126.012315      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780889126.996289      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780889126.996317      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780889126.996319      57 computation_placer.cc:177] computation placer alr


JSON test response:
{
    "classification": "NORMAL",
    "confidence": 1.0,
    "explanation": "This frame has an ID of 0x0000 and a DLC of 8. The data field is all zeros. This is a valid CAN frame with no suspicious activity. It is likely a broadcast message with no meaningful data."


In [ ]:
import pandas as pd
import numpy as np
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

# Load data
perturbed_dataset = pd.read_parquet(
    INPUT_DIR / "section10_perturbed_dataset.parquet"
)
classifier_results = pd.read_parquet(
    INPUT_DIR / "section9a_classifier_results.parquet"
)

# Build lookup
idx_to_text = classifier_results["text"].to_dict()

# Get P1 and P5 windows
p1_windows = perturbed_dataset[
    perturbed_dataset["perturbation_type"] == "P1_PAYLOAD"
].reset_index(drop=True)
p5_windows = perturbed_dataset[
    perturbed_dataset["perturbation_type"] == "P5_STEERING"
].reset_index(drop=True)

# Add original unperturbed text
p1_windows["original_text"] = p1_windows["window_idx"].map(idx_to_text)
p5_windows["original_text"] = p5_windows["window_idx"].map(idx_to_text)

# Sample 20 per class = 100 per perturbation type
SAMPLES_PER_CLASS = 20
np.random.seed(42)

def sample_balanced(df, n):
    samples = []
    for label in sorted(df["true_label_name"].unique()):
        class_df = df[df["true_label_name"] == label]
        samples.append(
            class_df.sample(n=min(n, len(class_df)), random_state=42)
        )
    return pd.concat(samples).reset_index(drop=True)

p1_sample = sample_balanced(p1_windows, SAMPLES_PER_CLASS)
p5_sample = sample_balanced(p5_windows, SAMPLES_PER_CLASS)

print(f"✓ P1 sample: {len(p1_sample)} windows")
print(f"✓ P5 sample: {len(p5_sample)} windows")
print(f"  Classes: {p1_sample['true_label_name'].value_counts().to_dict()}")

# Load embedder — same as Qwen experiment
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Embedder ready")

# ================================================================
# MISTRAL SYSTEM PROMPT
# Identical content to Qwen system prompt
# ================================================================
MISTRAL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_mistral(text, securebert_pred, securebert_conf, steering=""):
    """
    Generate Mistral explanation under identical conditions to Qwen:
    - Same system prompt content
    - Same SecureBERT confidence included in prompt
    - Same steering instruction handling
    - Same greedy decoding (do_sample=False)
    - Same max_new_tokens=512
    """
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {securebert_pred} (confidence: {securebert_conf:.3f})
"""
    if steering:
        user_content += f"\n{steering}\n"

    user_content += f"""
{text}

Provide your classification and detailed explanation in the required JSON format."""

    # Mistral uses [INST] format — system merged into user message
    messages = [
        {"role": "user",
         "content": f"{MISTRAL_SYSTEM}\n\n{user_content}"}
    ]

    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=1024  # increased to match window size
    ).to(device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=512,       # same as Qwen
            temperature=0.1,          # same as Qwen
            do_sample=False,          # greedy decoding — same as Qwen
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except:
        return None, "parse_error"


def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))


def run_mistral_inference(sample_df, ptype, label):
    """
    Run Mistral inference under identical conditions to Qwen:
    - Baseline: original unperturbed text + securebert pred/conf
    - Perturbed P1: mutated text + securebert pred/conf
    - Perturbed P5: original text + steering + securebert pred/conf
    """
    results = []
    errors = 0

    for idx, row in tqdm(sample_df.iterrows(),
                          total=len(sample_df),
                          desc=f"Mistral {label}"):
        t0 = time.time()
        try:
            original_text    = row["original_text"]
            securebert_pred  = row["securebert_pred_name"]
            securebert_conf  = float(row["securebert_confidence"])
            prompt_conf      = float(row["securebert_conf_for_prompt"])
            steering         = row.get("steering_instruction", "")

            # Baseline — original text, full confidence
            resp_base = generate_mistral(
                original_text, securebert_pred, securebert_conf,
                steering=""
            )
            parsed_base, _ = safe_parse(resp_base)
            exp_base = parsed_base.get("explanation", "") \
                if parsed_base else ""

            # Perturbed
            if ptype == "P5_STEERING":
                # P5: same text, same conf, add steering
                resp_pert = generate_mistral(
                    original_text, securebert_pred, prompt_conf,
                    steering=steering
                )
            else:
                # P1: mutated text, same conf
                resp_pert = generate_mistral(
                    row["perturbed_text"], securebert_pred, prompt_conf,
                    steering=""
                )
            parsed_pert, _ = safe_parse(resp_pert)
            exp_pert = parsed_pert.get("explanation", "") \
                if parsed_pert else ""

            sim = cosine_sim(exp_base, exp_pert)

            results.append({
                "window_idx":        row["window_idx"],
                "true_label_name":   row["true_label_name"],
                "perturbation_type": ptype,
                "baseline_exp":      exp_base,
                "perturbed_exp":     exp_pert,
                "baseline_class":    parsed_base.get("classification") \
                    if parsed_base else None,
                "perturbed_class":   parsed_pert.get("classification") \
                    if parsed_pert else None,
                "cosine_similarity": sim,
                "json_valid":        parsed_base is not None \
                    and parsed_pert is not None,
                "inference_time_s":  time.time() - t0,
            })

        except Exception as e:
            errors += 1
            results.append({
                "window_idx":        row["window_idx"],
                "true_label_name":   row["true_label_name"],
                "perturbation_type": ptype,
                "baseline_exp":      "",
                "perturbed_exp":     "",
                "baseline_class":    None,
                "perturbed_class":   None,
                "cosine_similarity": None,
                "json_valid":        False,
                "inference_time_s":  time.time() - t0,
            })

        if len(results) % 25 == 0:
            cp = pd.DataFrame(results)
            cp.to_parquet(
                f"/kaggle/working/task5_mistral_{label}"
                f"_checkpoint_{len(results)}.parquet"
            )
            print(f"  Checkpoint {label}: {len(results)}")

    print(f"\n✓ {label} complete: {len(results)}, Errors: {errors}")
    print(f"  JSON valid: "
          f"{sum(r['json_valid'] for r in results)}/{len(results)}")
    return pd.DataFrame(results)


# ================================================================
# RUN P1
# ================================================================
print("\nRunning Mistral-7B P1 (100 windows × 2 explanations)...")
print(f"Estimated: ~{100 * 2 * 12 / 3600:.1f} hours")
p1_df = run_mistral_inference(p1_sample, "P1_PAYLOAD", "P1")
p1_df.to_parquet("/kaggle/working/task5_mistral_p1.parquet", index=False)

# ================================================================
# RUN P5
# ================================================================
print("\nRunning Mistral-7B P5 (100 windows × 2 explanations)...")
p5_df = run_mistral_inference(p5_sample, "P5_STEERING", "P5")
p5_df.to_parquet("/kaggle/working/task5_mistral_p5.parquet", index=False)

# ================================================================
# SUMMARY
# ================================================================
print("\n" + "="*65)
print("MISTRAL-7B vs QWEN2.5-1.5B — IDENTICAL CONDITIONS")
print("="*65)
print("Same windows, same perturbations, same prompt content,")
print("same SecureBERT confidence, same greedy decoding.")
print()

qwen_p1 = 0.738
qwen_p5 = 0.753
mistral_p1 = p1_df["cosine_similarity"].dropna().mean()
mistral_p5 = p5_df["cosine_similarity"].dropna().mean()

print(f"{'Perturbation':<15} {'Qwen2.5-1.5B':>15} "
      f"{'Mistral-7B':>12} {'Difference':>12}")
print("-"*57)
print(f"{'P1 Payload':<15} {qwen_p1:>15.3f} "
      f"{mistral_p1:>12.3f} {mistral_p1-qwen_p1:>+12.3f}")
print(f"{'P5 Steering':<15} {qwen_p5:>15.3f} "
      f"{mistral_p5:>12.3f} {mistral_p5-qwen_p5:>+12.3f}")

print(f"\nPer-class breakdown:")
print(f"{'Class':<10} {'Qwen P1':>10} {'Mistral P1':>12} "
      f"{'Qwen P5':>10} {'Mistral P5':>12}")
print("-"*57)

qwen_p1_class = {
    "NORMAL": 0.895, "DOS": 0.737, "FUZZY": 0.820,
    "GEAR": 0.785, "RPM": 0.469
}
qwen_p5_class = {
    "NORMAL": 0.847, "DOS": 0.807, "FUZZY": 0.812,
    "GEAR": 0.744, "RPM": 0.558
}

for name in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    mp1 = p1_df[p1_df["true_label_name"]==name][
        "cosine_similarity"].dropna().mean()
    mp5 = p5_df[p5_df["true_label_name"]==name][
        "cosine_similarity"].dropna().mean()
    print(f"  {name:7s}  {qwen_p1_class[name]:>10.3f} "
          f"{mp1:>12.3f} {qwen_p5_class[name]:>10.3f} {mp5:>12.3f}")

print()
if abs(mistral_p1-qwen_p1) < 0.10 and abs(mistral_p5-qwen_p5) < 0.10:
    print("✓ CONSISTENT — explanation drift holds across model families")
    print("  Qwen2.5-1.5B and Mistral-7B show similar drift patterns")
    print("  Finding is not architecture-specific")
elif mistral_p1 > qwen_p1 + 0.10:
    print("~ Mistral MORE stable than Qwen")
    print("  Larger model (7B) shows reduced explanation drift")
    print("  Scale may partially mitigate explanation instability")
else:
    print("~ Mistral LESS stable than Qwen")
    print("  Different architecture shows higher vulnerability")

# Save zip for download
import zipfile, os
zip_path = "/kaggle/working/task5_mistral_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if "mistral" in f and f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")
print(f"✓ Zip: {os.path.getsize(zip_path)/1e6:.2f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/task5_mistral_results.zip')

In [ ]:
# Check what Mistral actually produced
cp = pd.read_parquet("/kaggle/working/task5_mistral_P1_checkpoint_100.parquet")

print(f"JSON valid: {cp['json_valid'].sum()}/100")
print(f"\nSample baseline responses (first 3):")
for i in range(3):
    row = cp.iloc[i]
    print(f"\nWindow {i+1} — {row['true_label_name']}")
    print(f"Baseline exp: '{str(row['baseline_exp'])[:300]}'")
    print(f"Perturbed exp: '{str(row['perturbed_exp'])[:300]}'")
    print("-"*40)

# Check if explanations exist even without valid JSON
has_baseline = cp['baseline_exp'].apply(lambda x: len(str(x)) > 10).sum()
has_perturbed = cp['perturbed_exp'].apply(lambda x: len(str(x)) > 10).sum()
print(f"\nNon-empty baseline explanations: {has_baseline}/100")
print(f"Non-empty perturbed explanations: {has_perturbed}/100")

In [ ]:
# Check token lengths for Mistral inputs
sample_row = p1_sample.iloc[0]

user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {sample_row['securebert_pred_name']} (confidence: {sample_row['securebert_confidence']:.3f})

{sample_row['original_text']}

Provide your classification and detailed explanation in the required JSON format."""

messages = [
    {"role": "user", "content": f"{MISTRAL_SYSTEM}\n\n{user_content}"}
]
formatted = mistral_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
tokens = mistral_tokenizer([formatted], return_tensors="pt")
print(f"Input token length: {tokens['input_ids'].shape[1]}")
print(f"Current max_length: 1024")
print(f"Tokens left for generation: {1024 - tokens['input_ids'].shape[1]}")

In [ ]:
# Test with increased max_length
sample_row = p1_sample.iloc[0]

user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {sample_row['securebert_pred_name']} (confidence: {sample_row['securebert_confidence']:.3f})

{sample_row['original_text']}

Provide your classification and detailed explanation in the required JSON format."""

messages = [
    {"role": "user", "content": f"{MISTRAL_SYSTEM}\n\n{user_content}"}
]
formatted = mistral_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# Increased max_length
inputs = mistral_tokenizer(
    [formatted], return_tensors="pt",
    truncation=True, max_length=2048
).to(device)

print(f"Input tokens: {inputs['input_ids'].shape[1]}")
print(f"Tokens available for generation: {2048 - inputs['input_ids'].shape[1]}")

with torch.no_grad():
    outputs = mistral_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        pad_token_id=mistral_tokenizer.eos_token_id,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = mistral_tokenizer.decode(new_tokens, skip_special_tokens=True)
print(f"\nResponse length: {len(response)} chars")
print(f"First 300 chars: {response[:300]}")

# Try parsing
parsed, err = safe_parse(response)
print(f"\nJSON valid: {parsed is not None}")
if parsed:
    print(f"Classification: {parsed.get('classification')}")
    print(f"Explanation: {parsed.get('explanation', '')[:150]}")

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

perturbed_dataset = pd.read_parquet(
    INPUT_DIR / "section10_perturbed_dataset.parquet"
)
classifier_results = pd.read_parquet(
    INPUT_DIR / "section9a_classifier_results.parquet"
)

idx_to_text = classifier_results["text"].to_dict()
perturbed_dataset["original_text"] = perturbed_dataset["window_idx"].map(
    idx_to_text
)

SAMPLES_PER_CLASS = 20
np.random.seed(42)

def sample_balanced(df, n):
    samples = []
    for label in sorted(df["true_label_name"].unique()):
        class_df = df[df["true_label_name"] == label]
        samples.append(
            class_df.sample(n=min(n, len(class_df)), random_state=42)
        )
    return pd.concat(samples).reset_index(drop=True)

all_samples = []
for ptype in ["P1_PAYLOAD", "P2_INSERTION", "P3_REORDER",
              "P4_PROMPT", "P5_STEERING"]:
    ptype_df = perturbed_dataset[
        perturbed_dataset["perturbation_type"] == ptype
    ]
    sampled = sample_balanced(ptype_df, SAMPLES_PER_CLASS)
    all_samples.append(sampled)
    print(f"  {ptype}: {len(sampled)} windows")

full_sample = pd.concat(all_samples).reset_index(drop=True)
print(f"\n✓ Total: {len(full_sample)} windows")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Embedder ready")

MISTRAL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_mistral(text, securebert_pred, securebert_conf, steering=""):
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {securebert_pred} (confidence: {securebert_conf:.3f})
"""
    if steering:
        user_content += f"\n{steering}\n"

    user_content += f"""
{text}

Provide your classification and detailed explanation in the required JSON format."""

    messages = [
        {"role": "user",
         "content": f"{MISTRAL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except:
        return None, "parse_error"

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

print(f"\nRunning Mistral-7B HCRL — all 5 perturbation types")
print(f"Total: {len(full_sample)} windows × 2 runs each")
print(f"Estimated: ~{len(full_sample) * 2 * 55 / 3600:.1f} hours")
print()

results = []
errors = 0

for idx, row in tqdm(full_sample.iterrows(),
                      total=len(full_sample),
                      desc="Mistral HCRL"):
    t0 = time.time()
    try:
        original_text   = row["original_text"]
        securebert_pred = row["securebert_pred_name"]
        securebert_conf = float(row["securebert_confidence"])
        prompt_conf     = float(row["securebert_conf_for_prompt"])
        steering        = row.get("steering_instruction", "")
        ptype           = row["perturbation_type"]

        # BASELINE
        resp_base = generate_mistral(
            original_text, securebert_pred, securebert_conf, steering=""
        )
        parsed_base, _ = safe_parse(resp_base)
        exp_base  = parsed_base.get("explanation", "") if parsed_base else ""
        cls_base  = parsed_base.get("classification") if parsed_base else None
        conf_base = parsed_base.get("confidence") if parsed_base else None

        # PERTURBED
        if ptype == "P4_PROMPT":
            resp_pert = generate_mistral(
                original_text, securebert_pred, prompt_conf, steering=""
            )
        elif ptype == "P5_STEERING":
            resp_pert = generate_mistral(
                original_text, securebert_pred, prompt_conf,
                steering=steering
            )
        else:
            resp_pert = generate_mistral(
                row["perturbed_text"], securebert_pred, prompt_conf,
                steering=""
            )
        parsed_pert, _ = safe_parse(resp_pert)
        exp_pert  = parsed_pert.get("explanation", "") if parsed_pert else ""
        cls_pert  = parsed_pert.get("classification") if parsed_pert else None
        conf_pert = parsed_pert.get("confidence") if parsed_pert else None

        sim = cosine_sim(exp_base, exp_pert)

        results.append({
            "window_idx":           row["window_idx"],
            "true_label_name":      row["true_label_name"],
            "perturbation_type":    ptype,
            "model":                "mistral-7b",
            "dataset":              "HCRL",
            "baseline_exp":         exp_base,
            "baseline_class":       cls_base,
            "baseline_conf":        conf_base,
            "baseline_json_valid":  parsed_base is not None,
            "perturbed_exp":        exp_pert,
            "perturbed_class":      cls_pert,
            "perturbed_conf":       conf_pert,
            "perturbed_json_valid": parsed_pert is not None,
            "cosine_similarity":    sim,
            "pred_stable":          cls_base == cls_pert
                if cls_base and cls_pert else None,
            "json_valid":           parsed_base is not None
                and parsed_pert is not None,
            "inference_time_s":     time.time() - t0,
        })

    except Exception as e:
        errors += 1
        results.append({
            "window_idx":           row["window_idx"],
            "true_label_name":      row["true_label_name"],
            "perturbation_type":    ptype,
            "model":                "mistral-7b",
            "dataset":              "HCRL",
            "baseline_exp":         "",
            "baseline_class":       None,
            "baseline_conf":        None,
            "baseline_json_valid":  False,
            "perturbed_exp":        "",
            "perturbed_class":      None,
            "perturbed_conf":       None,
            "perturbed_json_valid": False,
            "cosine_similarity":    None,
            "pred_stable":          None,
            "json_valid":           False,
            "inference_time_s":     time.time() - t0,
        })

    if len(results) % 50 == 0:
        cp = pd.DataFrame(results)
        cp.to_parquet(
            f"/kaggle/working/mistral_hcrl_checkpoint_{len(results)}.parquet"
        )
        valid_rate = sum(r["json_valid"] for r in results) / len(results)
        print(f"  Checkpoint {len(results)}: "
              f"JSON valid={valid_rate*100:.0f}%  Errors={errors}")

print(f"\n✓ Complete: {len(results)}, Errors: {errors}")
print(f"  JSON valid: {sum(r['json_valid'] for r in results)}/{len(results)}")

results_df = pd.DataFrame(results)
results_df.to_parquet(
    "/kaggle/working/mistral_hcrl_all_perturbations.parquet",
    index=False
)
size = Path("/kaggle/working/mistral_hcrl_all_perturbations.parquet"
            ).stat().st_size / 1e6
print(f"✓ Saved: mistral_hcrl_all_perturbations.parquet ({size:.2f} MB)")

In [ ]:
# SINGLE WINDOW VALIDATION TEST
# Run one window from each perturbation type before full inference
print("SINGLE WINDOW VALIDATION — 5 perturbation types")
print("="*60)

test_results = []

for ptype in ["P1_PAYLOAD", "P2_INSERTION", "P3_REORDER",
              "P4_PROMPT", "P5_STEERING"]:
    row = full_sample[
        full_sample["perturbation_type"] == ptype
    ].iloc[0]

    t0 = time.time()

    original_text   = row["original_text"]
    securebert_pred = row["securebert_pred_name"]
    securebert_conf = float(row["securebert_confidence"])
    prompt_conf     = float(row["securebert_conf_for_prompt"])
    steering        = row.get("steering_instruction", "")

    # BASELINE
    resp_base = generate_mistral(
        original_text, securebert_pred, securebert_conf, steering=""
    )
    parsed_base, _ = safe_parse(resp_base)
    exp_base = parsed_base.get("explanation", "") if parsed_base else ""

    # PERTURBED
    if ptype == "P4_PROMPT":
        resp_pert = generate_mistral(
            original_text, securebert_pred, prompt_conf, steering=""
        )
    elif ptype == "P5_STEERING":
        resp_pert = generate_mistral(
            original_text, securebert_pred, prompt_conf,
            steering=steering
        )
    else:
        resp_pert = generate_mistral(
            row["perturbed_text"], securebert_pred, prompt_conf,
            steering=""
        )
    parsed_pert, _ = safe_parse(resp_pert)
    exp_pert = parsed_pert.get("explanation", "") if parsed_pert else ""

    sim = cosine_sim(exp_base, exp_pert)
    elapsed = time.time() - t0

    base_valid = parsed_base is not None
    pert_valid = parsed_pert is not None

    print(f"\n{ptype}:")
    print(f"  Class:            {row['true_label_name']}")
    print(f"  Baseline valid:   {base_valid}")
    print(f"  Perturbed valid:  {pert_valid}")
    print(f"  Cosine sim:       {sim:.3f}" if sim else "  Cosine sim: None")
    print(f"  Baseline class:   {parsed_base.get('classification') if parsed_base else 'N/A'}")
    print(f"  Perturbed class:  {parsed_pert.get('classification') if parsed_pert else 'N/A'}")
    print(f"  Baseline exp:     {exp_base[:100]}...")
    print(f"  Perturbed exp:    {exp_pert[:100]}...")
    print(f"  Time:             {elapsed:.1f}s")

    test_results.append({
        "ptype": ptype,
        "base_valid": base_valid,
        "pert_valid": pert_valid,
        "sim": sim,
        "time": elapsed
    })

print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)
all_valid = all(r["base_valid"] and r["pert_valid"] for r in test_results)
avg_time  = np.mean([r["time"] for r in test_results])

for r in test_results:
    status = "✓" if r["base_valid"] and r["pert_valid"] else "✗"
    sim_str = f"{r['sim']:.3f}" if r["sim"] else "None"
    print(f"  {status} {r['ptype']:15s}  "
          f"sim={sim_str}  time={r['time']:.1f}s")

print()
print(f"All valid:    {all_valid}")
print(f"Avg time:     {avg_time:.1f}s per window pair")
print(f"Total est:    {500 * avg_time * 2 / 3600:.1f} hours for full run")
print()

if all_valid:
    print("✓ ALL CHECKS PASSED — safe to start full inference")
else:
    print("✗ SOME CHECKS FAILED — do not start full inference")
    print("  Fix issues above before proceeding")

In [ ]:
# Check actual explanation lengths being stored
print("EXPLANATION LENGTH CHECK")
print("="*60)

for ptype in ["P1_PAYLOAD", "P5_STEERING"]:
    row = full_sample[
        full_sample["perturbation_type"] == ptype
    ].iloc[0]

    original_text   = row["original_text"]
    securebert_pred = row["securebert_pred_name"]
    securebert_conf = float(row["securebert_confidence"])
    prompt_conf     = float(row["securebert_conf_for_prompt"])
    steering        = row.get("steering_instruction", "")

    # Baseline
    resp_base = generate_mistral(
        original_text, securebert_pred, securebert_conf, steering=""
    )
    parsed_base, _ = safe_parse(resp_base)
    exp_base = parsed_base.get("explanation", "") if parsed_base else ""

    # Perturbed
    if ptype == "P5_STEERING":
        resp_pert = generate_mistral(
            original_text, securebert_pred, prompt_conf,
            steering=steering
        )
    else:
        resp_pert = generate_mistral(
            row["perturbed_text"], securebert_pred, prompt_conf,
            steering=""
        )
    parsed_pert, _ = safe_parse(resp_pert)
    exp_pert = parsed_pert.get("explanation", "") if parsed_pert else ""

    print(f"\n{ptype} — {row['true_label_name']}")
    print(f"  Raw response length:      {len(resp_base)} chars")
    print(f"  Full baseline exp length: {len(exp_base)} chars")
    print(f"  Full perturbed exp length:{len(exp_pert)} chars")
    print(f"\n  FULL BASELINE EXPLANATION:")
    print(f"  {exp_base}")
    print(f"\n  FULL PERTURBED EXPLANATION:")
    print(f"  {exp_pert}")
    print(f"\n  Ends mid-sentence baseline: {not exp_base.endswith('.'  )}")
    print(f"  Ends mid-sentence perturbed: {not exp_pert.endswith('.')}")
    print("-"*60)

In [ ]:
# SEMANTIC DRIFT MEASUREMENT ON VALIDATION WINDOWS
print("SEMANTIC DRIFT — VALIDATION WINDOWS")
print("="*60)
print("Measuring drift on the 5 test windows before full run")
print()

validation_results = []

for ptype in ["P1_PAYLOAD", "P2_INSERTION", "P3_REORDER",
              "P4_PROMPT", "P5_STEERING"]:
    row = full_sample[
        full_sample["perturbation_type"] == ptype
    ].iloc[0]

    original_text   = row["original_text"]
    securebert_pred = row["securebert_pred_name"]
    securebert_conf = float(row["securebert_confidence"])
    prompt_conf     = float(row["securebert_conf_for_prompt"])
    steering        = row.get("steering_instruction", "")

    # Baseline
    resp_base = generate_mistral(
        original_text, securebert_pred, securebert_conf, steering=""
    )
    parsed_base, _ = safe_parse(resp_base)
    exp_base = parsed_base.get("explanation", "") if parsed_base else ""

    # Perturbed
    if ptype == "P4_PROMPT":
        resp_pert = generate_mistral(
            original_text, securebert_pred, prompt_conf, steering=""
        )
    elif ptype == "P5_STEERING":
        resp_pert = generate_mistral(
            original_text, securebert_pred, prompt_conf,
            steering=steering
        )
    else:
        resp_pert = generate_mistral(
            row["perturbed_text"], securebert_pred, prompt_conf,
            steering=""
        )
    parsed_pert, _ = safe_parse(resp_pert)
    exp_pert = parsed_pert.get("explanation", "") if parsed_pert else ""

    # Compute similarity
    sim = cosine_sim(exp_base, exp_pert)

    # Word overlap check
    base_words  = set(exp_base.lower().split())
    pert_words  = set(exp_pert.lower().split())
    word_overlap = len(base_words & pert_words) / \
                   len(base_words | pert_words) if base_words | pert_words else 0

    # Length change
    len_change = len(exp_pert) - len(exp_base)

    print(f"{ptype}:")
    print(f"  Class:           {row['true_label_name']}")
    print(f"  Cosine sim:      {sim:.3f}")
    print(f"  Word overlap:    {word_overlap:.3f}")
    print(f"  Length change:   {len_change:+d} chars "
          f"({'longer' if len_change > 0 else 'shorter'})")
    print(f"  Base class:      {parsed_base.get('classification') if parsed_base else 'N/A'}")
    print(f"  Pert class:      {parsed_pert.get('classification') if parsed_pert else 'N/A'}")
    print(f"  Pred stable:     {parsed_base.get('classification') == parsed_pert.get('classification') if parsed_base and parsed_pert else 'N/A'}")
    print()
    print(f"  BASELINE:  {exp_base[:200]}")
    print()
    print(f"  PERTURBED: {exp_pert[:200]}")
    print("-"*60)

    validation_results.append({
        "ptype":        ptype,
        "label":        row["true_label_name"],
        "cosine_sim":   sim,
        "word_overlap": word_overlap,
        "len_change":   len_change,
        "pred_stable":  parsed_base.get("classification") == \
                        parsed_pert.get("classification") \
                        if parsed_base and parsed_pert else None,
    })

print("\nSUMMARY")
print("="*60)
print(f"{'Perturbation':<20} {'Sim':>6} {'Overlap':>8} "
      f"{'Len Δ':>8} {'Stable':>8}")
print("-"*55)
for r in validation_results:
    print(f"  {r['ptype']:<18} {r['cosine_sim']:>6.3f} "
          f"{r['word_overlap']:>8.3f} {r['len_change']:>+8d} "
          f"{str(r['pred_stable']):>8}")

mean_sim = np.mean([r["cosine_sim"] for r in validation_results
                    if r["cosine_sim"]])
print(f"\n  Mean cosine similarity: {mean_sim:.3f}")
print(f"  Qwen mean (all 5 types): 0.771")
print(f"  Difference: {mean_sim - 0.771:+.3f}")

if mean_sim < 0.85:
    print(f"\n  ✓ Meaningful drift detected on validation windows")
    print(f"  ✓ Safe to proceed with full inference")
elif mean_sim < 0.95:
    print(f"\n  ~ Moderate drift detected")
    print(f"  ~ Proceed with full inference")
else:
    print(f"\n  ⚠ Low drift on validation windows")
    print(f"  ⚠ Investigate before full run")

In [ ]:
# EXTENDED VALIDATION — Qwen vs Mistral WITH VISUALIZATIONS
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

print("EXTENDED VALIDATION — QWEN vs MISTRAL")
print("All 5 classes × All 5 perturbation types")
print("="*70)

PTYPES  = ["P1_PAYLOAD", "P2_INSERTION", "P3_REORDER",
           "P4_PROMPT",  "P5_STEERING"]
CLASSES = ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]

QWEN_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_qwen(text, securebert_pred, securebert_conf, steering=""):
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {securebert_pred} (confidence: {securebert_conf:.3f})
"""
    if steering:
        user_content += f"\n{steering}\n"
    user_content += f"""
{text}

Provide your classification and detailed explanation in the required JSON format."""

    messages = [
        {"role": "system", "content": QWEN_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer([formatted], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

all_results = []

for ptype in PTYPES:
    print(f"\n--- {ptype} ---")
    for cls in CLASSES:
        subset = full_sample[
            (full_sample["perturbation_type"] == ptype) &
            (full_sample["true_label_name"] == cls)
        ]
        if len(subset) == 0:
            print(f"  {cls}: NO SAMPLE — skipping")
            continue

        row = subset.iloc[0]
        original_text   = row["original_text"]
        securebert_pred = row["securebert_pred_name"]
        securebert_conf = float(row["securebert_confidence"])
        prompt_conf     = float(row["securebert_conf_for_prompt"])
        steering        = row.get("steering_instruction", "")

        # ---- QWEN ----
        resp_q_base = generate_qwen(
            original_text, securebert_pred, securebert_conf, steering=""
        )
        parsed_q_base, _ = safe_parse(resp_q_base)
        exp_q_base = parsed_q_base.get("explanation", "") \
            if parsed_q_base else ""

        if ptype == "P4_PROMPT":
            resp_q_pert = generate_qwen(
                original_text, securebert_pred, prompt_conf, steering=""
            )
        elif ptype == "P5_STEERING":
            resp_q_pert = generate_qwen(
                original_text, securebert_pred, prompt_conf,
                steering=steering
            )
        else:
            resp_q_pert = generate_qwen(
                row["perturbed_text"], securebert_pred, prompt_conf,
                steering=""
            )
        parsed_q_pert, _ = safe_parse(resp_q_pert)
        exp_q_pert = parsed_q_pert.get("explanation", "") \
            if parsed_q_pert else ""
        sim_qwen = cosine_sim(exp_q_base, exp_q_pert)

        # ---- MISTRAL ----
        resp_m_base = generate_mistral(
            original_text, securebert_pred, securebert_conf, steering=""
        )
        parsed_m_base, _ = safe_parse(resp_m_base)
        exp_m_base = parsed_m_base.get("explanation", "") \
            if parsed_m_base else ""

        if ptype == "P4_PROMPT":
            resp_m_pert = generate_mistral(
                original_text, securebert_pred, prompt_conf, steering=""
            )
        elif ptype == "P5_STEERING":
            resp_m_pert = generate_mistral(
                original_text, securebert_pred, prompt_conf,
                steering=steering
            )
        else:
            resp_m_pert = generate_mistral(
                row["perturbed_text"], securebert_pred, prompt_conf,
                steering=""
            )
        parsed_m_pert, _ = safe_parse(resp_m_pert)
        exp_m_pert = parsed_m_pert.get("explanation", "") \
            if parsed_m_pert else ""
        sim_mistral = cosine_sim(exp_m_base, exp_m_pert)

        q_stable = parsed_q_base.get("classification") == \
            parsed_q_pert.get("classification") \
            if parsed_q_base and parsed_q_pert else None
        m_stable = parsed_m_base.get("classification") == \
            parsed_m_pert.get("classification") \
            if parsed_m_base and parsed_m_pert else None

        print(f"  {cls:<8} Qwen={sim_qwen:.3f}  "
              f"Mistral={sim_mistral:.3f}  "
              f"Diff={sim_mistral-sim_qwen:+.3f}  "
              f"Q_stable={q_stable}  M_stable={m_stable}")

        all_results.append({
            "ptype":             ptype,
            "class":             cls,
            "qwen_sim":          sim_qwen,
            "mistral_sim":       sim_mistral,
            "diff":              sim_mistral - sim_qwen
                if sim_qwen and sim_mistral else None,
            "qwen_base_len":     len(exp_q_base),
            "qwen_pert_len":     len(exp_q_pert),
            "mistral_base_len":  len(exp_m_base),
            "mistral_pert_len":  len(exp_m_pert),
            "qwen_pred_stable":  q_stable,
            "mistral_pred_stable": m_stable,
            "qwen_base_exp":     exp_q_base,
            "qwen_pert_exp":     exp_q_pert,
            "mistral_base_exp":  exp_m_base,
            "mistral_pert_exp":  exp_m_pert,
        })

val_df = pd.DataFrame(all_results)

# ================================================================
# TEXT SUMMARY
# ================================================================
print(f"\n\n{'='*70}")
print("SUMMARY TABLE")
print(f"{'='*70}")
print(f"\n{'Perturbation':<20} {'Class':<10} "
      f"{'Qwen':>8} {'Mistral':>10} {'Diff':>8}")
print("-"*60)
for ptype in PTYPES:
    for cls in CLASSES:
        r = val_df[
            (val_df["ptype"] == ptype) & (val_df["class"] == cls)
        ]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        print(f"  {ptype:<18} {cls:<10} "
              f"{r['qwen_sim']:>8.3f} "
              f"{r['mistral_sim']:>10.3f} "
              f"{r['diff']:>+8.3f}")

print(f"\n{'Perturbation':<20} {'Qwen Mean':>10} "
      f"{'Mistral Mean':>14} {'Diff':>8}")
print("-"*56)
for ptype in PTYPES:
    s = val_df[val_df["ptype"] == ptype]
    qm = s["qwen_sim"].dropna().mean()
    mm = s["mistral_sim"].dropna().mean()
    print(f"  {ptype:<18} {qm:>10.3f} {mm:>14.3f} {mm-qm:>+8.3f}")

print(f"\n  Overall Qwen mean:    "
      f"{val_df['qwen_sim'].dropna().mean():.3f}")
print(f"  Overall Mistral mean: "
      f"{val_df['mistral_sim'].dropna().mean():.3f}")
print(f"  Overall difference:   "
      f"{val_df['mistral_sim'].dropna().mean() - val_df['qwen_sim'].dropna().mean():+.3f}")

# ================================================================
# VISUALIZATIONS
# ================================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    "Qwen2.5-1.5B vs Mistral-7B — Explanation Drift Validation\n"
    "Cosine Similarity (higher = less drift)",
    fontsize=13, fontweight="bold"
)

COLORS = {"Qwen": "#2196F3", "Mistral": "#FF5722"}
x = np.arange(len(CLASSES))
width = 0.35

# --- Plot 1-5: One plot per perturbation type ---
for i, ptype in enumerate(PTYPES):
    ax = axes[i // 3][i % 3]
    subset = val_df[val_df["ptype"] == ptype]

    qwen_sims    = [subset[subset["class"] == c]["qwen_sim"].values[0]
                    if len(subset[subset["class"] == c]) > 0 else 0
                    for c in CLASSES]
    mistral_sims = [subset[subset["class"] == c]["mistral_sim"].values[0]
                    if len(subset[subset["class"] == c]) > 0 else 0
                    for c in CLASSES]

    bars1 = ax.bar(x - width/2, qwen_sims, width,
                   label="Qwen2.5-1.5B", color=COLORS["Qwen"],
                   alpha=0.85)
    bars2 = ax.bar(x + width/2, mistral_sims, width,
                   label="Mistral-7B", color=COLORS["Mistral"],
                   alpha=0.85)

    ax.axhline(y=0.85, color="red", linestyle="--",
               alpha=0.6, linewidth=1.5, label="Drift threshold (0.85)")
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, fontsize=9)
    ax.set_ylim(0.3, 1.05)
    ax.set_ylabel("Cosine Similarity")
    ax.set_title(f"{ptype}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=7)

    # Value labels on bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7,
                color=COLORS["Qwen"], fontweight="bold")
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7,
                color=COLORS["Mistral"], fontweight="bold")

# --- Plot 6: Overall comparison across perturbation types ---
ax6 = axes[1][2]
ptype_labels  = ["P1\nPayload", "P2\nInsert",
                 "P3\nReorder", "P4\nConfid", "P5\nSteer"]
qwen_means    = [val_df[val_df["ptype"] == p]["qwen_sim"].dropna().mean()
                 for p in PTYPES]
mistral_means = [val_df[val_df["ptype"] == p]["mistral_sim"].dropna().mean()
                 for p in PTYPES]

x6 = np.arange(len(PTYPES))
bars_q = ax6.bar(x6 - width/2, qwen_means, width,
                  label="Qwen2.5-1.5B", color=COLORS["Qwen"], alpha=0.85)
bars_m = ax6.bar(x6 + width/2, mistral_means, width,
                  label="Mistral-7B", color=COLORS["Mistral"], alpha=0.85)

ax6.axhline(y=0.85, color="red", linestyle="--",
            alpha=0.6, linewidth=1.5, label="Drift threshold")
ax6.set_xticks(x6)
ax6.set_xticklabels(ptype_labels, fontsize=9)
ax6.set_ylim(0.3, 1.05)
ax6.set_ylabel("Mean Cosine Similarity")
ax6.set_title("Overall: Mean Across All Classes",
              fontsize=10, fontweight="bold")
ax6.legend(fontsize=7)

for bar in bars_q:
    ax6.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.01,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=8,
             color=COLORS["Qwen"], fontweight="bold")
for bar in bars_m:
    ax6.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.01,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=8,
             color=COLORS["Mistral"], fontweight="bold")

plt.tight_layout()
plt.savefig("/kaggle/working/validation_qwen_vs_mistral.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved")

# ================================================================
# HEATMAP — Difference (Mistral - Qwen)
# ================================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5))
fig2.suptitle(
    "Qwen vs Mistral — Detailed Drift Heatmaps",
    fontsize=13, fontweight="bold"
)

# Build matrices
qwen_matrix    = np.zeros((len(PTYPES), len(CLASSES)))
mistral_matrix = np.zeros((len(PTYPES), len(CLASSES)))
diff_matrix    = np.zeros((len(PTYPES), len(CLASSES)))

for i, ptype in enumerate(PTYPES):
    for j, cls in enumerate(CLASSES):
        r = val_df[
            (val_df["ptype"] == ptype) & (val_df["class"] == cls)
        ]
        if len(r) > 0:
            qwen_matrix[i, j]    = r.iloc[0]["qwen_sim"]
            mistral_matrix[i, j] = r.iloc[0]["mistral_sim"]
            diff_matrix[i, j]    = r.iloc[0]["diff"] \
                if r.iloc[0]["diff"] else 0

ptype_short = ["P1 Payload", "P2 Insert",
               "P3 Reorder", "P4 Confidence", "P5 Steering"]

# Heatmap 1 — Side by side Qwen vs Mistral
combined = np.hstack([qwen_matrix, mistral_matrix])
im1 = axes2[0].imshow(combined, cmap="RdYlGn",
                       vmin=0.4, vmax=1.0, aspect="auto")
axes2[0].set_xticks(range(10))
axes2[0].set_xticklabels(
    [f"Q\n{c}" for c in CLASSES] + [f"M\n{c}" for c in CLASSES],
    fontsize=8
)
axes2[0].set_yticks(range(len(PTYPES)))
axes2[0].set_yticklabels(ptype_short, fontsize=9)
axes2[0].set_title("Cosine Similarity — Qwen (Q) vs Mistral (M)",
                    fontsize=10)
axes2[0].axvline(x=4.5, color="black", linewidth=2)
plt.colorbar(im1, ax=axes2[0], label="Cosine Similarity")

# Add values in cells
for i in range(len(PTYPES)):
    for j in range(10):
        val = combined[i, j]
        axes2[0].text(j, i, f'{val:.2f}',
                      ha='center', va='center',
                      fontsize=7,
                      color='white' if val < 0.65 else 'black')

# Heatmap 2 — Difference (Mistral - Qwen)
im2 = axes2[1].imshow(diff_matrix, cmap="RdBu",
                       vmin=-0.3, vmax=0.3, aspect="auto")
axes2[1].set_xticks(range(len(CLASSES)))
axes2[1].set_xticklabels(CLASSES, fontsize=9)
axes2[1].set_yticks(range(len(PTYPES)))
axes2[1].set_yticklabels(ptype_short, fontsize=9)
axes2[1].set_title("Difference: Mistral − Qwen\n"
                    "(Blue = Mistral more stable, Red = Qwen more stable)",
                    fontsize=10)
plt.colorbar(im2, ax=axes2[1], label="Similarity Difference")

for i in range(len(PTYPES)):
    for j in range(len(CLASSES)):
        val = diff_matrix[i, j]
        axes2[1].text(j, i, f'{val:+.2f}',
                      ha='center', va='center',
                      fontsize=8,
                      color='white' if abs(val) > 0.15 else 'black')

plt.tight_layout()
plt.savefig("/kaggle/working/validation_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Heatmap saved")

# Save validation results
val_df.to_parquet(
    "/kaggle/working/validation_qwen_vs_mistral.parquet",
    index=False
)
print(f"✓ Results saved")
print(f"\nTotal inference calls: {len(all_results) * 4} "
      f"({len(all_results)} windows × 2 models × 2 runs)")

In [ ]:
# Check ROAD data structure
import pandas as pd
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")

road_df = pd.read_parquet(INPUT_DIR / "task4_road_windows.parquet")
road_p1 = pd.read_parquet("/kaggle/working/task4_road_results.parquet")
road_p5 = pd.read_parquet("/kaggle/working/task4_road_p5_results.parquet")

print(f"ROAD windows: {len(road_df)}")
print(f"Columns: {list(road_df.columns)}")
print(f"\nROAD P1 results: {len(road_p1)}")
print(f"P1 Columns: {list(road_p1.columns)}")
print(f"\nROAD P5 results: {len(road_p5)}")
print(f"P5 Columns: {list(road_p5.columns)}")
print(f"\nROAD classes: {road_df['label_name'].unique()}")
print(f"\nSample row:")
print(road_df.iloc[0])

In [ ]:
import json
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path("/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark")
road_df = pd.read_parquet(INPUT_DIR / "task4_road_windows.parquet")

ROAD_CLASSES = ["AMBIENT", "ACCELERATOR", "FUZZING", "SPEEDOMETER", "CORRELATED"]
PTYPES = ["P1_PAYLOAD", "P2_INSERTION", "P3_REORDER", "P4_PROMPT", "P5_STEERING"]

ROAD_STEERING = "Note: Pay particular attention to inter-frame timing irregularities and arrival interval patterns."

# ================================================================
# ROAD PERTURBATION FUNCTIONS
# ================================================================

def frames_to_text(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

def perturb_road_p1(frames, seed=42):
    """Mutate one byte by ±1 in middle frame"""
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)
    target_idx = len(frames) // 2
    frame = perturbed[target_idx]
    byte_idx = rng.randint(0, 7)
    original_val = frame["data"][byte_idx]
    delta = rng.choice([-1, 1])
    frame["data"][byte_idx] = (original_val + delta) % 256
    return frames_to_text(perturbed)

def perturb_road_p2(frames, seed=42):
    """Insert one ambient frame into middle of window"""
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)
    insert_idx = len(frames) // 2

    # Create a realistic ambient frame
    ambient_ids = [0x162, 0x0A7, 0x464, 0x577, 0x230, 0x3B9]
    inserted_frame = {
        "timestamp": perturbed[insert_idx]["timestamp"] + 0.001,
        "can_id":    rng.choice(ambient_ids),
        "dlc":       8,
        "data":      [rng.randint(0, 255) for _ in range(8)]
    }
    perturbed.insert(insert_idx, inserted_frame)
    # Keep only 14 frames
    perturbed = perturbed[:14]
    return frames_to_text(perturbed)

def perturb_road_p3(frames, seed=42):
    """Swap two adjacent frames in middle of window"""
    perturbed = copy.deepcopy(frames)
    swap_idx = len(frames) // 2
    if swap_idx + 1 < len(perturbed):
        perturbed[swap_idx], perturbed[swap_idx + 1] = \
            perturbed[swap_idx + 1], perturbed[swap_idx]
    return frames_to_text(perturbed)

def get_road_perturbation(frames, original_text, ptype, seed=42):
    """Get perturbed text and steering for each perturbation type"""
    if ptype == "P1_PAYLOAD":
        return perturb_road_p1(frames, seed), ""
    elif ptype == "P2_INSERTION":
        return perturb_road_p2(frames, seed), ""
    elif ptype == "P3_REORDER":
        return perturb_road_p3(frames, seed), ""
    elif ptype == "P4_PROMPT":
        # P4: same text, we will reduce confidence in prompt
        # For ROAD we have no SecureBERT conf so we simulate
        # by adding uncertainty statement
        return original_text, "Note: The classifier confidence for this sample is moderate (0.623). Please analyze carefully."
    elif ptype == "P5_STEERING":
        return original_text, ROAD_STEERING
    return original_text, ""

# ================================================================
# ROAD GENERATION FUNCTIONS
# ================================================================
ROAD_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_qwen_road(text, steering=""):
    user_content = "Analyze this CAN bus telemetry sequence and classify it."
    if steering:
        user_content += f"\n\n{steering}"
    user_content += f"\n\n{text}\n\nProvide your classification and detailed explanation in JSON format."

    messages = [
        {"role": "system", "content": ROAD_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer([formatted], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def generate_mistral_road(text, steering=""):
    user_content = "Analyze this CAN bus telemetry sequence and classify it."
    if steering:
        user_content += f"\n\n{steering}"
    user_content += f"\n\n{text}\n\nProvide your classification and detailed explanation in JSON format."

    messages = [
        {"role": "user",
         "content": f"{ROAD_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ================================================================
# RUN VALIDATION — 1 window per class per perturbation type
# ================================================================
print("ROAD VALIDATION — QWEN vs MISTRAL")
print("All 5 classes × All 5 perturbation types")
print("="*70)

road_results = []

for ptype in PTYPES:
    print(f"\n--- {ptype} ---")
    for cls in ROAD_CLASSES:
        subset = road_df[road_df["label_name"] == cls]
        if len(subset) == 0:
            print(f"  {cls}: NO SAMPLE — skipping")
            continue

        row = subset.iloc[0]
        original_text = row["text"]
        frames = json.loads(row["frames_json"])

        # Get perturbed text and steering
        perturbed_text, steering = get_road_perturbation(
            frames, original_text, ptype, seed=42
        )

        # ---- QWEN ----
        resp_q_base = generate_qwen_road(original_text, steering="")
        parsed_q_base, _ = safe_parse(resp_q_base)
        exp_q_base = parsed_q_base.get("explanation", "") \
            if parsed_q_base else ""

        if ptype in ["P4_PROMPT", "P5_STEERING"]:
            resp_q_pert = generate_qwen_road(original_text, steering=steering)
        else:
            resp_q_pert = generate_qwen_road(perturbed_text, steering="")
        parsed_q_pert, _ = safe_parse(resp_q_pert)
        exp_q_pert = parsed_q_pert.get("explanation", "") \
            if parsed_q_pert else ""
        sim_qwen = cosine_sim(exp_q_base, exp_q_pert)

        # ---- MISTRAL ----
        resp_m_base = generate_mistral_road(original_text, steering="")
        parsed_m_base, _ = safe_parse(resp_m_base)
        exp_m_base = parsed_m_base.get("explanation", "") \
            if parsed_m_base else ""

        if ptype in ["P4_PROMPT", "P5_STEERING"]:
            resp_m_pert = generate_mistral_road(
                original_text, steering=steering
            )
        else:
            resp_m_pert = generate_mistral_road(perturbed_text, steering="")
        parsed_m_pert, _ = safe_parse(resp_m_pert)
        exp_m_pert = parsed_m_pert.get("explanation", "") \
            if parsed_m_pert else ""
        sim_mistral = cosine_sim(exp_m_base, exp_m_pert)

        diff = sim_mistral - sim_qwen \
            if sim_qwen and sim_mistral else None

        q_stable = parsed_q_base.get("classification") == \
            parsed_q_pert.get("classification") \
            if parsed_q_base and parsed_q_pert else None
        m_stable = parsed_m_base.get("classification") == \
            parsed_m_pert.get("classification") \
            if parsed_m_base and parsed_m_pert else None

        print(f"  {cls:<12} Qwen={sim_qwen:.3f}  "
              f"Mistral={sim_mistral:.3f}  "
              f"Diff={diff:+.3f}  "
              f"Q_stable={q_stable}  M_stable={m_stable}")

        road_results.append({
            "ptype":               ptype,
            "class":               cls,
            "qwen_sim":            sim_qwen,
            "mistral_sim":         sim_mistral,
            "diff":                diff,
            "qwen_base_len":       len(exp_q_base),
            "qwen_pert_len":       len(exp_q_pert),
            "mistral_base_len":    len(exp_m_base),
            "mistral_pert_len":    len(exp_m_pert),
            "qwen_pred_stable":    q_stable,
            "mistral_pred_stable": m_stable,
            "qwen_base_exp":       exp_q_base,
            "qwen_pert_exp":       exp_q_pert,
            "mistral_base_exp":    exp_m_base,
            "mistral_pert_exp":    exp_m_pert,
        })

road_val_df = pd.DataFrame(road_results)

# ================================================================
# SUMMARY TABLE
# ================================================================
print(f"\n\n{'='*70}")
print("ROAD SUMMARY TABLE")
print(f"{'='*70}")
print(f"\n{'Perturbation':<20} {'Class':<14} "
      f"{'Qwen':>8} {'Mistral':>10} {'Diff':>8}")
print("-"*65)
for ptype in PTYPES:
    for cls in ROAD_CLASSES:
        r = road_val_df[
            (road_val_df["ptype"] == ptype) &
            (road_val_df["class"] == cls)
        ]
        if len(r) == 0:
            continue
        r = r.iloc[0]
        print(f"  {ptype:<18} {cls:<14} "
              f"{r['qwen_sim']:>8.3f} "
              f"{r['mistral_sim']:>10.3f} "
              f"{r['diff']:>+8.3f}")

print(f"\n{'Perturbation':<20} {'Qwen Mean':>10} "
      f"{'Mistral Mean':>14} {'Diff':>8}")
print("-"*56)
for ptype in PTYPES:
    s = road_val_df[road_val_df["ptype"] == ptype]
    qm = s["qwen_sim"].dropna().mean()
    mm = s["mistral_sim"].dropna().mean()
    print(f"  {ptype:<18} {qm:>10.3f} {mm:>14.3f} {mm-qm:>+8.3f}")

print(f"\n  Overall Qwen mean:    "
      f"{road_val_df['qwen_sim'].dropna().mean():.3f}")
print(f"  Overall Mistral mean: "
      f"{road_val_df['mistral_sim'].dropna().mean():.3f}")
print(f"  Overall difference:   "
      f"{road_val_df['mistral_sim'].dropna().mean() - road_val_df['qwen_sim'].dropna().mean():+.3f}")

# ================================================================
# VISUALIZATIONS
# ================================================================
COLORS = {"Qwen": "#2196F3", "Mistral": "#FF5722"}
x = np.arange(len(ROAD_CLASSES))
width = 0.35

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    "Qwen2.5-1.5B vs Mistral-7B — ROAD Dataset Validation\n"
    "Cosine Similarity (higher = less drift)",
    fontsize=13, fontweight="bold"
)

for i, ptype in enumerate(PTYPES):
    ax = axes[i // 3][i % 3]
    subset = road_val_df[road_val_df["ptype"] == ptype]

    qwen_sims = [
        subset[subset["class"] == c]["qwen_sim"].values[0]
        if len(subset[subset["class"] == c]) > 0 else 0
        for c in ROAD_CLASSES
    ]
    mistral_sims = [
        subset[subset["class"] == c]["mistral_sim"].values[0]
        if len(subset[subset["class"] == c]) > 0 else 0
        for c in ROAD_CLASSES
    ]

    bars1 = ax.bar(x - width/2, qwen_sims, width,
                   label="Qwen2.5-1.5B",
                   color=COLORS["Qwen"], alpha=0.85)
    bars2 = ax.bar(x + width/2, mistral_sims, width,
                   label="Mistral-7B",
                   color=COLORS["Mistral"], alpha=0.85)

    ax.axhline(y=0.85, color="red", linestyle="--",
               alpha=0.6, linewidth=1.5, label="Drift threshold")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [c[:5] for c in ROAD_CLASSES], fontsize=8
    )
    ax.set_ylim(0.3, 1.05)
    ax.set_ylabel("Cosine Similarity")
    ax.set_title(f"{ptype}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=7)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7,
                color=COLORS["Qwen"], fontweight="bold")
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.01,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7,
                color=COLORS["Mistral"], fontweight="bold")

# Overall summary plot
ax6 = axes[1][2]
ptype_labels = ["P1\nPayload", "P2\nInsert",
                "P3\nReorder", "P4\nConfid", "P5\nSteer"]
qwen_means = [
    road_val_df[road_val_df["ptype"] == p]["qwen_sim"].dropna().mean()
    for p in PTYPES
]
mistral_means = [
    road_val_df[road_val_df["ptype"] == p]["mistral_sim"].dropna().mean()
    for p in PTYPES
]

x6 = np.arange(len(PTYPES))
bars_q = ax6.bar(x6 - width/2, qwen_means, width,
                  label="Qwen2.5-1.5B",
                  color=COLORS["Qwen"], alpha=0.85)
bars_m = ax6.bar(x6 + width/2, mistral_means, width,
                  label="Mistral-7B",
                  color=COLORS["Mistral"], alpha=0.85)

ax6.axhline(y=0.85, color="red", linestyle="--",
            alpha=0.6, linewidth=1.5, label="Drift threshold")
ax6.set_xticks(x6)
ax6.set_xticklabels(ptype_labels, fontsize=9)
ax6.set_ylim(0.3, 1.05)
ax6.set_ylabel("Mean Cosine Similarity")
ax6.set_title("Overall: Mean Across All Classes",
              fontsize=10, fontweight="bold")
ax6.legend(fontsize=7)

for bar in bars_q:
    ax6.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.01,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=8,
             color=COLORS["Qwen"], fontweight="bold")
for bar in bars_m:
    ax6.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.01,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=8,
             color=COLORS["Mistral"], fontweight="bold")

plt.tight_layout()
plt.savefig("/kaggle/working/road_validation_qwen_vs_mistral.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Bar chart saved")

# ================================================================
# HEATMAPS
# ================================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5))
fig2.suptitle(
    "ROAD Dataset — Qwen vs Mistral Drift Heatmaps",
    fontsize=13, fontweight="bold"
)

qwen_matrix    = np.zeros((len(PTYPES), len(ROAD_CLASSES)))
mistral_matrix = np.zeros((len(PTYPES), len(ROAD_CLASSES)))
diff_matrix    = np.zeros((len(PTYPES), len(ROAD_CLASSES)))

for i, ptype in enumerate(PTYPES):
    for j, cls in enumerate(ROAD_CLASSES):
        r = road_val_df[
            (road_val_df["ptype"] == ptype) &
            (road_val_df["class"] == cls)
        ]
        if len(r) > 0:
            qwen_matrix[i, j]    = r.iloc[0]["qwen_sim"]
            mistral_matrix[i, j] = r.iloc[0]["mistral_sim"]
            diff_matrix[i, j]    = r.iloc[0]["diff"] \
                if r.iloc[0]["diff"] else 0

ptype_short = ["P1 Payload", "P2 Insert",
               "P3 Reorder", "P4 Confidence", "P5 Steering"]
class_short = [c[:5] for c in ROAD_CLASSES]

combined = np.hstack([qwen_matrix, mistral_matrix])
im1 = axes2[0].imshow(combined, cmap="RdYlGn",
                       vmin=0.4, vmax=1.0, aspect="auto")
axes2[0].set_xticks(range(10))
axes2[0].set_xticklabels(
    [f"Q\n{c}" for c in class_short] +
    [f"M\n{c}" for c in class_short],
    fontsize=8
)
axes2[0].set_yticks(range(len(PTYPES)))
axes2[0].set_yticklabels(ptype_short, fontsize=9)
axes2[0].set_title("Cosine Similarity — Qwen (Q) vs Mistral (M)",
                    fontsize=10)
axes2[0].axvline(x=4.5, color="black", linewidth=2)
plt.colorbar(im1, ax=axes2[0], label="Cosine Similarity")

for i in range(len(PTYPES)):
    for j in range(10):
        val = combined[i, j]
        axes2[0].text(j, i, f'{val:.2f}',
                      ha='center', va='center', fontsize=7,
                      color='white' if val < 0.65 else 'black')

im2 = axes2[1].imshow(diff_matrix, cmap="RdBu",
                       vmin=-0.3, vmax=0.3, aspect="auto")
axes2[1].set_xticks(range(len(ROAD_CLASSES)))
axes2[1].set_xticklabels(class_short, fontsize=9)
axes2[1].set_yticks(range(len(PTYPES)))
axes2[1].set_yticklabels(ptype_short, fontsize=9)
axes2[1].set_title(
    "Difference: Mistral − Qwen\n"
    "(Blue = Mistral more stable, Red = Qwen more stable)",
    fontsize=10
)
plt.colorbar(im2, ax=axes2[1], label="Similarity Difference")

for i in range(len(PTYPES)):
    for j in range(len(ROAD_CLASSES)):
        val = diff_matrix[i, j]
        axes2[1].text(j, i, f'{val:+.2f}',
                      ha='center', va='center', fontsize=8,
                      color='white' if abs(val) > 0.15 else 'black')

plt.tight_layout()
plt.savefig("/kaggle/working/road_validation_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Heatmap saved")

road_val_df.to_parquet(
    "/kaggle/working/road_validation_results.parquet",
    index=False
)
print(f"✓ Results saved")
print(f"\nTotal inference calls: {len(road_results) * 4} "
      f"({len(road_results)} windows × 2 models × 2 runs)")

In [ ]:
# ROAD P5 VALIDATION — 1 window per class
# Qwen vs Mistral side by side
print("ROAD P5 VALIDATION — QWEN vs MISTRAL")
print("1 window per class")
print("="*60)

ROAD_CLASSES  = ["AMBIENT", "ACCELERATOR", "FUZZING",
                 "SPEEDOMETER", "CORRELATED"]
ROAD_STEERING = "Note: Pay particular attention to inter-frame \
timing irregularities and arrival interval patterns."

road_val_results = []

for cls in ROAD_CLASSES:
    subset = road_df[road_df["label_name"] == cls]
    row    = subset.iloc[0]
    text   = row["text"]

    # ---- QWEN ----
    resp_q_base = generate_qwen_road(text, steering="")
    parsed_q_base, _ = safe_parse(resp_q_base)
    exp_q_base = parsed_q_base.get("explanation", "") \
        if parsed_q_base else ""

    resp_q_pert = generate_qwen_road(text, steering=ROAD_STEERING)
    parsed_q_pert, _ = safe_parse(resp_q_pert)
    exp_q_pert = parsed_q_pert.get("explanation", "") \
        if parsed_q_pert else ""

    sim_qwen = cosine_sim(exp_q_base, exp_q_pert)

    # ---- MISTRAL ----
    resp_m_base = generate_mistral_road(text, steering="")
    parsed_m_base, _ = safe_parse(resp_m_base)
    exp_m_base = parsed_m_base.get("explanation", "") \
        if parsed_m_base else ""

    resp_m_pert = generate_mistral_road(text, steering=ROAD_STEERING)
    parsed_m_pert, _ = safe_parse(resp_m_pert)
    exp_m_pert = parsed_m_pert.get("explanation", "") \
        if parsed_m_pert else ""

    sim_mistral = cosine_sim(exp_m_base, exp_m_pert)
    diff = (sim_mistral - sim_qwen) \
        if (sim_qwen and sim_mistral) else None

    q_str = f"{sim_qwen:.3f}" if sim_qwen else "None"
    m_str = f"{sim_mistral:.3f}" if sim_mistral else "None"
    d_str = f"{diff:+.3f}" if diff else "None"

    print(f"\n{cls}:")
    print(f"  Qwen sim:    {q_str}")
    print(f"  Mistral sim: {m_str}")
    print(f"  Diff:        {d_str}")
    print(f"  Q_stable:    {parsed_q_base.get('classification') == parsed_q_pert.get('classification') if parsed_q_base and parsed_q_pert else 'N/A'}")
    print(f"  M_stable:    {parsed_m_base.get('classification') == parsed_m_pert.get('classification') if parsed_m_base and parsed_m_pert else 'N/A'}")
    print(f"\n  Qwen baseline:    {exp_q_base[:120]}")
    print(f"  Qwen steered:     {exp_q_pert[:120]}")
    print(f"  Mistral baseline: {exp_m_base[:120]}")
    print(f"  Mistral steered:  {exp_m_pert[:120]}")

    road_val_results.append({
        "class":       cls,
        "qwen_sim":    sim_qwen,
        "mistral_sim": sim_mistral,
        "diff":        diff,
    })

road_val_df = pd.DataFrame(road_val_results)

print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"\n{'Class':<14} {'Qwen P5':>10} {'Mistral P5':>12} {'Diff':>8}")
print("-"*48)
for _, r in road_val_df.iterrows():
    q_str = f"{r['qwen_sim']:.3f}" if r['qwen_sim'] else "None"
    m_str = f"{r['mistral_sim']:.3f}" if r['mistral_sim'] else "None"
    d_str = f"{r['diff']:+.3f}" if r['diff'] else "None"
    print(f"  {r['class']:<12} {q_str:>10} {m_str:>12} {d_str:>8}")

qwen_mean    = road_val_df["qwen_sim"].dropna().mean()
mistral_mean = road_val_df["mistral_sim"].dropna().mean()
print(f"\n  Qwen mean:    {qwen_mean:.3f}  (Task 4 full run: 0.864)")
print(f"  Mistral mean: {mistral_mean:.3f}")
print(f"  Difference:   {mistral_mean-qwen_mean:+.3f}")

if qwen_mean < 0.95 and mistral_mean < 0.95:
    print(f"\n  ✓ Both models show drift on ROAD P5")
    print(f"  ✓ Prompt-layer vulnerability confirmed on both models")
elif qwen_mean < 0.95:
    print(f"\n  ~ Only Qwen shows drift — Mistral more robust on ROAD")
else:
    print(f"\n  ⚠ Neither model shows meaningful drift on ROAD P5")

In [ ]:
# Show full explanations for one class
cls = "ACCELERATOR"
subset = road_df[road_df["label_name"] == cls]
row = subset.iloc[0]
text = row["text"]

resp_q_base = generate_qwen_road(text, steering="")
parsed_q_base, _ = safe_parse(resp_q_base)
exp_q_base = parsed_q_base.get("explanation", "") if parsed_q_base else ""

resp_q_pert = generate_qwen_road(text, steering=ROAD_STEERING)
parsed_q_pert, _ = safe_parse(resp_q_pert)
exp_q_pert = parsed_q_pert.get("explanation", "") if parsed_q_pert else ""

resp_m_base = generate_mistral_road(text, steering="")
parsed_m_base, _ = safe_parse(resp_m_base)
exp_m_base = parsed_m_base.get("explanation", "") if parsed_m_base else ""

resp_m_pert = generate_mistral_road(text, steering=ROAD_STEERING)
parsed_m_pert, _ = safe_parse(resp_m_pert)
exp_m_pert = parsed_m_pert.get("explanation", "") if parsed_m_pert else ""

print(f"ACCELERATOR — FULL EXPLANATIONS")
print(f"{'='*60}")
print(f"\nQWEN BASELINE ({len(exp_q_base)} chars):")
print(exp_q_base)
print(f"\nQWEN STEERED ({len(exp_q_pert)} chars):")
print(exp_q_pert)
print(f"\nMISTRAL BASELINE ({len(exp_m_base)} chars):")
print(exp_m_base)
print(f"\nMISTRAL STEERED ({len(exp_m_pert)} chars):")
print(exp_m_pert)

sim_q = cosine_sim(exp_q_base, exp_q_pert)
sim_m = cosine_sim(exp_m_base, exp_m_pert)
print(f"\nQwen similarity:   {sim_q:.3f}")
print(f"Mistral similarity: {sim_m:.3f}")

In [ ]:
# ROAD P5 FULL EXPLANATIONS — All classes, Qwen vs Mistral
# Complete explanations + comparison

print("ROAD P5 — FULL EXPLANATIONS ALL CLASSES")
print("Qwen2.5-1.5B vs Mistral-7B")
print("="*70)

ROAD_CLASSES  = ["AMBIENT", "ACCELERATOR", "FUZZING",
                 "SPEEDOMETER", "CORRELATED"]
ROAD_STEERING = "Note: Pay particular attention to inter-frame \
timing irregularities and arrival interval patterns."

full_results = []

for cls in ROAD_CLASSES:
    print(f"\n{'='*70}")
    print(f"CLASS: {cls}")
    print(f"{'='*70}")

    subset = road_df[road_df["label_name"] == cls]
    row    = subset.iloc[0]
    text   = row["text"]

    # ---- QWEN ----
    resp_q_base = generate_qwen_road(text, steering="")
    parsed_q_base, _ = safe_parse(resp_q_base)
    exp_q_base = parsed_q_base.get("explanation", "") \
        if parsed_q_base else ""

    resp_q_pert = generate_qwen_road(text, steering=ROAD_STEERING)
    parsed_q_pert, _ = safe_parse(resp_q_pert)
    exp_q_pert = parsed_q_pert.get("explanation", "") \
        if parsed_q_pert else ""

    sim_q = cosine_sim(exp_q_base, exp_q_pert)

    # ---- MISTRAL ----
    resp_m_base = generate_mistral_road(text, steering="")
    parsed_m_base, _ = safe_parse(resp_m_base)
    exp_m_base = parsed_m_base.get("explanation", "") \
        if parsed_m_base else ""

    resp_m_pert = generate_mistral_road(text, steering=ROAD_STEERING)
    parsed_m_pert, _ = safe_parse(resp_m_pert)
    exp_m_pert = parsed_m_pert.get("explanation", "") \
        if parsed_m_pert else ""

    sim_m = cosine_sim(exp_m_base, exp_m_pert)
    diff  = (sim_m - sim_q) if (sim_q and sim_m) else None

    # Word overlap
    q_base_words = set(exp_q_base.lower().split())
    q_pert_words = set(exp_q_pert.lower().split())
    m_base_words = set(exp_m_base.lower().split())
    m_pert_words = set(exp_m_pert.lower().split())

    q_overlap = len(q_base_words & q_pert_words) / \
        len(q_base_words | q_pert_words) \
        if q_base_words | q_pert_words else 0
    m_overlap = len(m_base_words & m_pert_words) / \
        len(m_base_words | m_pert_words) \
        if m_base_words | m_pert_words else 0

    # Print metrics
    print(f"\n  METRICS:")
    print(f"  {'':25s} {'QWEN':>12} {'MISTRAL':>12}")
    print(f"  {'Cosine similarity':25s} "
          f"{sim_q:>12.3f} {sim_m:>12.3f}")
    print(f"  {'Word overlap':25s} "
          f"{q_overlap:>12.3f} {m_overlap:>12.3f}")
    print(f"  {'Baseline length':25s} "
          f"{len(exp_q_base):>12d} {len(exp_m_base):>12d}")
    print(f"  {'Steered length':25s} "
          f"{len(exp_q_pert):>12d} {len(exp_m_pert):>12d}")
    print(f"  {'Length change':25s} "
          f"{len(exp_q_pert)-len(exp_q_base):>+12d} "
          f"{len(exp_m_pert)-len(exp_m_base):>+12d}")
    print(f"  {'Pred stable':25s} "
          f"{str(parsed_q_base.get('classification')==parsed_q_pert.get('classification') if parsed_q_base and parsed_q_pert else 'N/A'):>12} "
          f"{str(parsed_m_base.get('classification')==parsed_m_pert.get('classification') if parsed_m_base and parsed_m_pert else 'N/A'):>12}")

    # Print full explanations
    print(f"\n  ── QWEN BASELINE ──")
    print(f"  {exp_q_base}")

    print(f"\n  ── QWEN STEERED ──")
    print(f"  {exp_q_pert}")

    print(f"\n  ── MISTRAL BASELINE ──")
    print(f"  {exp_m_base}")

    print(f"\n  ── MISTRAL STEERED ──")
    print(f"  {exp_m_pert}")

    full_results.append({
        "class":          cls,
        "qwen_sim":       sim_q,
        "mistral_sim":    sim_m,
        "diff":           diff,
        "qwen_overlap":   q_overlap,
        "mistral_overlap": m_overlap,
        "qwen_base_len":  len(exp_q_base),
        "qwen_pert_len":  len(exp_q_pert),
        "mistral_base_len": len(exp_m_base),
        "mistral_pert_len": len(exp_m_pert),
        "qwen_base_exp":  exp_q_base,
        "qwen_pert_exp":  exp_q_pert,
        "mistral_base_exp": exp_m_base,
        "mistral_pert_exp": exp_m_pert,
    })

full_df = pd.DataFrame(full_results)

# ================================================================
# SUMMARY TABLE
# ================================================================
print(f"\n\n{'='*70}")
print("ROAD P5 SUMMARY — QWEN vs MISTRAL")
print(f"{'='*70}")
print(f"\n{'Class':<14} {'Qwen Sim':>10} {'Mistral Sim':>12} "
      f"{'Diff':>8} {'Q_overlap':>11} {'M_overlap':>11}")
print("-"*70)
for _, r in full_df.iterrows():
    d_str = f"{r['diff']:+.3f}" if r['diff'] is not None else "N/A"
    print(f"  {r['class']:<12} {r['qwen_sim']:>10.3f} "
          f"{r['mistral_sim']:>12.3f} {d_str:>8} "
          f"{r['qwen_overlap']:>11.3f} {r['mistral_overlap']:>11.3f}")

print(f"\n  {'Overall mean':<12} "
      f"{full_df['qwen_sim'].mean():>10.3f} "
      f"{full_df['mistral_sim'].mean():>12.3f} "
      f"{full_df['mistral_sim'].mean()-full_df['qwen_sim'].mean():>+8.3f}")

print(f"\n  Task 4 Qwen full run mean: 0.864")
print(f"  Validation Qwen mean:      "
      f"{full_df['qwen_sim'].mean():.3f}")
print(f"  Validation Mistral mean:   "
      f"{full_df['mistral_sim'].mean():.3f}")

# ================================================================
# VISUALIZATION
# ================================================================
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "ROAD Dataset P5 Semantic Steering\n"
    "Qwen2.5-1.5B vs Mistral-7B",
    fontsize=12, fontweight="bold"
)

COLORS = {"Qwen": "#2196F3", "Mistral": "#FF5722"}
x     = np.arange(len(ROAD_CLASSES))
width = 0.35

# Plot 1: Cosine similarity
ax1 = axes[0]
bars_q = ax1.bar(x - width/2, full_df["qwen_sim"], width,
                  label="Qwen2.5-1.5B",
                  color=COLORS["Qwen"], alpha=0.85)
bars_m = ax1.bar(x + width/2, full_df["mistral_sim"], width,
                  label="Mistral-7B",
                  color=COLORS["Mistral"], alpha=0.85)
ax1.axhline(y=0.85, color="red", linestyle="--",
            alpha=0.6, label="Drift threshold")
ax1.set_xticks(x)
ax1.set_xticklabels([c[:5] for c in ROAD_CLASSES], fontsize=9)
ax1.set_ylim(0.5, 1.05)
ax1.set_ylabel("Cosine Similarity")
ax1.set_title("P5 Cosine Similarity\n(higher = less drift)")
ax1.legend(fontsize=8)
for bar in bars_q:
    ax1.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.005,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=7,
             color=COLORS["Qwen"], fontweight="bold")
for bar in bars_m:
    ax1.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.005,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=7,
             color=COLORS["Mistral"], fontweight="bold")

# Plot 2: Word overlap
ax2 = axes[1]
bars_q2 = ax2.bar(x - width/2, full_df["qwen_overlap"], width,
                   label="Qwen2.5-1.5B",
                   color=COLORS["Qwen"], alpha=0.85)
bars_m2 = ax2.bar(x + width/2, full_df["mistral_overlap"], width,
                   label="Mistral-7B",
                   color=COLORS["Mistral"], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([c[:5] for c in ROAD_CLASSES], fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("Jaccard Word Overlap")
ax2.set_title("Word Overlap\n(higher = more similar vocabulary)")
ax2.legend(fontsize=8)
for bar in bars_q2:
    ax2.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.005,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=7,
             color=COLORS["Qwen"], fontweight="bold")
for bar in bars_m2:
    ax2.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.005,
             f'{bar.get_height():.3f}',
             ha='center', va='bottom', fontsize=7,
             color=COLORS["Mistral"], fontweight="bold")

# Plot 3: Difference (Mistral - Qwen)
ax3 = axes[2]
diffs = full_df["diff"].fillna(0).tolist()
colors_diff = ["#4CAF50" if d >= 0 else "#F44336" for d in diffs]
bars_d = ax3.bar(x, diffs, 0.5,
                  color=colors_diff, alpha=0.85)
ax3.axhline(y=0, color="black", linewidth=1)
ax3.set_xticks(x)
ax3.set_xticklabels([c[:5] for c in ROAD_CLASSES], fontsize=9)
ax3.set_ylabel("Similarity Difference (Mistral − Qwen)")
ax3.set_title("Difference\n(green = Mistral more stable,\nred = Qwen more stable)")
for bar, d in zip(bars_d, diffs):
    ax3.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 0.002 if d >= 0 else bar.get_height() - 0.012,
             f'{d:+.3f}',
             ha='center', va='bottom', fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig("/kaggle/working/road_p5_full_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved")

# Save results
full_df.to_parquet(
    "/kaggle/working/road_p5_validation_full.parquet",
    index=False
)
print("✓ Results saved")

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Fix word overlap calculation
def jaccard_overlap(text1, text2):
    if not text1 or not text2:
        return 0
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    intersection = words1 & words2
    union = words1 | words2
    return len(intersection) / len(union) if union else 0

In [ ]:
# VERIFICATION 1 — P6 on 5 windows per class
print("P6 VERIFICATION — 5 windows per class")
print("="*60)

N_WINDOWS = 5
verification_results = []

# HCRL
for cls in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    subset = hcrl_p6[hcrl_p6["true_label_name"] == cls].head(N_WINDOWS)
    correct_q = 0
    correct_m = 0

    for _, row in subset.iterrows():
        text = row["original_text"]
        pred = row["securebert_pred_name"]
        conf = float(row["securebert_confidence"])
        fc   = row["false_context"]

        # Qwen perturbed only — we want to check classification
        resp_q = generate_qwen_hcrl(text, pred, conf, false_context=fc)
        parsed_q, _ = safe_parse(resp_q)
        cls_q = parsed_q.get("classification") if parsed_q else None
        if cls_q == cls:
            correct_q += 1

        # Mistral perturbed only
        resp_m = generate_mistral_hcrl(text, pred, conf, false_context=fc)
        parsed_m, _ = safe_parse(resp_m)
        cls_m = parsed_m.get("classification") if parsed_m else None
        if cls_m == cls:
            correct_m += 1

    print(f"HCRL {cls:<8} Qwen correct: {correct_q}/{N_WINDOWS}  "
          f"Mistral correct: {correct_m}/{N_WINDOWS}")
    verification_results.append({
        "dataset": "HCRL", "class": cls,
        "qwen_correct": correct_q,
        "mistral_correct": correct_m,
        "n": N_WINDOWS
    })

print()

# ROAD
for cls in ["AMBIENT", "ACCELERATOR", "FUZZING",
            "SPEEDOMETER", "CORRELATED"]:
    subset = road_p6[road_p6["label_name"] == cls].head(N_WINDOWS)
    correct_q = 0
    correct_m = 0

    for _, row in subset.iterrows():
        text = row["original_text"]
        fc   = row["false_context"]

        # Qwen
        resp_q = generate_qwen_road(text, false_context=fc)
        parsed_q, _ = safe_parse(resp_q)
        cls_q = parsed_q.get("classification") if parsed_q else None
        # For ROAD we check if attack is detected
        # AMBIENT should stay NORMAL
        # Others should NOT be NORMAL
        if cls == "AMBIENT":
            if cls_q == "NORMAL":
                correct_q += 1
        else:
            if cls_q != "NORMAL":
                correct_q += 1

        # Mistral
        resp_m = generate_mistral_road(text, false_context=fc)
        parsed_m, _ = safe_parse(resp_m)
        cls_m = parsed_m.get("classification") if parsed_m else None
        if cls == "AMBIENT":
            if cls_m == "NORMAL":
                correct_m += 1
        else:
            if cls_m != "NORMAL":
                correct_m += 1

    print(f"ROAD {cls:<12} Qwen attack detected: {correct_q}/{N_WINDOWS}  "
          f"Mistral attack detected: {correct_m}/{N_WINDOWS}")
    verification_results.append({
        "dataset": "ROAD", "class": cls,
        "qwen_correct": correct_q,
        "mistral_correct": correct_m,
        "n": N_WINDOWS
    })

# Summary
ver_df = pd.DataFrame(verification_results)
print(f"\nSUMMARY")
print(f"="*60)

for dataset in ["HCRL", "ROAD"]:
    subset = ver_df[ver_df["dataset"] == dataset]
    q_total = subset["qwen_correct"].sum()
    m_total = subset["mistral_correct"].sum()
    total   = subset["n"].sum()
    print(f"\n{dataset}:")
    print(f"  Qwen correct classifications:   {q_total}/{total} "
          f"({q_total/total*100:.0f}%)")
    print(f"  Mistral correct classifications:{m_total}/{total} "
          f"({m_total/total*100:.0f}%)")

ver_df.to_parquet(
    "/kaggle/working/p6_verification.parquet", index=False
)
print(f"\n✓ Saved")

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark"
)

hcrl_p6 = pd.read_parquet(
    INPUT_DIR / "section10_p6_false_context.parquet"
)

print(f"Rows: {len(hcrl_p6)}")
print(f"Columns: {list(hcrl_p6.columns)}")
print(f"\nClass distribution:")
print(hcrl_p6["true_label_name"].value_counts().to_string())

# Verify text is correct for DOS
dos_row = hcrl_p6[hcrl_p6["true_label_name"] == "DOS"].iloc[0]
print(f"\nDOS sample text (first 150):")
print(dos_row["original_text"][:150])
print(f"\nExpected: ID=0000, DATA=00 00 00 00")
print(f"Correct: {'ID=0000' in dos_row['original_text']}")

# Verify GEAR
gear_row = hcrl_p6[hcrl_p6["true_label_name"] == "GEAR"].iloc[0]
print(f"\nGEAR sample text (first 150):")
print(gear_row["original_text"][:150])
print(f"Correct: {'ID=043F' in gear_row['original_text'] or 'ID=0316' in gear_row['original_text']}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import json
from pathlib import Path
from tqdm import tqdm

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark"
)

hcrl_p6 = pd.read_parquet(
    INPUT_DIR / "section10_p6_false_context.parquet"
)
road_p6 = pd.read_parquet(
    INPUT_DIR / "task4_road_p6_dataset.parquet"
)

print(f"✓ HCRL P6: {len(hcrl_p6)} records")
print(f"✓ ROAD P6: {len(road_p6)} records")

# ================================================================
# SYSTEM PROMPTS
# ================================================================
HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

ROAD_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

# ================================================================
# GENERATION FUNCTIONS
# ================================================================
def generate_qwen_hcrl(text, pred, conf, false_context=""):
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {pred} (confidence: {conf:.3f})
"""
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += f"""
{text}

Provide your classification and detailed explanation in the required JSON format."""

    messages = [
        {"role": "system", "content": HCRL_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_mistral_hcrl(text, pred, conf, false_context=""):
    user_content = f"""Analyze this CAN bus telemetry sequence and classify it.

SecureBERT classifier prediction: {pred} (confidence: {conf:.3f})
"""
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += f"""
{text}

Provide your classification and detailed explanation in the required JSON format."""

    messages = [
        {"role": "user",
         "content": f"{HCRL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_qwen_road(text, false_context=""):
    user_content = "Analyze this CAN bus telemetry sequence and classify it."
    if false_context:
        user_content += f"\n\n{false_context}"
    user_content += f"\n\n{text}\n\nProvide your classification and detailed explanation in JSON format."
    messages = [
        {"role": "system", "content": ROAD_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_mistral_road(text, false_context=""):
    user_content = "Analyze this CAN bus telemetry sequence and classify it."
    if false_context:
        user_content += f"\n\n{false_context}"
    user_content += f"\n\n{text}\n\nProvide your classification and detailed explanation in JSON format."
    messages = [
        {"role": "user",
         "content": f"{ROAD_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def jaccard_overlap(text1, text2):
    if not text1 or not text2:
        return 0
    w1 = set(text1.lower().split())
    w2 = set(text2.lower().split())
    return len(w1 & w2) / len(w1 | w2) if w1 | w2 else 0

# ================================================================
# PART 1 — VALIDATION (1 window per class, full explanations)
# ================================================================
print("\n" + "="*70)
print("PART 1 — VALIDATION (1 window per class)")
print("="*70)

HCRL_CLASSES = ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]
ROAD_CLASSES  = ["AMBIENT", "ACCELERATOR", "FUZZING",
                 "SPEEDOMETER", "CORRELATED"]

all_val_results = []

for dataset, classes, df in [
    ("HCRL", HCRL_CLASSES, hcrl_p6),
    ("ROAD", ROAD_CLASSES, road_p6)
]:
    print(f"\n--- {dataset} ---")
    label_col = "true_label_name" if dataset == "HCRL" else "label_name"

    for cls in classes:
        subset = df[df[label_col] == cls]
        row    = subset.iloc[0]
        text   = row["original_text"]
        fc     = row["false_context"]
        pred   = row.get("securebert_pred_name", "UNKNOWN")
        conf   = float(row.get("securebert_confidence", 1.0))

        # Baseline
        if dataset == "HCRL":
            resp_q_base = generate_qwen_hcrl(
                text, pred, conf, false_context=""
            )
            resp_m_base = generate_mistral_hcrl(
                text, pred, conf, false_context=""
            )
            resp_q_pert = generate_qwen_hcrl(
                text, pred, conf, false_context=fc
            )
            resp_m_pert = generate_mistral_hcrl(
                text, pred, conf, false_context=fc
            )
        else:
            resp_q_base = generate_qwen_road(text, false_context="")
            resp_m_base = generate_mistral_road(text, false_context="")
            resp_q_pert = generate_qwen_road(text, false_context=fc)
            resp_m_pert = generate_mistral_road(text, false_context=fc)

        parsed_q_base, _ = safe_parse(resp_q_base)
        parsed_q_pert, _ = safe_parse(resp_q_pert)
        parsed_m_base, _ = safe_parse(resp_m_base)
        parsed_m_pert, _ = safe_parse(resp_m_pert)

        exp_q_base = parsed_q_base.get("explanation","") if parsed_q_base else ""
        exp_q_pert = parsed_q_pert.get("explanation","") if parsed_q_pert else ""
        exp_m_base = parsed_m_base.get("explanation","") if parsed_m_base else ""
        exp_m_pert = parsed_m_pert.get("explanation","") if parsed_m_pert else ""

        sim_q = cosine_sim(exp_q_base, exp_q_pert)
        sim_m = cosine_sim(exp_m_base, exp_m_pert)
        diff  = (sim_m - sim_q) if (sim_q and sim_m) else None

        q_stable = parsed_q_base.get("classification") == \
            parsed_q_pert.get("classification") \
            if parsed_q_base and parsed_q_pert else None
        m_stable = parsed_m_base.get("classification") == \
            parsed_m_pert.get("classification") \
            if parsed_m_base and parsed_m_pert else None

        q_str = f"{sim_q:.3f}" if sim_q else "None"
        m_str = f"{sim_m:.3f}" if sim_m else "None"
        d_str = f"{diff:+.3f}" if diff else "None"

        print(f"  {cls:<14} Qwen={q_str}  Mistral={m_str}  "
              f"Diff={d_str}  "
              f"Q_stable={q_stable}  M_stable={m_stable}")
        print(f"    Q: {parsed_q_base.get('classification') if parsed_q_base else 'N/A'}"
              f"→{parsed_q_pert.get('classification') if parsed_q_pert else 'N/A'}  "
              f"M: {parsed_m_base.get('classification') if parsed_m_base else 'N/A'}"
              f"→{parsed_m_pert.get('classification') if parsed_m_pert else 'N/A'}")
        print(f"    Q_base: {exp_q_base[:100]}")
        print(f"    Q_pert: {exp_q_pert[:100]}")
        print(f"    M_base: {exp_m_base[:100]}")
        print(f"    M_pert: {exp_m_pert[:100]}")

        all_val_results.append({
            "dataset": dataset, "class": cls,
            "qwen_sim": sim_q, "mistral_sim": sim_m,
            "diff": diff,
            "qwen_overlap": jaccard_overlap(exp_q_base, exp_q_pert),
            "mistral_overlap": jaccard_overlap(exp_m_base, exp_m_pert),
            "qwen_pred_stable": q_stable,
            "mistral_pred_stable": m_stable,
            "qwen_base_class": parsed_q_base.get("classification") if parsed_q_base else None,
            "qwen_pert_class": parsed_q_pert.get("classification") if parsed_q_pert else None,
            "mistral_base_class": parsed_m_base.get("classification") if parsed_m_base else None,
            "mistral_pert_class": parsed_m_pert.get("classification") if parsed_m_pert else None,
            "qwen_base_exp": exp_q_base,
            "qwen_pert_exp": exp_q_pert,
            "mistral_base_exp": exp_m_base,
            "mistral_pert_exp": exp_m_pert,
            "false_context": fc,
        })

val_df = pd.DataFrame(all_val_results)

# ================================================================
# PART 2 — VERIFICATION (5 windows per class)
# ================================================================
print(f"\n\n{'='*70}")
print("PART 2 — VERIFICATION (5 windows per class)")
print("="*70)

N = 5
ver_results = []

for dataset, classes, df in [
    ("HCRL", HCRL_CLASSES, hcrl_p6),
    ("ROAD", ROAD_CLASSES, road_p6)
]:
    print(f"\n--- {dataset} ---")
    label_col = "true_label_name" if dataset == "HCRL" else "label_name"

    for cls in classes:
        subset = df[df[label_col] == cls].head(N)
        q_correct = 0
        m_correct = 0

        for _, row in subset.iterrows():
            text = row["original_text"]
            fc   = row["false_context"]
            pred = row.get("securebert_pred_name", "UNKNOWN")
            conf = float(row.get("securebert_confidence", 1.0))

            if dataset == "HCRL":
                resp_q = generate_qwen_hcrl(
                    text, pred, conf, false_context=fc
                )
                resp_m = generate_mistral_hcrl(
                    text, pred, conf, false_context=fc
                )
            else:
                resp_q = generate_qwen_road(text, false_context=fc)
                resp_m = generate_mistral_road(text, false_context=fc)

            pq, _ = safe_parse(resp_q)
            pm, _ = safe_parse(resp_m)

            cls_q = pq.get("classification") if pq else None
            cls_m = pm.get("classification") if pm else None

            if dataset == "ROAD":
                # Attack detected = not NORMAL
                q_ok = cls_q != "NORMAL" if cls != "AMBIENT" \
                    else cls_q == "NORMAL"
                m_ok = cls_m != "NORMAL" if cls != "AMBIENT" \
                    else cls_m == "NORMAL"
            else:
                q_ok = cls_q == cls
                m_ok = cls_m == cls

            if q_ok: q_correct += 1
            if m_ok: m_correct += 1

        print(f"  {cls:<14} Qwen={q_correct}/{N}  "
              f"Mistral={m_correct}/{N}")

        ver_results.append({
            "dataset": dataset, "class": cls,
            "qwen_correct": q_correct,
            "mistral_correct": m_correct,
            "n": N,
        })

ver_df = pd.DataFrame(ver_results)

print(f"\nSUMMARY:")
for dataset in ["HCRL", "ROAD"]:
    s = ver_df[ver_df["dataset"] == dataset]
    qt = s["qwen_correct"].sum()
    mt = s["mistral_correct"].sum()
    total = s["n"].sum()
    print(f"  {dataset}: Qwen={qt}/{total} "
          f"({qt/total*100:.0f}%)  "
          f"Mistral={mt}/{total} "
          f"({mt/total*100:.0f}%)")

# ================================================================
# PART 3 — SUMMARY TABLE
# ================================================================
print(f"\n\n{'='*70}")
print("P6 COMBINED SUMMARY")
print("="*70)

for dataset in ["HCRL", "ROAD"]:
    subset = val_df[val_df["dataset"] == dataset]
    print(f"\n{dataset}:")
    print(f"  {'Class':<14} {'Qwen':>8} {'Mistral':>10} "
          f"{'Diff':>8} {'Q_stable':>10} {'M_stable':>10}")
    print(f"  {'-'*65}")
    for _, r in subset.iterrows():
        q_s = f"{r['qwen_sim']:.3f}" if r['qwen_sim'] else "N/A"
        m_s = f"{r['mistral_sim']:.3f}" if r['mistral_sim'] else "N/A"
        d_s = f"{r['diff']:+.3f}" if r['diff'] else "N/A"
        print(f"  {r['class']:<14} {q_s:>8} {m_s:>10} "
              f"{d_s:>8} {str(r['qwen_pred_stable']):>10} "
              f"{str(r['mistral_pred_stable']):>10}")
    qm = subset["qwen_sim"].dropna().mean()
    mm = subset["mistral_sim"].dropna().mean()
    print(f"\n  Mean: Qwen={qm:.3f}  Mistral={mm:.3f}  "
          f"Diff={mm-qm:+.3f}")

# Classification changes
print(f"\nCLASSIFICATION CHANGES UNDER P6:")
changes = val_df[
    (val_df["qwen_pred_stable"] == False) |
    (val_df["mistral_pred_stable"] == False)
]
if len(changes) > 0:
    for _, r in changes.iterrows():
        print(f"  ⚠ {r['dataset']} {r['class']}: "
              f"Qwen {r['qwen_base_class']}→{r['qwen_pert_class']}  "
              f"Mistral {r['mistral_base_class']}→{r['mistral_pert_class']}")
else:
    print("  ✓ No classification changes")

# ================================================================
# PART 4 — VISUALIZATION
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "P6 False Context Injection — Fixed Dataset\n"
    "HCRL + ROAD — Qwen2.5-1.5B vs Mistral-7B",
    fontsize=13, fontweight="bold"
)

COLORS = {"Qwen": "#2196F3", "Mistral": "#FF5722"}
width = 0.35

for plot_idx, (dataset, classes) in enumerate(
    [("HCRL", HCRL_CLASSES), ("ROAD", ROAD_CLASSES)]
):
    ax_sim  = axes[0][plot_idx]
    ax_diff = axes[1][plot_idx]

    subset = val_df[val_df["dataset"] == dataset]
    x = np.arange(len(classes))

    q_sims = [
        subset[subset["class"]==c]["qwen_sim"].values[0]
        if len(subset[subset["class"]==c]) > 0 else 0
        for c in classes
    ]
    m_sims = [
        subset[subset["class"]==c]["mistral_sim"].values[0]
        if len(subset[subset["class"]==c]) > 0 else 0
        for c in classes
    ]
    diffs = [
        subset[subset["class"]==c]["diff"].values[0]
        if len(subset[subset["class"]==c]) > 0 else 0
        for c in classes
    ]

    # Similarity bars
    b1 = ax_sim.bar(x - width/2, q_sims, width,
                     label="Qwen", color=COLORS["Qwen"], alpha=0.85)
    b2 = ax_sim.bar(x + width/2, m_sims, width,
                     label="Mistral", color=COLORS["Mistral"], alpha=0.85)
    ax_sim.axhline(y=0.85, color="red", linestyle="--",
                   alpha=0.6, label="Drift threshold")
    ax_sim.set_xticks(x)
    ax_sim.set_xticklabels(
        [c[:5] for c in classes], fontsize=9
    )
    ax_sim.set_ylim(0.3, 1.05)
    ax_sim.set_ylabel("Cosine Similarity")
    ax_sim.set_title(f"{dataset} P6 — Cosine Similarity")
    ax_sim.legend(fontsize=8)
    for bar in b1:
        ax_sim.text(bar.get_x() + bar.get_width()/2.,
                    bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}',
                    ha='center', va='bottom', fontsize=7,
                    color=COLORS["Qwen"], fontweight="bold")
    for bar in b2:
        ax_sim.text(bar.get_x() + bar.get_width()/2.,
                    bar.get_height() + 0.01,
                    f'{bar.get_height():.3f}',
                    ha='center', va='bottom', fontsize=7,
                    color=COLORS["Mistral"], fontweight="bold")

    # Difference bars
    colors_d = ["#4CAF50" if d >= 0 else "#F44336"
                for d in diffs]
    bars_d = ax_diff.bar(x, diffs, 0.5,
                          color=colors_d, alpha=0.85)
    ax_diff.axhline(y=0, color="black", linewidth=1)
    ax_diff.set_xticks(x)
    ax_diff.set_xticklabels(
        [c[:5] for c in classes], fontsize=9
    )
    ax_diff.set_ylabel("Mistral − Qwen")
    ax_diff.set_title(
        f"{dataset} P6 — Difference\n"
        "(green=Mistral stable, red=Qwen stable)"
    )
    for bar, d in zip(bars_d, diffs):
        if d is not None:
            ax_diff.text(
                bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.005
                if d >= 0 else bar.get_height() - 0.015,
                f'{d:+.3f}',
                ha='center', va='bottom',
                fontsize=8, fontweight="bold"
            )

plt.tight_layout()
plt.savefig("/kaggle/working/p6_fixed_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figure saved")

# Save all results
val_df.to_parquet(
    "/kaggle/working/p6_validation_results.parquet",
    index=False
)
ver_df.to_parquet(
    "/kaggle/working/p6_verification_results.parquet",
    index=False
)
print("✓ Results saved")
print(f"\nTotal inference calls: "
      f"{len(all_val_results)*4 + len(ver_results)*N*2}")

In [ ]:
import zipfile, os

zip_path = "/kaggle/working/progress_backup.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")

from IPython.display import FileLink
FileLink('/kaggle/working/progress_backup.zip')

In [ ]:
import zipfile, os

zip_path = "/kaggle/working/progress_backup.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if f.endswith(".parquet") or f.endswith(".png"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")

print(f"\n✓ Size: {os.path.getsize(zip_path)/1e6:.2f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/progress_backup.zip')

In [ ]:
# PREREQUISITE CHECK
import torch
import pandas as pd
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("PREREQUISITE CHECK")
print("="*60)

# 1. GPU
print(f"\n1. GPU:")
print(f"   CUDA available: {torch.cuda.is_available()}")
print(f"   GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"   GPU name: {torch.cuda.get_device_name(0)}")

# 2. Models loaded
print(f"\n2. Models:")
try:
    print(f"   Qwen loaded: {qwen_model is not None}")
    print(f"   Qwen device: {next(qwen_model.parameters()).device}")
except NameError:
    print(f"   ✗ Qwen NOT loaded — run cells 1-4 first")

try:
    print(f"   Mistral loaded: {mistral_model is not None}")
    print(f"   Mistral device: {next(mistral_model.parameters()).device}")
except NameError:
    print(f"   ✗ Mistral NOT loaded — load Mistral first")

# 3. Input files
print(f"\n3. Input files:")
INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark"
)
required_files = [
    "task4_road_windows.parquet",
    "task4_road_p6_dataset.parquet",
]
for f in required_files:
    path = INPUT_DIR / f
    exists = path.exists()
    size = path.stat().st_size / 1e6 if exists else 0
    print(f"   {'✓' if exists else '✗'} {f} ({size:.2f} MB)")

# 4. Data integrity
print(f"\n4. Data integrity:")
road_df = pd.read_parquet(INPUT_DIR / "task4_road_windows.parquet")
road_p6 = pd.read_parquet(INPUT_DIR / "task4_road_p6_dataset.parquet")
print(f"   ROAD windows: {len(road_df)} "
      f"(expected 250)")
print(f"   ROAD P6: {len(road_p6)} "
      f"(expected 250)")
print(f"   ROAD classes: "
      f"{road_df['label_name'].value_counts().to_dict()}")
print(f"   ROAD P6 classes: "
      f"{road_p6['label_name'].value_counts().to_dict()}")

# 5. Quick generation test
print(f"\n5. Generation test (1 window):")
try:
    sample = road_df.iloc[0]
    text = sample["text"]

    # Qwen test
    t0 = __import__("time").time()
    resp = generate_qwen_road(text, steering="")
    parsed, _ = safe_parse(resp)
    elapsed = __import__("time").time() - t0
    print(f"   Qwen: {'✓ valid JSON' if parsed else '✗ invalid JSON'} "
          f"({elapsed:.1f}s)")
    if parsed:
        print(f"   Qwen class: {parsed.get('classification')}")

    # Mistral test
    t0 = __import__("time").time()
    resp = generate_mistral_road(text, steering="")
    parsed, _ = safe_parse(resp)
    elapsed = __import__("time").time() - t0
    print(f"   Mistral: {'✓ valid JSON' if parsed else '✗ invalid JSON'} "
          f"({elapsed:.1f}s)")
    if parsed:
        print(f"   Mistral class: {parsed.get('classification')}")

except Exception as e:
    print(f"   ✗ Generation test failed: {e}")

# 6. Embedder test
print(f"\n6. Embedder test:")
try:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    e = embedder.encode(["test sentence"])
    print(f"   ✓ Embedder ready, vector dim: {e.shape[1]}")
except Exception as e:
    print(f"   ✗ Embedder failed: {e}")

# 7. Disk space
print(f"\n7. Disk space:")
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"   Free: {free/1e9:.1f} GB")
print(f"   (Need ~2 GB for all output files)")

print(f"\n{'='*60}")
print("SUMMARY")
print("="*60)
print("If all checks show ✓ — safe to start full inference")
print("If any ✗ — fix before starting")

In [ ]:
# COMBINED INFERENCE — 3 runs back to back
# Run 1: Qwen ROAD P2+P3+P4
# Run 2: Mistral ROAD P5 full
# Run 3: Mistral ROAD P6 full
# Total estimated: ~8.3 hours

import pandas as pd
import numpy as np
import json
import time
import copy
import random
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07/icidea-llm-ids-benchmark"
)

road_df   = pd.read_parquet(INPUT_DIR / "task4_road_windows.parquet")
road_p6   = pd.read_parquet(INPUT_DIR / "task4_road_p6_dataset.parquet")

embedder  = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Data loaded")
print(f"  ROAD windows: {len(road_df)}")
print(f"  ROAD P6: {len(road_p6)}")

ROAD_STEERING = "Note: Pay particular attention to \
inter-frame timing irregularities and arrival interval patterns."

ROAD_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

QWEN_SYSTEM = ROAD_SYSTEM

def frames_to_text(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex = " ".join(f"{b:02X}" for b in data_bytes)
        ts = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

def perturb_p2(frames, seed=42):
    rng = random.Random(seed)
    perturbed = copy.deepcopy(frames)
    insert_idx = len(frames) // 2
    ambient_ids = [0x162, 0x0A7, 0x464, 0x577, 0x230, 0x3B9]
    inserted = {
        "timestamp": perturbed[insert_idx]["timestamp"] + 0.001,
        "can_id":    rng.choice(ambient_ids),
        "dlc":       8,
        "data":      [rng.randint(0, 255) for _ in range(8)]
    }
    perturbed.insert(insert_idx, inserted)
    return frames_to_text(perturbed[:14])

def perturb_p3(frames, seed=42):
    perturbed = copy.deepcopy(frames)
    idx = len(frames) // 2
    if idx + 1 < len(perturbed):
        perturbed[idx], perturbed[idx+1] = \
            perturbed[idx+1], perturbed[idx]
    return frames_to_text(perturbed)

def perturb_p4_road(original_text):
    # P4 for ROAD: add uncertainty note
    return original_text, \
        "Note: The classifier confidence for this sample " \
        "is moderate (0.623). Please analyze carefully."

def generate_qwen_road(text, steering=""):
    user_content = "Analyze this CAN bus telemetry sequence and classify it."
    if steering:
        user_content += f"\n\n{steering}"
    user_content += f"\n\n{text}\n\nProvide your classification and detailed explanation in JSON format."
    messages = [
        {"role": "system", "content": QWEN_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_mistral_road(text, steering=""):
    user_content = "Analyze this CAN bus telemetry sequence and classify it."
    if steering:
        user_content += f"\n\n{steering}"
    user_content += f"\n\n{text}\n\nProvide your classification and detailed explanation in JSON format."
    messages = [
        {"role": "user",
         "content": f"{ROAD_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except:
        return None, "parse_error"

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

# ================================================================
# RUN 1 — QWEN ROAD P2 + P3 + P4
# ================================================================
print("\n" + "="*60)
print("RUN 1 — QWEN ROAD P2 + P3 + P4")
print(f"Windows: {len(road_df)} × 3 perturbations")
print(f"Estimated: ~2.5 hours")
print("="*60)

qwen_road_results = []
errors = 0

for ptype in ["P2_INSERTION", "P3_REORDER", "P4_PROMPT"]:
    print(f"\n--- {ptype} ---")
    for idx, row in tqdm(
        road_df.iterrows(),
        total=len(road_df),
        desc=f"Qwen ROAD {ptype}"
    ):
        t0 = time.time()
        try:
            original_text = row["text"]
            frames = json.loads(row["frames_json"])

            # Baseline
            resp_base = generate_qwen_road(original_text, steering="")
            parsed_base, _ = safe_parse(resp_base)
            exp_base = parsed_base.get("explanation","") \
                if parsed_base else ""

            # Perturbed
            if ptype == "P2_INSERTION":
                pert_text = perturb_p2(frames, seed=42)
                steering  = ""
            elif ptype == "P3_REORDER":
                pert_text = perturb_p3(frames, seed=42)
                steering  = ""
            else:  # P4
                pert_text, steering = perturb_p4_road(original_text)

            resp_pert = generate_qwen_road(pert_text, steering=steering)
            parsed_pert, _ = safe_parse(resp_pert)
            exp_pert = parsed_pert.get("explanation","") \
                if parsed_pert else ""

            sim = cosine_sim(exp_base, exp_pert)

            qwen_road_results.append({
                "label_name":        row["label_name"],
                "perturbation_type": ptype,
                "model":             "qwen",
                "dataset":           "ROAD",
                "baseline_exp":      exp_base,
                "perturbed_exp":     exp_pert,
                "baseline_class":    parsed_base.get("classification") if parsed_base else None,
                "perturbed_class":   parsed_pert.get("classification") if parsed_pert else None,
                "cosine_similarity": sim,
                "json_valid":        parsed_base is not None and parsed_pert is not None,
                "inference_time_s":  time.time() - t0,
            })

        except Exception as e:
            errors += 1
            qwen_road_results.append({
                "label_name":        row["label_name"],
                "perturbation_type": ptype,
                "model":             "qwen",
                "dataset":           "ROAD",
                "baseline_exp":      "",
                "perturbed_exp":     "",
                "baseline_class":    None,
                "perturbed_class":   None,
                "cosine_similarity": None,
                "json_valid":        False,
                "inference_time_s":  time.time() - t0,
            })

        if len(qwen_road_results) % 50 == 0:
            cp = pd.DataFrame(qwen_road_results)
            cp.to_parquet(
                f"/kaggle/working/qwen_road_p234_checkpoint_"
                f"{len(qwen_road_results)}.parquet"
            )
            valid = sum(r["json_valid"] for r in qwen_road_results)
            print(f"  Checkpoint {len(qwen_road_results)}: "
                  f"valid={valid}/{len(qwen_road_results)}")

qwen_road_df = pd.DataFrame(qwen_road_results)
qwen_road_df.to_parquet(
    "/kaggle/working/qwen_road_p234_full.parquet", index=False
)
print(f"\n✓ Run 1 complete: {len(qwen_road_df)} records, "
      f"errors={errors}")
print(f"  JSON valid: {qwen_road_df['json_valid'].sum()}"
      f"/{len(qwen_road_df)}")

# Per-perturbation summary
for ptype in ["P2_INSERTION", "P3_REORDER", "P4_PROMPT"]:
    subset = qwen_road_df[qwen_road_df["perturbation_type"]==ptype]
    sims = subset["cosine_similarity"].dropna()
    print(f"  {ptype}: mean={sims.mean():.3f} std={sims.std():.3f}")

# ================================================================
# RUN 2 — MISTRAL ROAD P5 FULL
# ================================================================
print("\n" + "="*60)
print("RUN 2 — MISTRAL ROAD P5 FULL")
print(f"Windows: {len(road_df)}")
print(f"Estimated: ~3.8 hours")
print("="*60)

mistral_road_p5 = []
errors = 0

for idx, row in tqdm(
    road_df.iterrows(),
    total=len(road_df),
    desc="Mistral ROAD P5"
):
    t0 = time.time()
    try:
        text = row["text"]

        resp_base = generate_mistral_road(text, steering="")
        parsed_base, _ = safe_parse(resp_base)
        exp_base = parsed_base.get("explanation","") \
            if parsed_base else ""

        resp_pert = generate_mistral_road(
            text, steering=ROAD_STEERING
        )
        parsed_pert, _ = safe_parse(resp_pert)
        exp_pert = parsed_pert.get("explanation","") \
            if parsed_pert else ""

        sim = cosine_sim(exp_base, exp_pert)

        mistral_road_p5.append({
            "label_name":        row["label_name"],
            "perturbation_type": "P5_STEERING",
            "model":             "mistral",
            "dataset":           "ROAD",
            "baseline_exp":      exp_base,
            "perturbed_exp":     exp_pert,
            "baseline_class":    parsed_base.get("classification") if parsed_base else None,
            "perturbed_class":   parsed_pert.get("classification") if parsed_pert else None,
            "cosine_similarity": sim,
            "json_valid":        parsed_base is not None and parsed_pert is not None,
            "inference_time_s":  time.time() - t0,
        })

    except Exception as e:
        errors += 1
        mistral_road_p5.append({
            "label_name":        row["label_name"],
            "perturbation_type": "P5_STEERING",
            "model":             "mistral",
            "dataset":           "ROAD",
            "baseline_exp":      "",
            "perturbed_exp":     "",
            "baseline_class":    None,
            "perturbed_class":   None,
            "cosine_similarity": None,
            "json_valid":        False,
            "inference_time_s":  time.time() - t0,
        })

    if len(mistral_road_p5) % 50 == 0:
        cp = pd.DataFrame(mistral_road_p5)
        cp.to_parquet(
            f"/kaggle/working/mistral_road_p5_checkpoint_"
            f"{len(mistral_road_p5)}.parquet"
        )
        valid = sum(r["json_valid"] for r in mistral_road_p5)
        print(f"  Checkpoint {len(mistral_road_p5)}: "
              f"valid={valid}/{len(mistral_road_p5)}")

mistral_p5_df = pd.DataFrame(mistral_road_p5)
mistral_p5_df.to_parquet(
    "/kaggle/working/mistral_road_p5_full.parquet", index=False
)
print(f"\n✓ Run 2 complete: {len(mistral_p5_df)} records, "
      f"errors={errors}")
print(f"  JSON valid: {mistral_p5_df['json_valid'].sum()}"
      f"/{len(mistral_p5_df)}")

sims = mistral_p5_df["cosine_similarity"].dropna()
print(f"  P5 mean: {sims.mean():.3f}  std: {sims.std():.3f}")
print(f"  Qwen P5 ROAD mean (Task 4): 0.864")
print(f"  Difference: {sims.mean()-0.864:+.3f}")

# ================================================================
# RUN 3 — MISTRAL ROAD P6 FULL
# ================================================================
print("\n" + "="*60)
print("RUN 3 — MISTRAL ROAD P6 FULL")
print(f"Windows: {len(road_p6)}")
print(f"Estimated: ~2 hours")
print("="*60)

mistral_road_p6 = []
errors = 0

for idx, row in tqdm(
    road_p6.iterrows(),
    total=len(road_p6),
    desc="Mistral ROAD P6"
):
    t0 = time.time()
    try:
        text = row["original_text"]
        fc   = row["false_context"]

        resp_base = generate_mistral_road(text, steering="")
        parsed_base, _ = safe_parse(resp_base)
        exp_base = parsed_base.get("explanation","") \
            if parsed_base else ""

        resp_pert = generate_mistral_road(text, steering=fc)
        parsed_pert, _ = safe_parse(resp_pert)
        exp_pert = parsed_pert.get("explanation","") \
            if parsed_pert else ""

        sim = cosine_sim(exp_base, exp_pert)

        mistral_road_p6.append({
            "label_name":        row["label_name"],
            "perturbation_type": "P6_FALSE_CONTEXT",
            "model":             "mistral",
            "dataset":           "ROAD",
            "baseline_exp":      exp_base,
            "perturbed_exp":     exp_pert,
            "baseline_class":    parsed_base.get("classification") if parsed_base else None,
            "perturbed_class":   parsed_pert.get("classification") if parsed_pert else None,
            "cosine_similarity": sim,
            "json_valid":        parsed_base is not None and parsed_pert is not None,
            "inference_time_s":  time.time() - t0,
        })

    except Exception as e:
        errors += 1
        mistral_road_p6.append({
            "label_name":        row["label_name"],
            "perturbation_type": "P6_FALSE_CONTEXT",
            "model":             "mistral",
            "dataset":           "ROAD",
            "baseline_exp":      "",
            "perturbed_exp":     "",
            "baseline_class":    None,
            "perturbed_class":   None,
            "cosine_similarity": None,
            "json_valid":        False,
            "inference_time_s":  time.time() - t0,
        })

    if len(mistral_road_p6) % 50 == 0:
        cp = pd.DataFrame(mistral_road_p6)
        cp.to_parquet(
            f"/kaggle/working/mistral_road_p6_checkpoint_"
            f"{len(mistral_road_p6)}.parquet"
        )
        valid = sum(r["json_valid"] for r in mistral_road_p6)
        print(f"  Checkpoint {len(mistral_road_p6)}: "
              f"valid={valid}/{len(mistral_road_p6)}")

mistral_p6_df = pd.DataFrame(mistral_road_p6)
mistral_p6_df.to_parquet(
    "/kaggle/working/mistral_road_p6_full.parquet", index=False
)
print(f"\n✓ Run 3 complete: {len(mistral_p6_df)} records, "
      f"errors={errors}")
print(f"  JSON valid: {mistral_p6_df['json_valid'].sum()}"
      f"/{len(mistral_p6_df)}")

sims = mistral_p6_df["cosine_similarity"].dropna()
print(f"  P6 mean: {sims.mean():.3f}  std: {sims.std():.3f}")

# ================================================================
# FINAL ZIP
# ================================================================
import zipfile, os
zip_path = "/kaggle/working/tonight_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if f.endswith(".parquet") or f.endswith(".png"):
            zipf.write(f"/kaggle/working/{f}", f)
print(f"\n✓ All results zipped: "
      f"{os.path.getsize(zip_path)/1e6:.1f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/tonight_results.zip')

In [ ]:
import zipfile, os
zip_path = "/kaggle/working/tonight_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if f.endswith(".parquet") or f.endswith(".png"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")
print(f"✓ Size: {os.path.getsize(zip_path)/1e6:.1f} MB")
from IPython.display import FileLink
FileLink('/kaggle/working/tonight_results.zip')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ARTIFACTS_DIR = Path.home() / "icidea_llm_ids" / "artifacts"
RESULTS_DIR   = Path.home() / "icidea_llm_ids" / "results"

print("LOCAL ARTIFACTS")
print("="*60)
for f in sorted(ARTIFACTS_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / 1e6
    try:
        df = pd.read_parquet(f)
        print(f"  {f.name:<50} {size_mb:>6.2f} MB  {len(df):>6} rows")
    except:
        print(f"  {f.name:<50} {size_mb:>6.2f} MB  (unreadable)")

In [ ]:
# PREREQUISITE CHECK — Mistral ROAD P1+P2+P3+P4
import torch
import json
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("PREREQUISITE CHECK")
print("="*60)

# 1. GPU
print(f"\n1. GPU:")
print(f"   CUDA available: {torch.cuda.is_available()}")
print(f"   GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# 2. Models
print(f"\n2. Models:")
try:
    print(f"   Mistral loaded: {mistral_model is not None}")
    print(f"   Mistral device: "
          f"{next(mistral_model.parameters()).device}")
    params = sum(p.numel()
                 for p in mistral_model.parameters()) / 1e9
    print(f"   Mistral params: {params:.2f}B")
except NameError:
    print(f"   ✗ Mistral NOT loaded — load Mistral first")

# 3. Input files
print(f"\n3. Input files:")
INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)
required = ["task4_road_windows.parquet"]
for f in required:
    path  = INPUT_DIR / f
    exists = path.exists()
    size   = path.stat().st_size / 1e6 if exists else 0
    print(f"   {'✓' if exists else '✗'} {f} ({size:.2f} MB)")

# 4. Data integrity
print(f"\n4. Data integrity:")
road_df = pd.read_parquet(
    INPUT_DIR / "task4_road_windows.parquet"
)
print(f"   ROAD windows: {len(road_df)} (expected 250)")
print(f"   Classes: "
      f"{road_df['label_name'].value_counts().to_dict()}")
print(f"   Has frames_json: "
      f"{'frames_json' in road_df.columns}")

# Check one frame parses correctly
sample_frames = json.loads(road_df.iloc[0]["frames_json"])
print(f"   Sample frames: {len(sample_frames)} frames ✓")

# 5. Perturbation functions test
print(f"\n5. Perturbation function test:")

def frames_to_text_test(frames):
    n = len(frames)
    lines = [f"CAN Bus Telemetry Sequence ({n} frames):"]
    for i, frame in enumerate(frames, start=1):
        can_id_hex = f"{frame['can_id']:04X}"
        dlc        = frame["dlc"]
        data_bytes = frame["data"][:dlc]
        data_hex   = " ".join(f"{b:02X}" for b in data_bytes)
        ts         = frame["timestamp"]
        lines.append(
            f"[{i:03d}] T={ts:.3f} ID={can_id_hex} "
            f"DLC={dlc} DATA={data_hex}"
        )
    return "\n".join(lines)

# P1 test
frames_test = copy.deepcopy(sample_frames)
rng = random.Random(42)
target = frames_test[len(frames_test)//2]
byte_idx = rng.randint(0, 7)
orig_val = target["data"][byte_idx]
target["data"][byte_idx] = (orig_val + 1) % 256
p1_text = frames_to_text_test(frames_test)
print(f"   P1 mutation: byte {byte_idx}: "
      f"{orig_val}→{target['data'][byte_idx]} ✓")

# P2 test
frames_test2 = copy.deepcopy(sample_frames)
frames_test2.insert(7, {
    "timestamp": frames_test2[7]["timestamp"] + 0.001,
    "can_id": 0x162, "dlc": 8,
    "data": [0]*8
})
p2_text = frames_to_text_test(frames_test2[:14])
print(f"   P2 insertion: {len(frames_test2[:14])} frames ✓")

# P3 test
frames_test3 = copy.deepcopy(sample_frames)
frames_test3[7], frames_test3[8] = \
    frames_test3[8], frames_test3[7]
p3_text = frames_to_text_test(frames_test3)
print(f"   P3 reorder: frames 7,8 swapped ✓")

print(f"   P4: confidence note added to prompt ✓")

# 6. Generation test
print(f"\n6. Generation test (1 window, Mistral):")
try:
    ROAD_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

    sample_text = road_df.iloc[0]["text"]
    user_content = \
        f"Analyze this CAN bus telemetry sequence and classify it." \
        f"\n\n{sample_text}\n\nProvide your classification and " \
        f"detailed explanation in JSON format."
    messages = [
        {"role": "user",
         "content": f"{ROAD_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)

    t0 = __import__("time").time()
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.1, do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    elapsed = __import__("time").time() - t0

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response   = mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    import json as json_mod
    try:
        clean = response.strip()
        if "```" in clean:
            parts = clean.split("```")
            for part in parts:
                part = part.strip()
                if part.startswith("json"):
                    part = part[4:].strip()
                if part.startswith("{"):
                    clean = part
                    break
        parsed = json_mod.loads(clean)
        print(f"   ✓ Valid JSON ({elapsed:.1f}s)")
        print(f"   Classification: "
              f"{parsed.get('classification')}")
        print(f"   Explanation: "
              f"{parsed.get('explanation','')[:100]}")
    except:
        print(f"   ✗ Invalid JSON ({elapsed:.1f}s)")
        print(f"   Response: {response[:150]}")

except NameError as e:
    print(f"   ✗ Generation failed: {e}")

# 7. Embedder test
print(f"\n7. Embedder test:")
try:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    e = embedder.encode(["test sentence"])
    print(f"   ✓ Embedder ready, dim={e.shape[1]}")
except Exception as e:
    print(f"   ✗ Embedder failed: {e}")

# 8. Disk space
print(f"\n8. Disk space:")
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"   Free: {free/1e9:.1f} GB (need ~2 GB)")

# 9. Estimated time
print(f"\n9. Time estimate:")
windows_per_ptype = len(road_df)
n_ptypes          = 4
runs_per_window   = 2
secs_per_run      = 30
total_secs = windows_per_ptype * n_ptypes * \
             runs_per_window * secs_per_run
print(f"   {windows_per_ptype} windows × {n_ptypes} types × "
      f"{runs_per_window} runs × {secs_per_run}s")
print(f"   Estimated total: {total_secs/3600:.1f} hours")

print(f"\n{'='*60}")
print("SUMMARY")
print("="*60)
print("If all checks show ✓ — safe to start full inference")
print("If any ✗ — fix before starting")

In [ ]:
import torch
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("PREREQUISITE CHECK — Qwen P6 HCRL + ROAD")
print("="*60)

# 1. GPU
print(f"\n1. GPU:")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# 2. Models
print(f"\n2. Models:")
try:
    print(f"   Qwen loaded: {qwen_model is not None}")
    print(f"   Device: {next(qwen_model.parameters()).device}")
except NameError:
    print(f"   ✗ Qwen NOT loaded")

# 3. Input files
print(f"\n3. Input files:")
INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)
for f in [
    "section10_p6_false_context.parquet",
    "task4_road_p6_dataset.parquet"
]:
    path = INPUT_DIR / f
    exists = path.exists()
    if exists:
        df = pd.read_parquet(path)
        print(f"   ✓ {f}: {len(df)} rows")
        # Verify DOS text
        if "section10" in f:
            dos = df[df["true_label_name"]=="DOS"].iloc[0]
            correct = "ID=0000" in dos["original_text"]
            print(f"     DOS text correct: {correct}")
            print(f"     SecureBERT pred: "
                  f"{dos['securebert_pred_name']}")
    else:
        print(f"   ✗ {f} NOT FOUND")

# 4. Quick generation test
print(f"\n4. Generation test:")
try:
    INPUT_DIR2 = Path(
        "/kaggle/input/datasets/deepakpatnaik07"
        "/icidea-llm-ids-benchmark"
    )
    hcrl_p6 = pd.read_parquet(
        INPUT_DIR2 / "section10_p6_false_context.parquet"
    )
    sample = hcrl_p6[
        hcrl_p6["true_label_name"] == "DOS"
    ].iloc[0]

    import json, time
    HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
Respond ONLY in valid JSON:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float>,
    "explanation": "<reasoning>",
    "key_indicators": ["<ind1>", "<ind2>"],
    "temporal_reasoning": "<pattern observations>"
}"""

    messages = [
        {"role": "system", "content": HCRL_SYSTEM},
        {"role": "user", "content":
            f"SecureBERT: {sample['securebert_pred_name']} "
            f"({sample['securebert_confidence']:.3f})\n\n"
            f"{sample['original_text']}\n\nClassify in JSON."}
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt"
    ).to(device)
    t0 = time.time()
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )
    elapsed = time.time() - t0
    resp = qwen_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
    clean = resp
    if "```" in clean:
        for part in clean.split("```"):
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    parsed = json.loads(clean)
    print(f"   ✓ Valid JSON ({elapsed:.1f}s)")
    print(f"   Classification: {parsed.get('classification')}")
    print(f"   Explanation: "
          f"{parsed.get('explanation','')[:80]}")
except Exception as e:
    print(f"   ✗ Failed: {e}")

# 5. Disk
print(f"\n5. Disk space:")
import shutil
_, _, free = shutil.disk_usage("/kaggle/working")
print(f"   Free: {free/1e9:.1f} GB")

print(f"\n{'='*60}")
print("If all ✓ — safe to run P6 inference")

In [ ]:
try:
    print(f"Mistral loaded: {mistral_model is not None}")
    print(f"Mistral device: {next(mistral_model.parameters()).device}")
    print(f"Mistral params: {sum(p.numel() for p in mistral_model.parameters())/1e9:.2f}B")
except NameError:
    print("Mistral NOT loaded")

In [ ]:
# P6 FALSE CONTEXT INJECTION — FIXED COMPLETE VERSION
# Qwen + Mistral × HCRL + ROAD
# Estimated: ~15 hours total

import pandas as pd
import numpy as np
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)

hcrl_p6  = pd.read_parquet(
    INPUT_DIR / "section10_p6_false_context.parquet"
)
road_p6  = pd.read_parquet(
    INPUT_DIR / "task4_road_p6_dataset.parquet"
)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✓ HCRL P6: {len(hcrl_p6)} records")
print(f"  Classes: "
      f"{hcrl_p6['true_label_name'].value_counts().to_dict()}")
print(f"✓ ROAD P6: {len(road_p6)} records")
print(f"  Classes: "
      f"{road_p6['label_name'].value_counts().to_dict()}")
print(f"✓ Embedder ready")
print(f"✓ Qwen: {qwen_model is not None}")
print(f"✓ Mistral: {mistral_model is not None}")

# ================================================================
# SYSTEM PROMPTS
# ================================================================
HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

ROAD_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, ATTACK, FUZZING, SPOOFING, ANOMALY
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|ATTACK|FUZZING|SPOOFING|ANOMALY>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

# ================================================================
# GENERATION FUNCTIONS
# Greedy decoding (do_sample=False) — deterministic output
# Temperature parameter inactive when do_sample=False
# ================================================================
def generate_qwen_hcrl(text, pred, conf, false_context=""):
    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: {pred} "
        f"(confidence: {conf:.3f})\n"
    )
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += (
        f"\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "system", "content": HCRL_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=1024
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_mistral_hcrl(text, pred, conf, false_context=""):
    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: {pred} "
        f"(confidence: {conf:.3f})\n"
    )
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += (
        f"\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{HCRL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_qwen_road(text, false_context=""):
    user_content = (
        "Analyze this CAN bus telemetry sequence "
        "and classify it."
    )
    if false_context:
        user_content += f"\n\n{false_context}"
    user_content += (
        f"\n\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in JSON format."
    )
    messages = [
        {"role": "system", "content": ROAD_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=1024
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_mistral_road(text, false_context=""):
    user_content = (
        "Analyze this CAN bus telemetry sequence "
        "and classify it."
    )
    if false_context:
        user_content += f"\n\n{false_context}"
    user_content += (
        f"\n\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{ROAD_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

def run_p6_inference(df, dataset, model_name,
                     gen_fn, label_col,
                     pred_col=None, conf_col=None):
    """
    P6 inference for one model-dataset combination.
    Runs baseline (no false context) and perturbed
    (with false context) for each window.
    """
    results = []
    errors  = 0

    for idx, row in tqdm(
        df.iterrows(), total=len(df),
        desc=f"{model_name} {dataset} P6"
    ):
        t0 = time.time()
        try:
            text = row["original_text"]
            fc   = row["false_context"]
            cls  = row[label_col]

            # Build generation kwargs
            if pred_col and conf_col:
                common_kwargs = {
                    "text": text,
                    "pred": row[pred_col],
                    "conf": float(row[conf_col]),
                }
            else:
                common_kwargs = {"text": text}

            # Baseline — no false context
            resp_base = gen_fn(
                **common_kwargs, false_context=""
            )
            parsed_base, _ = safe_parse(resp_base)
            exp_base  = parsed_base.get("explanation", "") \
                if parsed_base else ""
            cls_base  = parsed_base.get("classification") \
                if parsed_base else None
            conf_base = parsed_base.get("confidence") \
                if parsed_base else None

            # Perturbed — with false context
            resp_pert = gen_fn(
                **common_kwargs, false_context=fc
            )
            parsed_pert, _ = safe_parse(resp_pert)
            exp_pert  = parsed_pert.get("explanation", "") \
                if parsed_pert else ""
            cls_pert  = parsed_pert.get("classification") \
                if parsed_pert else None
            conf_pert = parsed_pert.get("confidence") \
                if parsed_pert else None

            sim = cosine_sim(exp_base, exp_pert)

            results.append({
                "window_idx":              row.get(
                    "window_idx", idx),
                "true_label":              cls,
                "perturbation_type":       "P6_FALSE_CONTEXT",
                "model":                   model_name,
                "dataset":                 dataset,
                "baseline_explanation":    exp_base,
                "perturbed_explanation":   exp_pert,
                "baseline_classification": cls_base,
                "perturbed_classification":cls_pert,
                "baseline_confidence":     conf_base,
                "perturbed_confidence":    conf_pert,
                "cosine_similarity":       sim,
                "pred_stable":             cls_base == cls_pert
                    if cls_base and cls_pert else None,
                "correct_baseline":        cls_base == cls
                    if cls_base else None,
                "correct_perturbed":       cls_pert == cls
                    if cls_pert else None,
                "json_valid":              parsed_base is not None
                    and parsed_pert is not None,
                "false_context":           fc,
                "inference_time_s":        time.time() - t0,
            })

        except Exception as e:
            errors += 1
            results.append({
                "window_idx":              row.get(
                    "window_idx", idx),
                "true_label":              row[label_col],
                "perturbation_type":       "P6_FALSE_CONTEXT",
                "model":                   model_name,
                "dataset":                 dataset,
                "baseline_explanation":    "",
                "perturbed_explanation":   "",
                "baseline_classification": None,
                "perturbed_classification":None,
                "baseline_confidence":     None,
                "perturbed_confidence":    None,
                "cosine_similarity":       None,
                "pred_stable":             None,
                "correct_baseline":        None,
                "correct_perturbed":       None,
                "json_valid":              False,
                "false_context":           row.get(
                    "false_context", ""),
                "inference_time_s":        time.time() - t0,
            })

        # Checkpoint every 50
        if len(results) % 50 == 0:
            cp = pd.DataFrame(results)
            fname = (
                f"/kaggle/working/p6_"
                f"{model_name.lower()}_"
                f"{dataset.lower()}_"
                f"checkpoint_{len(results)}.parquet"
            )
            cp.to_parquet(fname)
            valid  = sum(r["json_valid"] for r in results)
            stable = sum(
                r["pred_stable"] == True
                for r in results
                if r["pred_stable"] is not None
            )
            print(f"  [{model_name} {dataset}] "
                  f"Checkpoint {len(results)}: "
                  f"valid={valid}/{len(results)}  "
                  f"stable={stable}/{len(results)}")

    df_out = pd.DataFrame(results)
    fname  = (f"/kaggle/working/p6_"
              f"{model_name.lower()}_"
              f"{dataset.lower()}_full.parquet")
    df_out.to_parquet(fname, index=False)

    print(f"\n✓ {model_name} {dataset} P6 complete:")
    print(f"  Records:     {len(df_out)}, errors={errors}")
    print(f"  JSON valid:  "
          f"{df_out['json_valid'].sum()}/{len(df_out)}")
    print(f"  Mean sim:    "
          f"{df_out['cosine_similarity'].dropna().mean():.3f}")
    print(f"  Pred stable: "
          f"{df_out['pred_stable'].eq(True).sum()}"
          f"/{len(df_out)}")
    print(f"  Per class:")
    for lbl in sorted(df_out["true_label"].unique()):
        s = df_out[df_out["true_label"] == lbl]
        sims = s["cosine_similarity"].dropna()
        stable = s["pred_stable"].eq(True).sum()
        cb = s["correct_baseline"].eq(True).sum()
        cp2 = s["correct_perturbed"].eq(True).sum()
        print(f"    {lbl:<14} sim={sims.mean():.3f}  "
              f"stable={stable}/{len(s)}  "
              f"base_ok={cb}/{len(s)}  "
              f"pert_ok={cp2}/{len(s)}")
    return df_out

# ================================================================
# RUN ALL 4 COMBINATIONS
# ================================================================
print(f"\n{'='*60}")
print("P6 INFERENCE — ALL 4 COMBINATIONS")
print(f"Estimated total: ~15 hours")
print(f"  Qwen HCRL:    ~1.7h")
print(f"  Mistral HCRL: ~8.3h")
print(f"  Qwen ROAD:    ~0.7h")
print(f"  Mistral ROAD: ~4.2h")
print("="*60)

all_results = []

# RUN 1 — Qwen HCRL
print(f"\n{'='*50}")
print(f"RUN 1 — Qwen HCRL ({len(hcrl_p6)} windows)")
print(f"{'='*50}")
r1 = run_p6_inference(
    df        = hcrl_p6,
    dataset   = "HCRL",
    model_name= "Qwen",
    gen_fn    = generate_qwen_hcrl,
    label_col = "true_label_name",
    pred_col  = "securebert_pred_name",
    conf_col  = "securebert_confidence",
)
all_results.append(r1)

# RUN 2 — Mistral HCRL
print(f"\n{'='*50}")
print(f"RUN 2 — Mistral HCRL ({len(hcrl_p6)} windows)")
print(f"{'='*50}")
r2 = run_p6_inference(
    df        = hcrl_p6,
    dataset   = "HCRL",
    model_name= "Mistral",
    gen_fn    = generate_mistral_hcrl,
    label_col = "true_label_name",
    pred_col  = "securebert_pred_name",
    conf_col  = "securebert_confidence",
)
all_results.append(r2)

# RUN 3 — Qwen ROAD
print(f"\n{'='*50}")
print(f"RUN 3 — Qwen ROAD ({len(road_p6)} windows)")
print(f"{'='*50}")
r3 = run_p6_inference(
    df        = road_p6,
    dataset   = "ROAD",
    model_name= "Qwen",
    gen_fn    = generate_qwen_road,
    label_col = "label_name",
    pred_col  = None,
    conf_col  = None,
)
all_results.append(r3)

# RUN 4 — Mistral ROAD
print(f"\n{'='*50}")
print(f"RUN 4 — Mistral ROAD ({len(road_p6)} windows)")
print(f"{'='*50}")
r4 = run_p6_inference(
    df        = road_p6,
    dataset   = "ROAD",
    model_name= "Mistral",
    gen_fn    = generate_mistral_road,
    label_col = "label_name",
    pred_col  = None,
    conf_col  = None,
)
all_results.append(r4)

# ================================================================
# COMBINED SUMMARY
# ================================================================
combined_df = pd.concat(all_results).reset_index(drop=True)
combined_df.to_parquet(
    "/kaggle/working/p6_all_results.parquet",
    index=False
)

print(f"\n\n{'='*60}")
print("P6 COMPLETE — COMBINED SUMMARY")
print("="*60)

print(f"\n{'Model':<10} {'Dataset':<8} "
      f"{'Mean Sim':>10} {'Stable%':>10} {'Valid%':>10}")
print("-"*52)
for model in ["Qwen", "Mistral"]:
    for dataset in ["HCRL", "ROAD"]:
        s = combined_df[
            (combined_df["model"] == model) &
            (combined_df["dataset"] == dataset)
        ]
        if len(s) == 0:
            continue
        mean_sim = s["cosine_similarity"].dropna().mean()
        stable   = s["pred_stable"].eq(True).mean() * 100
        valid    = s["json_valid"].mean() * 100
        print(f"  {model:<8} {dataset:<8} "
              f"{mean_sim:>10.3f} "
              f"{stable:>10.1f}% "
              f"{valid:>10.1f}%")

print(f"\nClassification accuracy under P6:")
print(f"(HCRL: correct if LLM matches true label)")
print(f"(ROAD: label mismatch — shown for reference only)")
print()
for dataset in ["HCRL", "ROAD"]:
    print(f"{dataset}:")
    for model in ["Qwen", "Mistral"]:
        s = combined_df[
            (combined_df["model"] == model) &
            (combined_df["dataset"] == dataset)
        ]
        for lbl in sorted(s["true_label"].unique()):
            sub = s[s["true_label"] == lbl]
            cb  = sub["correct_baseline"].eq(True).mean()*100
            cp2 = sub["correct_perturbed"].eq(True).mean()*100
            print(f"  {model:<8} {lbl:<14} "
                  f"baseline={cb:.0f}%  "
                  f"perturbed={cp2:.0f}%")
    print()

# Zip
import zipfile, os
zip_path = "/kaggle/working/p6_all_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if "p6" in f and f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")
print(f"\n✓ Zip: {os.path.getsize(zip_path)/1e6:.1f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/p6_all_results.zip')

In [ ]:
# QUICK DOS P6 DIAGNOSTIC — max_new_tokens=1024
import pandas as pd
import json
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)
hcrl_p6  = pd.read_parquet(
    INPUT_DIR / "section10_p6_false_context.parquet"
)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_qwen_hcrl(text, pred, conf,
                        false_context="",
                        max_new_tokens=1024):
    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: {pred} "
        f"(confidence: {conf:.3f})\n"
    )
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += (
        f"\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "system", "content": HCRL_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    formatted = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=1024
    ).to(device)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

dos_windows = hcrl_p6[
    hcrl_p6["true_label_name"] == "DOS"
].head(5)

print("DOS P6 DIAGNOSTIC — max_new_tokens=1024")
print("="*60)

for i, (_, row) in enumerate(dos_windows.iterrows()):
    text = row["original_text"]
    pred = row["securebert_pred_name"]
    conf = float(row["securebert_confidence"])
    fc   = row["false_context"]

    import time
    t0 = time.time()
    raw = generate_qwen_hcrl(
        text, pred, conf,
        false_context=fc,
        max_new_tokens=1024
    )
    elapsed = time.time() - t0

    parsed, err = safe_parse(raw)

    print(f"\nWindow {i+1} ({elapsed:.1f}s):")
    print(f"  Raw length: {len(raw)} chars")
    print(f"  Parsed: {parsed is not None}")
    print(f"  Raw response:")
    print(f"    {raw[:600]}")
    if parsed:
        print(f"  Classification: "
              f"{parsed.get('classification')}")
        print(f"  Explanation: "
              f"{parsed.get('explanation','')[:200]}")
    else:
        print(f"  Parse error: {err}")
    print("-"*50)

In [ ]:
# PREREQUISITE CHECK — Mistral P6 HCRL + ROAD
import torch
import pandas as pd
import json
import time
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("PREREQUISITE CHECK — Mistral P6")
print("="*60)

# 1. GPU
print(f"\n1. GPU:")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# 2. Mistral
print(f"\n2. Model:")
try:
    print(f"   Mistral loaded: {mistral_model is not None}")
    print(f"   Device: "
          f"{next(mistral_model.parameters()).device}")
    print(f"   Params: "
          f"{sum(p.numel() for p in mistral_model.parameters())/1e9:.2f}B")
except NameError:
    print(f"   ✗ Mistral NOT loaded")

# 3. Input files
print(f"\n3. Input files:")
INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)
for f in [
    "section10_p6_false_context.parquet",
    "task4_road_p6_dataset.parquet",
]:
    path = INPUT_DIR / f
    if path.exists():
        df = pd.read_parquet(path)
        print(f"   ✓ {f}: {len(df)} rows")
        if "section10" in f:
            dos = df[df["true_label_name"]=="DOS"].iloc[0]
            print(f"     DOS text correct: "
                  f"{'ID=0000' in dos['original_text']}")
            print(f"     SecureBERT DOS pred: "
                  f"{dos['securebert_pred_name']}")
    else:
        print(f"   ✗ {f} NOT FOUND")

# 4. Quick generation test on DOS window
print(f"\n4. Generation test — DOS window:")
try:
    INPUT_DIR2 = Path(
        "/kaggle/input/datasets/deepakpatnaik07"
        "/icidea-llm-ids-benchmark"
    )
    hcrl_p6 = pd.read_parquet(
        INPUT_DIR2 / "section10_p6_false_context.parquet"
    )
    dos_row = hcrl_p6[
        hcrl_p6["true_label_name"] == "DOS"
    ].iloc[0]

    HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: "
        f"{dos_row['securebert_pred_name']} "
        f"(confidence: "
        f"{float(dos_row['securebert_confidence']):.3f})\n"
        f"\n{dos_row['false_context']}\n"
        f"\n{dos_row['original_text']}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{HCRL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)

    t0 = time.time()
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=1024,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    try:
        clean = raw.strip()
        if "```" in clean:
            for part in clean.split("```"):
                part = part.strip()
                if part.startswith("json"):
                    part = part[4:].strip()
                if part.startswith("{"):
                    clean = part
                    break
        parsed = json.loads(clean)
        print(f"   ✓ Valid JSON ({elapsed:.1f}s)")
        print(f"   Classification: "
              f"{parsed.get('classification')}")
        print(f"   Explanation: "
              f"{parsed.get('explanation','')[:150]}")
        print(f"   Raw length: {len(raw)} chars")
    except json.JSONDecodeError as e:
        print(f"   ✗ Invalid JSON ({elapsed:.1f}s)")
        print(f"   Raw ({len(raw)} chars): {raw[:300]}")
        print(f"   Error: {e}")

except NameError as e:
    print(f"   ✗ Failed: {e}")

# 5. Disk
print(f"\n5. Disk space:")
import shutil
_, _, free = shutil.disk_usage("/kaggle/working")
print(f"   Free: {free/1e9:.1f} GB")

print(f"\n{'='*60}")
print("If DOS test shows ✓ valid JSON — safe to start")
print("If DOS test shows ✗ — Mistral also has the problem")

In [ ]:
# MISTRAL P6 — HCRL + ROAD FULL INFERENCE
# Estimated: ~10 hours total
# HCRL: 500 windows × 2 runs × ~25s = ~7 hours
# ROAD: 250 windows × 2 runs × ~25s = ~3.5 hours

import pandas as pd
import numpy as np
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)

hcrl_p6  = pd.read_parquet(
    INPUT_DIR / "section10_p6_false_context.parquet"
)
road_p6  = pd.read_parquet(
    INPUT_DIR / "task4_road_p6_dataset.parquet"
)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✓ HCRL P6: {len(hcrl_p6)} records")
print(f"  Classes: "
      f"{hcrl_p6['true_label_name'].value_counts().to_dict()}")
print(f"✓ ROAD P6: {len(road_p6)} records")
print(f"  Classes: "
      f"{road_p6['label_name'].value_counts().to_dict()}")
print(f"✓ Mistral loaded: {mistral_model is not None}")
print(f"✓ Embedder ready")

# ================================================================
# SYSTEM PROMPTS
# ================================================================
HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

# ROAD uses correct label space
ROAD_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: AMBIENT, ACCELERATOR, FUZZING, SPEEDOMETER, CORRELATED
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<AMBIENT|ACCELERATOR|FUZZING|SPEEDOMETER|CORRELATED>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

# ================================================================
# GENERATION FUNCTIONS
# max_new_tokens=1024 to handle long P6 responses
# Greedy decoding (do_sample=False) — deterministic
# ================================================================
def generate_mistral_hcrl(text, pred, conf,
                           false_context=""):
    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: {pred} "
        f"(confidence: {conf:.3f})\n"
    )
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += (
        f"\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{HCRL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def generate_mistral_road(text, false_context=""):
    user_content = (
        "Analyze this CAN bus telemetry sequence "
        "and classify it."
    )
    if false_context:
        user_content += f"\n\n{false_context}"
    user_content += (
        f"\n\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{ROAD_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

def run_p6(df, dataset, gen_fn,
           label_col, pred_col=None, conf_col=None):
    """
    Run P6 inference for Mistral on one dataset.
    Saves raw response for diagnosis if parsing fails.
    """
    results = []
    errors  = 0

    for idx, row in tqdm(
        df.iterrows(), total=len(df),
        desc=f"Mistral {dataset} P6"
    ):
        t0 = time.time()
        try:
            text = row["original_text"]
            fc   = row["false_context"]
            cls  = row[label_col]

            if pred_col and conf_col:
                kwargs = {
                    "text": text,
                    "pred": row[pred_col],
                    "conf": float(row[conf_col]),
                }
            else:
                kwargs = {"text": text}

            # Baseline
            raw_base = gen_fn(
                **kwargs, false_context=""
            )
            parsed_base, _ = safe_parse(raw_base)
            exp_base  = parsed_base.get(
                "explanation", "") if parsed_base else ""
            cls_base  = parsed_base.get(
                "classification") if parsed_base else None
            conf_base = parsed_base.get(
                "confidence") if parsed_base else None

            # Perturbed
            raw_pert = gen_fn(
                **kwargs, false_context=fc
            )
            parsed_pert, err_pert = safe_parse(raw_pert)
            exp_pert  = parsed_pert.get(
                "explanation", "") if parsed_pert else ""
            cls_pert  = parsed_pert.get(
                "classification") if parsed_pert else None
            conf_pert = parsed_pert.get(
                "confidence") if parsed_pert else None

            sim = cosine_sim(exp_base, exp_pert)

            results.append({
                "window_idx":               row.get(
                    "window_idx", idx),
                "true_label":               cls,
                "perturbation_type":        "P6_FALSE_CONTEXT",
                "model":                    "Mistral",
                "dataset":                  dataset,
                "baseline_explanation":     exp_base,
                "perturbed_explanation":    exp_pert,
                "baseline_raw":             raw_base,
                "perturbed_raw":            raw_pert,
                "baseline_classification":  cls_base,
                "perturbed_classification": cls_pert,
                "baseline_confidence":      conf_base,
                "perturbed_confidence":     conf_pert,
                "cosine_similarity":        sim,
                "pred_stable":              cls_base == cls_pert
                    if cls_base and cls_pert else None,
                "correct_baseline":         cls_base == cls
                    if cls_base else None,
                "correct_perturbed":        cls_pert == cls
                    if cls_pert else None,
                "json_valid":               parsed_base is not None
                    and parsed_pert is not None,
                "parse_error":              err_pert or "",
                "false_context":            fc,
                "inference_time_s":         time.time() - t0,
            })

        except Exception as e:
            errors += 1
            results.append({
                "window_idx":               row.get(
                    "window_idx", idx),
                "true_label":               row[label_col],
                "perturbation_type":        "P6_FALSE_CONTEXT",
                "model":                    "Mistral",
                "dataset":                  dataset,
                "baseline_explanation":     "",
                "perturbed_explanation":    "",
                "baseline_raw":             "",
                "perturbed_raw":            "",
                "baseline_classification":  None,
                "perturbed_classification": None,
                "baseline_confidence":      None,
                "perturbed_confidence":     None,
                "cosine_similarity":        None,
                "pred_stable":              None,
                "correct_baseline":         None,
                "correct_perturbed":        None,
                "json_valid":               False,
                "parse_error":              str(e),
                "false_context":            row.get(
                    "false_context", ""),
                "inference_time_s":         time.time() - t0,
            })

        # Checkpoint every 50
        if len(results) % 50 == 0:
            cp = pd.DataFrame(results)
            cp.to_parquet(
                f"/kaggle/working/mistral_{dataset.lower()}"
                f"_p6_checkpoint_{len(results)}.parquet"
            )
            valid  = sum(r["json_valid"] for r in results)
            stable = sum(
                r["pred_stable"] == True
                for r in results
                if r["pred_stable"] is not None
            )
            # Show DOS specifically
            dos_results = [
                r for r in results
                if r["true_label"] in ["DOS", "ACCELERATOR"]
            ]
            dos_changed = sum(
                r["correct_baseline"] == True and
                r["correct_perturbed"] == False
                for r in dos_results
                if r["correct_baseline"] is not None
                and r["correct_perturbed"] is not None
            )
            print(f"  Checkpoint {len(results)}: "
                  f"valid={valid}/{len(results)}  "
                  f"stable={stable}/{len(results)}  "
                  f"attack_class_changes={dos_changed}")

        # GPU cleanup every 25 windows
        if idx % 25 == 0:
            torch.cuda.empty_cache()

    df_out = pd.DataFrame(results)
    fname = (f"/kaggle/working/mistral_"
             f"{dataset.lower()}_p6_full.parquet")
    df_out.to_parquet(fname, index=False)

    print(f"\n✓ Mistral {dataset} P6 complete:")
    print(f"  Records:     {len(df_out)}, errors={errors}")
    print(f"  JSON valid:  "
          f"{df_out['json_valid'].sum()}/{len(df_out)}")
    print(f"  Mean sim:    "
          f"{df_out['cosine_similarity'].dropna().mean():.3f}")
    print(f"  Pred stable: "
          f"{df_out['pred_stable'].eq(True).sum()}"
          f"/{len(df_out)}")
    print(f"\n  Per class:")
    for lbl in sorted(df_out["true_label"].unique()):
        s = df_out[df_out["true_label"] == lbl]
        sims   = s["cosine_similarity"].dropna()
        stable = s["pred_stable"].eq(True).sum()
        cb     = s["correct_baseline"].eq(True).sum()
        cp2    = s["correct_perturbed"].eq(True).sum()
        mean_s = f"{sims.mean():.3f}" \
            if len(sims) > 0 else "nan"
        print(f"    {lbl:<14} sim={mean_s}  "
              f"stable={stable}/{len(s)}  "
              f"base_ok={cb}/{len(s)}  "
              f"pert_ok={cp2}/{len(s)}")
    return df_out

# ================================================================
# RUN 1 — MISTRAL HCRL
# ================================================================
print(f"\n{'='*60}")
print(f"RUN 1 — MISTRAL HCRL ({len(hcrl_p6)} windows)")
print(f"Estimated: ~7 hours")
print(f"{'='*60}")

r1 = run_p6(
    df        = hcrl_p6,
    dataset   = "HCRL",
    gen_fn    = generate_mistral_hcrl,
    label_col = "true_label_name",
    pred_col  = "securebert_pred_name",
    conf_col  = "securebert_confidence",
)

# ================================================================
# RUN 2 — MISTRAL ROAD
# ================================================================
print(f"\n{'='*60}")
print(f"RUN 2 — MISTRAL ROAD ({len(road_p6)} windows)")
print(f"Estimated: ~3.5 hours")
print(f"{'='*60}")

r2 = run_p6(
    df        = road_p6,
    dataset   = "ROAD",
    gen_fn    = generate_mistral_road,
    label_col = "label_name",
    pred_col  = None,
    conf_col  = None,
)

# ================================================================
# COMBINED SUMMARY
# ================================================================
print(f"\n\n{'='*60}")
print("MISTRAL P6 COMPLETE — COMBINED SUMMARY")
print("="*60)

print(f"\n{'Dataset':<8} {'Mean Sim':>10} "
      f"{'Valid%':>8} {'Stable%':>10}")
print("-"*40)
for df_out, dataset in [(r1, "HCRL"), (r2, "ROAD")]:
    mean_sim = df_out[
        "cosine_similarity"].dropna().mean()
    valid_pct = df_out["json_valid"].mean() * 100
    stable_pct = df_out[
        "pred_stable"].eq(True).mean() * 100
    print(f"  {dataset:<6} {mean_sim:>10.3f} "
          f"{valid_pct:>8.1f}% {stable_pct:>10.1f}%")

print(f"\nDOS classification under P6 (HCRL):")
dos = r1[r1["true_label"] == "DOS"]
print(f"  Mistral base_ok:  "
      f"{dos['correct_baseline'].eq(True).sum()}/100")
print(f"  Mistral pert_ok:  "
      f"{dos['correct_perturbed'].eq(True).sum()}/100")
print(f"  Classification changes:")
changes = dos[
    dos["correct_baseline"] == True
][dos["correct_perturbed"] == False]
print(f"  {len(changes)} DOS windows reclassified "
      f"as non-DOS under P6")
if len(changes) > 0:
    print(f"  New classifications:")
    print(dos["perturbed_classification"].value_counts(
        dropna=False).to_string())

# Zip
import zipfile, os
zip_path = "/kaggle/working/mistral_p6_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if "mistral" in f and "p6" in f \
                and f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")
print(f"\n✓ Zip: {os.path.getsize(zip_path)/1e6:.1f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/mistral_p6_results.zip')

In [6]:
# PREREQUISITE CHECK — Mistral HCRL P6 only
import torch
import pandas as pd
import json
import time
from pathlib import Path

print("PREREQUISITE CHECK — Mistral HCRL P6")
print("="*60)

# 1. GPU
print(f"\n1. GPU:")
print(f"   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# 2. Mistral
print(f"\n2. Model:")
try:
    print(f"   Mistral loaded: {mistral_model is not None}")
    print(f"   Device: "
          f"{next(mistral_model.parameters()).device}")
    print(f"   Params: "
          f"{sum(p.numel() for p in mistral_model.parameters())/1e9:.2f}B")
except NameError:
    print(f"   ✗ Mistral NOT loaded")

# 3. Input file
print(f"\n3. Input file:")
INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)
path = INPUT_DIR / "section10_p6_false_context.parquet"
if path.exists():
    df = pd.read_parquet(path)
    print(f"   ✓ {len(df)} records")
    print(f"   Classes: "
          f"{df['true_label_name'].value_counts().to_dict()}")
    dos = df[df["true_label_name"]=="DOS"].iloc[0]
    print(f"   DOS text correct: "
          f"{'ID=0000' in dos['original_text']}")
else:
    print(f"   ✗ File not found")

# 4. DOS generation test
print(f"\n4. DOS generation test:")
try:
    hcrl_p6 = pd.read_parquet(path)
    dos_row  = hcrl_p6[
        hcrl_p6["true_label_name"] == "DOS"
    ].iloc[0]

    HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: "
        f"{dos_row['securebert_pred_name']} "
        f"(confidence: "
        f"{float(dos_row['securebert_confidence']):.3f})\n"
        f"\n{dos_row['false_context']}\n"
        f"\n{dos_row['original_text']}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{HCRL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)

    t0 = time.time()
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs, max_new_tokens=1024,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    clean = raw.strip()
    if "```" in clean:
        for part in clean.split("```"):
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    parsed = json.loads(clean)
    print(f"   ✓ Valid JSON ({elapsed:.1f}s)")
    print(f"   Classification: {parsed.get('classification')}")
    print(f"   Explanation: "
          f"{parsed.get('explanation','')[:150]}")

except Exception as e:
    print(f"   ✗ Failed: {e}")

# 5. Disk
print(f"\n5. Disk space:")
import shutil
_, _, free = shutil.disk_usage("/kaggle/working")
print(f"   Free: {free/1e9:.1f} GB")

print(f"\n{'='*60}")
print("If DOS test shows ✓ valid JSON — safe to start")

PREREQUISITE CHECK — Mistral HCRL P6

1. GPU:
   CUDA: True
   GPU 0: Tesla T4
   GPU 1: Tesla T4

2. Model:
   Mistral loaded: True
   Device: cuda:0
   Params: 7.24B

3. Input file:
   ✓ 500 records
   Classes: {'NORMAL': 100, 'DOS': 100, 'FUZZY': 100, 'GEAR': 100, 'RPM': 100}
   DOS text correct: True

4. DOS generation test:
   ✓ Valid JSON (12.2s)
   Classification: NORMAL
   Explanation: Based on the context provided, the high frequency of identical frames in this sequence is an expected artifact of the scheduled ECU stress test conduc

5. Disk space:
   Free: 20.9 GB

If DOS test shows ✓ valid JSON — safe to start


In [7]:
# MISTRAL HCRL P6 — FULL INFERENCE
# 500 windows × 2 runs × ~25s = ~7 hours

import pandas as pd
import numpy as np
import json
import time
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

INPUT_DIR = Path(
    "/kaggle/input/datasets/deepakpatnaik07"
    "/icidea-llm-ids-benchmark"
)

hcrl_p6  = pd.read_parquet(
    INPUT_DIR / "section10_p6_false_context.parquet"
)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"✓ HCRL P6: {len(hcrl_p6)} records")
print(f"  Classes: "
      f"{hcrl_p6['true_label_name'].value_counts().to_dict()}")
print(f"✓ Embedder ready")
print(f"✓ Mistral loaded: {mistral_model is not None}")
print(f"\nEstimated time: ~7 hours")

HCRL_SYSTEM = """You are a cybersecurity analyst specializing in Controller Area Network (CAN) bus intrusion detection.
You will be given a sequence of CAN bus frames from a vehicle network.
Your task is to:
1. Analyze the traffic pattern carefully
2. Classify the traffic as one of: NORMAL, DOS, FUZZY, GEAR, RPM
3. Provide a detailed explanation of your reasoning

You MUST respond in valid JSON format exactly as follows:
{
    "classification": "<NORMAL|DOS|FUZZY|GEAR|RPM>",
    "confidence": <float between 0.0 and 1.0>,
    "explanation": "<detailed reasoning>",
    "key_indicators": ["<indicator1>", "<indicator2>", "<indicator3>"],
    "temporal_reasoning": "<observations about patterns across the sequence>"
}"""

def generate_mistral_hcrl(text, pred, conf,
                           false_context=""):
    user_content = (
        f"Analyze this CAN bus telemetry sequence "
        f"and classify it.\n\n"
        f"SecureBERT classifier prediction: {pred} "
        f"(confidence: {conf:.3f})\n"
    )
    if false_context:
        user_content += f"\n{false_context}\n"
    user_content += (
        f"\n{text}\n\n"
        f"Provide your classification and detailed "
        f"explanation in the required JSON format."
    )
    messages = [
        {"role": "user",
         "content": f"{HCRL_SYSTEM}\n\n{user_content}"}
    ]
    formatted = mistral_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = mistral_tokenizer(
        [formatted], return_tensors="pt",
        truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            pad_token_id=mistral_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return mistral_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

def safe_parse(response):
    clean = response.strip()
    if "```" in clean:
        parts = clean.split("```")
        for part in parts:
            part = part.strip()
            if part.startswith("json"):
                part = part[4:].strip()
            if part.startswith("{"):
                clean = part
                break
    try:
        return json.loads(clean), None
    except json.JSONDecodeError as e:
        return None, str(e)

def cosine_sim(t1, t2):
    if not t1 or not t2:
        return None
    e1 = embedder.encode([t1])[0]
    e2 = embedder.encode([t2])[0]
    return float(np.dot(e1, e2) / (
        np.linalg.norm(e1) * np.linalg.norm(e2)
    ))

# ================================================================
# INFERENCE
# ================================================================
results = []
errors  = 0

for idx, row in tqdm(
    hcrl_p6.iterrows(),
    total=len(hcrl_p6),
    desc="Mistral HCRL P6"
):
    t0 = time.time()
    try:
        text = row["original_text"]
        pred = row["securebert_pred_name"]
        conf = float(row["securebert_confidence"])
        fc   = row["false_context"]
        cls  = row["true_label_name"]

        # Baseline
        raw_base = generate_mistral_hcrl(
            text, pred, conf, false_context=""
        )
        parsed_base, _ = safe_parse(raw_base)
        exp_base  = parsed_base.get(
            "explanation", "") if parsed_base else ""
        cls_base  = parsed_base.get(
            "classification") if parsed_base else None
        conf_base = parsed_base.get(
            "confidence") if parsed_base else None

        # Perturbed
        raw_pert = generate_mistral_hcrl(
            text, pred, conf, false_context=fc
        )
        parsed_pert, err_pert = safe_parse(raw_pert)
        exp_pert  = parsed_pert.get(
            "explanation", "") if parsed_pert else ""
        cls_pert  = parsed_pert.get(
            "classification") if parsed_pert else None
        conf_pert = parsed_pert.get(
            "confidence") if parsed_pert else None

        sim = cosine_sim(exp_base, exp_pert)

        results.append({
            "window_idx":               row["window_idx"],
            "true_label":               cls,
            "perturbation_type":        "P6_FALSE_CONTEXT",
            "model":                    "Mistral",
            "dataset":                  "HCRL",
            "baseline_explanation":     exp_base,
            "perturbed_explanation":    exp_pert,
            "baseline_raw":             raw_base,
            "perturbed_raw":            raw_pert,
            "baseline_classification":  cls_base,
            "perturbed_classification": cls_pert,
            "baseline_confidence":      conf_base,
            "perturbed_confidence":     conf_pert,
            "cosine_similarity":        sim,
            "pred_stable":              cls_base == cls_pert
                if cls_base and cls_pert else None,
            "correct_baseline":         cls_base == cls
                if cls_base else None,
            "correct_perturbed":        cls_pert == cls
                if cls_pert else None,
            "json_valid":               parsed_base is not None
                and parsed_pert is not None,
            "parse_error":              err_pert or "",
            "false_context":            fc,
            "inference_time_s":         time.time() - t0,
        })

    except Exception as e:
        errors += 1
        results.append({
            "window_idx":               row["window_idx"],
            "true_label":               row["true_label_name"],
            "perturbation_type":        "P6_FALSE_CONTEXT",
            "model":                    "Mistral",
            "dataset":                  "HCRL",
            "baseline_explanation":     "",
            "perturbed_explanation":    "",
            "baseline_raw":             "",
            "perturbed_raw":            "",
            "baseline_classification":  None,
            "perturbed_classification": None,
            "baseline_confidence":      None,
            "perturbed_confidence":     None,
            "cosine_similarity":        None,
            "pred_stable":              None,
            "correct_baseline":         None,
            "correct_perturbed":        None,
            "json_valid":               False,
            "parse_error":              str(e),
            "false_context":            row["false_context"],
            "inference_time_s":         time.time() - t0,
        })

    # Checkpoint every 50
    if len(results) % 50 == 0:
        cp = pd.DataFrame(results)
        cp.to_parquet(
            f"/kaggle/working/mistral_hcrl_p6_"
            f"checkpoint_{len(results)}.parquet"
        )
        valid  = sum(r["json_valid"] for r in results)
        stable = sum(
            r["pred_stable"] == True
            for r in results
            if r["pred_stable"] is not None
        )

        # Per class summary at checkpoint
        cp_df = pd.DataFrame(results)
        print(f"\n  Checkpoint {len(results)}: "
              f"valid={valid}/{len(results)}  "
              f"stable={stable}/{len(results)}")
        print(f"  Per class so far:")
        for lbl in sorted(cp_df["true_label"].unique()):
            s   = cp_df[cp_df["true_label"] == lbl]
            cb  = s["correct_baseline"].eq(True).sum()
            cp2 = s["correct_perturbed"].eq(True).sum()
            sim_mean = s["cosine_similarity"].dropna().mean()
            sim_str  = f"{sim_mean:.3f}" \
                if not np.isnan(sim_mean) else "nan"
            print(f"    {lbl:<8} "
                  f"base_ok={cb}/{len(s)}  "
                  f"pert_ok={cp2}/{len(s)}  "
                  f"sim={sim_str}")

    # GPU cleanup every 25 windows
    if idx % 25 == 0:
        torch.cuda.empty_cache()

# ================================================================
# SAVE
# ================================================================
df_out = pd.DataFrame(results)
df_out.to_parquet(
    "/kaggle/working/mistral_hcrl_p6_full.parquet",
    index=False
)

# ================================================================
# FINAL SUMMARY
# ================================================================
print(f"\n\n{'='*60}")
print("MISTRAL HCRL P6 COMPLETE")
print("="*60)
print(f"Records:    {len(df_out)}, errors={errors}")
print(f"JSON valid: {df_out['json_valid'].sum()}/{len(df_out)}")
print(f"Mean sim:   "
      f"{df_out['cosine_similarity'].dropna().mean():.3f}")

print(f"\nPer class:")
for lbl in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    s    = df_out[df_out["true_label"] == lbl]
    sims = s["cosine_similarity"].dropna()
    cb   = s["correct_baseline"].eq(True).sum()
    cp2  = s["correct_perturbed"].eq(True).sum()
    sim_str = f"{sims.mean():.3f}" \
        if len(sims) > 0 else "nan"
    print(f"  {lbl:<8} sim={sim_str}  "
          f"base_ok={cb}/{len(s)}  "
          f"pert_ok={cp2}/{len(s)}")

print(f"\nClassification changes under P6:")
print(f"{'Class':<10} {'Perturbed classification distribution'}")
for lbl in ["NORMAL", "DOS", "FUZZY", "GEAR", "RPM"]:
    s = df_out[df_out["true_label"] == lbl]
    dist = s["perturbed_classification"].value_counts(
        dropna=False
    ).to_dict()
    print(f"  {lbl:<8} {dist}")

# Zip
import zipfile, os
zip_path = "/kaggle/working/mistral_hcrl_p6_results.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for f in os.listdir("/kaggle/working"):
        if "mistral_hcrl_p6" in f \
                and f.endswith(".parquet"):
            zipf.write(f"/kaggle/working/{f}", f)
            print(f"  Added: {f}")
print(f"\n✓ Zip: "
      f"{os.path.getsize(zip_path)/1e6:.1f} MB")

from IPython.display import FileLink
FileLink('/kaggle/working/mistral_hcrl_p6_results.zip')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Mistral HCRL P6:  90%|█████████ | 450/500 [4:59:50<29:19, 35.18s/it]  


  Checkpoint 450: valid=444/450  stable=343/450
  Per class so far:
    DOS      base_ok=100/100  pert_ok=0/100  sim=0.511
    FUZZY    base_ok=97/100  pert_ok=96/100  sim=0.709
    GEAR     base_ok=100/100  pert_ok=100/100  sim=0.767
    NORMAL   base_ok=100/100  pert_ok=99/100  sim=0.867
    RPM      base_ok=50/50  pert_ok=50/50  sim=0.752


Mistral HCRL P6: 100%|██████████| 500/500 [5:28:47<00:00, 39.46s/it]


  Checkpoint 500: valid=494/500  stable=393/500
  Per class so far:
    DOS      base_ok=100/100  pert_ok=0/100  sim=0.511
    FUZZY    base_ok=97/100  pert_ok=96/100  sim=0.709
    GEAR     base_ok=100/100  pert_ok=100/100  sim=0.767
    NORMAL   base_ok=100/100  pert_ok=99/100  sim=0.867
    RPM      base_ok=100/100  pert_ok=100/100  sim=0.749


MISTRAL HCRL P6 COMPLETE
Records:    500, errors=0
JSON valid: 494/500
Mean sim:   0.721

Per class:
  NORMAL   sim=0.867  base_ok=100/100  pert_ok=99/100
  DOS      sim=0.511  base_ok=100/100  pert_ok=0/100
  FUZZY    sim=0.709  base_ok=97/100  pert_ok=96/100
  GEAR     sim=0.767  base_ok=100/100  pert_ok=100/100
  RPM      sim=0.749  base_ok=100/100  pert_ok=100/100

Classification changes under P6:
Class      Perturbed classification distribution
  NORMAL   {'NORMAL': 99, None: 1}
  DOS      {'NORMAL': 100}
  FUZZY    {'FUZZY': 96, None: 3, 'NORMAL': 1}
  GEAR     {'GEAR': 100}
  RPM      {'RPM': 100}
  Added: mistral_hcrl_p6_checkpoint_1

/kaggle/working/mistral_hcrl_p6_results.zip